# 🚀 DAQathon Session 1 Advanced Notebook: Tuning, Features, Better Targets, CNNs, and Transformers

This notebook picks up where the main notebook leaves off.

The focus here is not just to tune models for a higher score. It is to learn how to think about:

- better feature preparation,
- better target design,
- better model-family choices,
- stronger CNN baselines,
- and transformer-based sequence modelling, including good-data forecasting for anomaly detection.


## 🧰 One-Time FIR Setup

Before you use these notebooks on FIR for the first time, open a FIR terminal and run:

```bash
jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.


## 🧭 How To Use This Advanced Notebook

This notebook extends the Session 1 workflow with heavier experiments:

- revisit the dataset and fixed split,
- tune the Random Forest baseline,
- tune and extend the CNN sequence models,
- compare transformer classifiers against the CNN framing,
- try a good-data forecasting transformer for anomaly-style scoring,
- then decide which modelling choices are worth carrying forward.

A few suggestions while you work through the notebook:

- read the markdown cells first, then run the code cells underneath them,
- use `DATA_FRACTION` in the config cell to switch between a quick demo and a longer run,
- run the CNN and transformer sections when you want to compare sequence models,
- treat the default model settings as **baseline choices**, not as final answers.

This notebook includes additional hyperparameter search cells, so it is heavier than the main notebook.

Optional background reading:

- [Random Forests in scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [k-means clustering in scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [Neural networks in PyTorch](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [ONC QAQC reference](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)


## ⚙️ Configuration

The next few cells separate the main configuration ideas:

- choose a dataset preset,
- optionally override a few important dataset-specific fields,
- then adjust broad run controls like how much data to use.

On FIR, if `$SLURM_TMPDIR` is available, this notebook first stages the shared raw CSV directory and the prepared parquet cache into node-local job storage. It writes all generated outputs into a runtime directory chosen in this order:

1. `$SLURM_TMPDIR`
2. `$SCRATCH`
3. repo-local `tmp/session1_outputs`


In [ ]:
from pathlib import Path
import sys

# Find the repo root before importing utilities from scripts/.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_notebook_bootstrap import build_notebook_bootstrap_namespace

globals().update(build_notebook_bootstrap_namespace(NOTEBOOK_ROOT))
show_setup_json(
    {
        "NOTEBOOK_ROOT": NOTEBOOK_ROOT,
        "SHARED_DAQATHON_ROOT": SHARED_DAQATHON_ROOT,
        "LOCAL_CACHE_DIR": LOCAL_CACHE_DIR,
        "SHARED_CACHE_DIR": SHARED_CACHE_DIR,
    }
)


### Dataset Presets

The next cell defines the **supported scalar dataset presets** for this notebook.

Each preset bundles together the choices that usually travel as a group:

- where the raw files live,
- which label column is the supervised target,
- which measurement columns to use,
- which two columns should appear in plots,
- which device family is the primary stream,
- and whether k-means should work on window summaries or row-level features.

In most cases, you only need to change `DATASET_PROFILE_ID`.


In [ ]:
# Each profile captures the dataset-specific choices that would otherwise be
# repeated throughout the notebook. Once a profile is selected, later cells
# work with normalised variables such as TARGET_FLAG or MEASUREMENT_COLUMNS.
DATASET_PROFILES = {
    "ctd_conductivity": {
        # A classic CTD example: one instrument export with conductivity QC as the label.
        "label": "CTD conductivity QC",
        "description": "Strait of Georgia East CTD data with conductivity QC as the supervised target.",
        "raw_subpaths": ["SoGEast_CTD_202503_202603"],
        "cache_stem": "conductivity_scalar_session1",
        "target_flag": "Conductivity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Depth (m)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Conductivity (S/m)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "ctd",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "fluorometer_turbidity": {
        # A merged scalar example: fluorometer is the primary device, but CTD and oxygen
        # provide useful context columns during the merge.
        "label": "Fluorometer turbidity QC",
        "description": "Merged scalar data around a fluorometer/turbidity target, including CTD and oxygen context columns.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "Fluorometer/SoGCentral_test", "SoGCentral_test"],
        "cache_stem": "sogcentral_turbidity",
        "target_flag": "Turbidity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Chlorophyll (ug/l)",
            "Turbidity (NTU)",
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
        ],
        "optional_qc_columns": ["Chlorophyll QC Flag"],
        "plot_measurement_column": "Turbidity (NTU)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "fluorometer",
        "kmeans_feature_mode": "row_level",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "oxygen": {
        # A smaller oxygen-focused profile for the same scalar workflow.
        "label": "Oxygen QC",
        "description": "Scalar oxygen data with oxygen concentration QC as the supervised target.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "SoGEast_Oxygen_202503_202603", "Oxygen", "oxygen"],
        "cache_stem": "sogcentral_oxygen",
        "target_flag": "Oxygen Concentration Corrected QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 2, 3, 4, 9],
        "measurement_columns": [
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
            "Temperature (C)",
            "Pressure (decibar)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Oxygen Concentration Corrected (ml/l)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "oxygen",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_12_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "conductivity_plugs": {
        # A labelled conductivity-plug example with a compact CSV schema.
        "label": "Conductivity plugs",
        "description": "Conductivity-plug data with a custom ml_label target and CTD/oxygen context columns.",
        "raw_subpaths": ["ConductivityPlugs"],
        "cache_stem": "conductivity_plugs_session1",
        "target_flag": "ml_label",
        "task_mode": "multiclass",
        # Custom labels for the conductivity-plug dataset:
        # 0 = good, 1 = conductivity bad plug, 2 = conductivity bad other,
        # 3 = non-conductivity failure, 4 = missing data.
        "good_labels": [0],
        "issue_labels": [1, 2, 3, 4],
        "flag_example_classes": [1, 2, 3, 4],
        "measurement_columns": [
            "cond_value_ctd",
            "density_value_ctd",
            "Pressure_value_ctd",
            "salinity_value_ctd",
            "sigmaT_value_ctd",
            "SIGMA_THETA_value_ctd",
            "Sound_Speed_value_ctd",
            "Temperature_value_ctd",
            "oxygen_corrected_value_oxy",
            "oxygen_uncorrected_value_oxy",
            "temperature_value_oxy",
            "temperature_offset",
            "temperature_offset_anomaly",
            "temperature_offset_over_start_mean",
        ],
        "optional_qc_columns": ["cond_qaqc_ctd", "oxygen_corrected_qaqc_oxy", "temperature_qaqc_oxy"],
        "plot_measurement_column": "cond_value_ctd",
        "plot_secondary_column": "Temperature_value_ctd",
        "primary_device": "other",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "raw_multiclass",
        "cache_read_sample_rows": 250_000,
        "auto_build_missing_cache": False,
        "default_prep_sample_rows": 500_000,
    },
}

# Change this to switch the whole notebook to a different supported preset.
DATASET_PROFILE_ID = "ctd_conductivity"
DATASET_PROFILE = DATASET_PROFILES[DATASET_PROFILE_ID]

show_setup_json(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "DATASET_LABEL": DATASET_PROFILE["label"],
        "DESCRIPTION": DATASET_PROFILE["description"],
        "PRIMARY_DEVICE": DATASET_PROFILE["primary_device"],
        "GOOD_LABELS": DATASET_PROFILE.get("good_labels", [1]),
        "ISSUE_LABELS": DATASET_PROFILE.get("issue_labels", [3, 4, 9]),
    }
)


### Optional Dataset Overrides

Presets are the normal starting point, but these overrides let you change a few high-value pieces without editing the rest of the notebook.

Leave them as `None` to use the preset values. Typical reasons to override something here are:

- testing a different raw folder,
- changing the target flag,
- narrowing the measurement columns,
- or forcing a different k-means feature mode.


In [ ]:
# Leave these as None unless you intentionally want to override part of the preset.
# The simplest workflow is "pick a preset, then tweak only one or two fields."
MANUAL_RAW_DATA_DIR = None
MANUAL_TARGET_FLAG = None
MANUAL_MEASUREMENT_COLUMNS = None
MANUAL_OPTIONAL_QC_COLUMNS = None
MANUAL_PLOT_MEASUREMENT_COLUMN = None
MANUAL_PLOT_SECONDARY_COLUMN = None
MANUAL_CACHE_BUNDLE_NAME = None
MANUAL_KMEANS_FEATURE_MODE = None

# Build a short list of likely raw-data locations for this profile.
# On FIR we prefer the shared project directory; locally we fall back to repo data.
profile_raw_candidates = [Path(subpath) for subpath in DATASET_PROFILE["raw_subpaths"]]
local_raw_candidates = [NOTEBOOK_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
shared_raw_candidates = (
    [SHARED_DAQATHON_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
    if SHARED_DAQATHON_ROOT
    else []
)

# Resolve the best available raw-data and cache locations without forcing
# you to edit paths in multiple notebook sections.
detected_raw_data_dir = (
    first_existing_csv_dir([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates] if candidate is not None])
    or first_existing_path([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates, NOTEBOOK_ROOT / "data" / "raw"] if candidate is not None])
)
detected_cache_dir = first_existing_path([candidate for candidate in [LOCAL_CACHE_DIR, SHARED_CACHE_DIR] if candidate is not None])

# Normalise everything into the small set of variables used by the rest of the notebook.
# From this point onward, later cells should not need to care which preset was chosen.
READ_RAW_DATA_DIR = MANUAL_RAW_DATA_DIR or (
    str(detected_raw_data_dir)
    if detected_raw_data_dir is not None
    else str(local_raw_candidates[0] if local_raw_candidates else NOTEBOOK_ROOT / "data" / "raw")
)
READ_CACHE_DIR = str(detected_cache_dir) if detected_cache_dir is not None else str(LOCAL_CACHE_DIR)

TARGET_FLAG = MANUAL_TARGET_FLAG or DATASET_PROFILE["target_flag"]
TASK_MODE = DATASET_PROFILE["task_mode"]
GOOD_LABELS = [int(label) for label in DATASET_PROFILE.get("good_labels", [1])]
ISSUE_LABELS = [int(label) for label in DATASET_PROFILE.get("issue_labels", [3, 4, 9])]
FLAG_EXAMPLE_CLASSES = tuple(int(label) for label in DATASET_PROFILE.get("flag_example_classes", [1, 3, 4, 9]))
CACHE_BUNDLE_NAME = MANUAL_CACHE_BUNDLE_NAME or DATASET_PROFILE["cache_stem"]
PRIMARY_DEVICE = DATASET_PROFILE["primary_device"]
KMEANS_FEATURE_MODE = MANUAL_KMEANS_FEATURE_MODE or DATASET_PROFILE["kmeans_feature_mode"]
DEFAULT_SEQUENCE_OUTPUT_MODE = DATASET_PROFILE.get("default_sequence_output_mode", "window")
DEFAULT_SEQUENCE_TARGET_STRATEGY = DATASET_PROFILE.get("default_sequence_target_strategy", "collapsed_1_34_9")
RAW_CACHE_READ_SAMPLE_ROWS = DATASET_PROFILE.get("cache_read_sample_rows")
AUTO_BUILD_MISSING_CACHE = bool(DATASET_PROFILE.get("auto_build_missing_cache", True))
DEFAULT_PREP_SAMPLE_ROWS = DATASET_PROFILE.get("default_prep_sample_rows")

# Keep the feature list in one place so every later section reuses the same columns.
MEASUREMENT_COLUMNS = list(MANUAL_MEASUREMENT_COLUMNS or DATASET_PROFILE["measurement_columns"])
OPTIONAL_QC_COLUMNS = list(dict.fromkeys(MANUAL_OPTIONAL_QC_COLUMNS or DATASET_PROFILE["optional_qc_columns"]))
PLOT_MEASUREMENT_COLUMN = MANUAL_PLOT_MEASUREMENT_COLUMN or DATASET_PROFILE["plot_measurement_column"]
PLOT_SECONDARY_COLUMN = MANUAL_PLOT_SECONDARY_COLUMN or DATASET_PROFILE["plot_secondary_column"]

# These explicit column lists are one of the main reasons the notebook can stay fast:
# we only read the row-level or window-level fields we actually need downstream.
ROW_USE_COLUMNS = list(dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG, *OPTIONAL_QC_COLUMNS, *MEASUREMENT_COLUMNS]))
WINDOW_FEATURE_COLUMNS = [
    f"{column}_{stat}"
    for column in MEASUREMENT_COLUMNS
    for stat in ("mean", "std")
]
WINDOW_USE_COLUMNS = [
    "window_start",
    "window_end",
    "source_file",
    "issue_rate",
    *WINDOW_FEATURE_COLUMNS,
]

show_setup_json(
    {
        "READ_RAW_DATA_DIR": READ_RAW_DATA_DIR,
        "READ_CACHE_DIR": READ_CACHE_DIR,
        "TARGET_FLAG": TARGET_FLAG,
        "CACHE_BUNDLE_NAME": CACHE_BUNDLE_NAME,
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "DEFAULT_SEQUENCE_OUTPUT_MODE": DEFAULT_SEQUENCE_OUTPUT_MODE,
        "DEFAULT_SEQUENCE_TARGET_STRATEGY": DEFAULT_SEQUENCE_TARGET_STRATEGY,
        "RAW_CACHE_READ_SAMPLE_ROWS": RAW_CACHE_READ_SAMPLE_ROWS,
        "AUTO_BUILD_MISSING_CACHE": AUTO_BUILD_MISSING_CACHE,
        "DEFAULT_PREP_SAMPLE_ROWS": DEFAULT_PREP_SAMPLE_ROWS,
        "GOOD_LABELS": GOOD_LABELS,
        "ISSUE_LABELS": ISSUE_LABELS,
        "MEASUREMENT_COLUMNS": MEASUREMENT_COLUMNS,
        "PLOT_MEASUREMENT_COLUMN": PLOT_MEASUREMENT_COLUMN,
        "PLOT_SECONDARY_COLUMN": PLOT_SECONDARY_COLUMN,
    }
)


### Run Controls

These are the notebook-wide knobs you are most likely to change.

The most important is `DATA_FRACTION`:

- use a small value for quick live demos,
- increase it when you want a stronger run,
- keep it near `1.0` when you want the largest training budget.

`DATA_FRACTION` controls the routine row/window budgets used for reviewed dataset construction, model fitting, tuning, and k-means. Other controls should change modelling choices, not dataset size.


In [ ]:
# One easy top-level knob: start at 0.1 for a quick run, then raise it if you want a bigger experiment.
DATA_FRACTION = 0.1
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Part 1 uses only a tiny raw-data preview so we can look at real examples quickly.
ORIENTATION_FILE_SELECTION_MODE = "spread"
ORIENTATION_FILE_LIMIT = 3
BASE_FLAG_EXAMPLE_POINTS_PER_PANEL = 30_000

# Split fractions are reused across every split strategy. The strategy itself is
# configured in the next cell.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Keep stochastic model training reproducible. Hyperparameter search is still a
# separate advanced-workflow choice because it can take much longer than one baseline run.
SEED = 21
RUN_HYPERPARAMETER_SEARCH = True

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "ORIENTATION_FILE_SELECTION_MODE": ORIENTATION_FILE_SELECTION_MODE,
        "ORIENTATION_FILE_LIMIT": ORIENTATION_FILE_LIMIT,
        "SEED": SEED,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
        "RUN_HYPERPARAMETER_SEARCH": RUN_HYPERPARAMETER_SEARCH,
    }
)


In [ ]:
import sys

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_intro_notebook_setup import build_intro_notebook_namespace

globals().update(
    build_intro_notebook_namespace(
        notebook_root=NOTEBOOK_ROOT,
        read_raw_data_dir=READ_RAW_DATA_DIR,
        read_cache_dir=READ_CACHE_DIR,
        cache_bundle_name=CACHE_BUNDLE_NAME,
        seed=SEED,
    )
)

# Compatibility alias for older advanced-only cache cells.
choose_cache_paths = choose_cache_bundle_paths

print(
    {
        **INTRO_NOTEBOOK_SETUP_SUMMARY,
        "TASK_MODE": TASK_MODE,
        "TARGET_FLAG": TARGET_FLAG,
        "DATA_FRACTION": DATA_FRACTION,
        "RUN_HYPERPARAMETER_SEARCH": RUN_HYPERPARAMETER_SEARCH,
        "SHARED_DAQATHON_ROOT": str(SHARED_DAQATHON_ROOT) if SHARED_DAQATHON_ROOT else None,
    }
)


## 🚀 FIR Input Staging

On FIR, the shared project space is the long-lived home for the data. But interactive notebook jobs on Alliance systems usually also get a directory called `$SLURM_TMPDIR`.

`SLURM_TMPDIR` is a **job-specific temporary working directory** created for you when Slurm starts a scheduled job or interactive session on a compute node. In other words:

- it appears when your Jupyter job is actually running on allocated resources,
- it lives on storage attached to that compute node,
- and it disappears when the job ends.

That makes it very different from your shared `HOME`, `SCRATCH`, or project directories:

- shared filesystems are persistent and great for canonical storage,
- `SLURM_TMPDIR` is temporary but usually much faster for repeated reads and writes during one run.

That speed matters here because notebooks repeatedly scan parquet parts, load row samples, and build model inputs. Copying the raw CSV directory and the prepared parquet cache into `SLURM_TMPDIR` once at the start usually makes the rest of the session feel much more responsive and reduces repeated traffic to the shared filesystem.

That means:

- the shared project space stays as the canonical source,
- the notebook reads from a fast local working copy during the run,
- and all generated files stay in your job-local runtime area instead of polluting the shared project tree.

One important caution: `SLURM_TMPDIR` is **not** long-term storage. Anything you want to keep after the job ends should be copied to `SCRATCH`, project storage, or a Git repo. This notebook uses `SLURM_TMPDIR` for speed, not for permanence.

The next code cell shows a progress bar while those copies are happening.


In [ ]:
# On FIR, stage shared inputs into node-local storage before we start reading them.
read_raw_data_dir = Path(READ_RAW_DATA_DIR)
runtime_raw_data_dir = Path(RUNTIME_RAW_DATA_DIR)
read_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)

raw_stage_result = {"staged": False, "reason": "raw-data staging is disabled because SLURM_TMPDIR is not available"}
cache_stage_result = {"staged": False, "reason": "cache staging is disabled because SLURM_TMPDIR is not available"}
if USE_RUNTIME_RAW_DATA_FOR_READS:
    raw_stage_result = stage_directory_into_runtime(
        source_dir=read_raw_data_dir,
        runtime_dir=runtime_raw_data_dir,
        force=False,
        show_progress=True,
        progress_label=f"Staging raw scalar CSV files for {DATASET_PROFILE['label']} to SLURM_TMPDIR",
    )
if USE_RUNTIME_CACHE_FOR_READS:
    cache_stage_result = stage_cache_into_runtime(
        persistent_cache_dir=read_cache_dir,
        runtime_cache_dir=runtime_cache_dir,
        force=False,
        show_progress=True,
        progress_label="Staging parquet cache to SLURM_TMPDIR",
    )

active_raw_data_dir = runtime_raw_data_dir if runtime_raw_data_dir.exists() and any(runtime_raw_data_dir.glob("*.csv")) else read_raw_data_dir
active_cache_paths = choose_cache_paths([runtime_cache_dir, read_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
RAW_DATA_DIR = str(active_raw_data_dir)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

print(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "raw_stage_result": raw_stage_result,
        "cache_stage_result": cache_stage_result,
        "active_raw_data_dir": RAW_DATA_DIR,
        "active_cache_dir": CACHE_DIR,
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "resolved_cache_stem": active_cache_paths.stem,
    }
)


## Part 1 — Orientation and Dataset

Start by understanding what the selected scalar dataset contains, what the target labels mean, and what success looks like before we touch any modelling code.


#### About The Dataset

This notebook now supports a small set of **scalar dataset presets** rather than one hard-coded instrument export.

The active preset is chosen in the configuration cell near the top. That preset tells the notebook:

- which raw-data folder to read,
- which target-label column is used,
- which measurement columns to model,
- which two variables to emphasize in plots,
- and whether k-means should cluster window summaries or row-level values.

Across all of these presets, the mental model stays the same:

- each row is one timestamp in UTC,
- measurement columns hold scalar sensor values,
- target-label columns hold the reviewed label we want the models to learn.

The printed config summary tells you exactly which preset is active for the current run.


#### What The QC Flags Mean

Many ONC scalar datasets use the standard QAQC flag system. Some presets, such as conductivity plugs, use a custom label column instead; the active target column and label meanings are shown in the setup output above. For standard QAQC flags, the official reference is:

- [ONC Quality Assurance Quality Control](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)

A practical reading of the most common flags is:

- `0`: no quality control has been applied to that value yet
- `1`: passed all tests and is treated as good data
- `2`: probably good, but worth a little more caution than `1`
- `3`: probably bad or suspect
- `4`: bad and usually treated as failed data
- `6`: insufficient valid data for reliable down-sampling
- `7`: averaged value
- `8`: interpolated value
- `9`: missing value (`NaN`)

A few practical notes are especially important for this notebook:

- `1` is the clearest "normal / healthy" label.
- `3` and `4` are the main issue labels we usually care about in supervised QC experiments.
- `9` means the value is missing, not merely noisy.
- `0` does **not** mean good; it means no QC decision has been applied.

ONC's reference page also explains that manual review can override automated tests, which is useful context when you see the same instrument behaviour receive different final flags over time.

The selected target flag column is shown in the config summary, and later cells will show which QC values actually appear in the active dataset and which ones we treat as reviewed labels for modelling.


### Looking At Real Examples Early

Before moving on to caching or feature engineering, it helps to inspect a lightweight sample of the raw CSV data directly.

In the next few cells we:

- load a small sample from a few raw CSV files spread across the time range,
- look at a few real rows,
- and plot representative QC intervals in context.

This gives you a concrete picture of what one timestamp looks like before the notebook starts building model inputs from it.


### Small Utilities For Exploration Samples

The next helper cell defines a few short utilities that we first need here in Part 1:

- to spread a sample across time instead of taking only the earliest rows,
- to select files across the full time range,
- and to read a small raw-CSV sample without editing the original exports.

They stay available for later sections too, but this is the first place they become useful.


In [ ]:
# Take an evenly spaced subset without scrambling time order.
def evenly_spaced_take(frame: pd.DataFrame, limit: int | None, *, time_column: str = "Time UTC") -> pd.DataFrame:
    """Return rows spread across the selected time axis.

    Use `time_column="Time UTC"` for row-level data and
    `time_column="window_start"` for window-summary data.
    """
    sort_column = time_column if time_column in frame.columns else "window_start" if "window_start" in frame.columns else None
    ordered = frame.sort_values(sort_column).reset_index(drop=True).copy() if sort_column else frame.reset_index(drop=True).copy()
    if limit is None or len(ordered) <= limit:
        return ordered
    indices = np.linspace(0, len(ordered) - 1, num=limit, dtype=int)
    return ordered.iloc[indices].reset_index(drop=True)


# Choose files spread across the whole time range instead of just the first few.
def select_part_paths(part_paths: list[Path], limit: int | None, mode: str) -> list[Path]:
    if limit is None or limit >= len(part_paths):
        return part_paths
    if mode == "first":
        return part_paths[:limit]
    if mode == "spread":
        indices = np.linspace(0, len(part_paths) - 1, num=limit, dtype=int)
        selected = []
        seen = set()
        for index in indices:
            candidate = part_paths[int(index)]
            if candidate not in seen:
                selected.append(candidate)
                seen.add(candidate)
        return selected
    raise ValueError(f"Unsupported selection mode: {mode}")


# Read a lightweight sample directly from raw CSV files for early exploration.
def load_raw_scalar_sample(
    csv_paths: list[Path],
    sample_rows_per_file: int | None,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    row_frames = []
    for path in csv_paths:
        # For quick notebook previews, sample during the CSV read instead of
        # loading an entire multi-gigabyte file and trimming it afterward.
        frame = read_scalar_csv(
            path,
            sample_rows=sample_rows_per_file,
            required_columns=columns,
            allow_missing_columns=True,
        ).sort_values("Time UTC").reset_index(drop=True)
        row_frames.append(frame)

    return pd.concat(row_frames, ignore_index=True).sort_values("Time UTC").reset_index(drop=True)


# Keep only raw CSVs that actually contain the columns needed for
# the early orientation plots. Some dataset folders contain mixed
# device exports (for example CTD + fluorometer + oxygen), so the
# target flag may only exist in a subset of files.
def filter_csv_paths_with_required_columns(
    csv_paths: list[Path],
    required_columns: list[str] | tuple[str, ...],
) -> list[Path]:
    required = [column for column in required_columns if column]
    matching_paths = []
    for path in csv_paths:
        _, csv_columns = locate_header(path)
        if all(column in csv_columns for column in required):
            matching_paths.append(path)
    return matching_paths


# Scan raw CSV files in chunks and pull local context windows around
# requested target labels. This keeps the Part 1 plots tied to the raw CSVs
# while avoiding the "first N rows only" artifact from the preview table.
def load_raw_flag_context_sample(
    csv_paths: list[Path],
    *,
    target_flag: str,
    classes: list[int] | tuple[int, ...],
    context_rows_per_class: int,
    columns: list[str] | None = None,
    chunk_rows: int | None = None,
) -> pd.DataFrame:
    requested_classes = [int(flag) for flag in classes]
    remaining_classes = list(dict.fromkeys(requested_classes))
    context_rows_per_class = max(int(context_rows_per_class), 1)
    chunk_rows = max(int(chunk_rows or max(context_rows_per_class * 2, 50000)), context_rows_per_class)
    use_columns = list(dict.fromkeys(["Time UTC", target_flag, *(columns or [])]))
    context_frames = []

    for path in csv_paths:
        if not remaining_classes:
            break

        header_line_number, csv_columns = locate_header(path)
        available_columns = [column for column in use_columns if column in csv_columns]
        if "Time UTC" not in available_columns or target_flag not in available_columns:
            continue
        trailing_context = pd.DataFrame(columns=available_columns)

        for chunk in pd.read_csv(
            path,
            header=None,
            names=csv_columns,
            skiprows=header_line_number,
            usecols=available_columns,
            chunksize=chunk_rows,
            low_memory=False,
        ):
            chunk["Time UTC"] = pd.to_datetime(chunk["Time UTC"], utc=True, errors="coerce", format="ISO8601")
            for column in [column for column in chunk.columns if column != "Time UTC"]:
                chunk[column] = pd.to_numeric(chunk[column], errors="coerce")
            chunk = chunk.dropna(subset=["Time UTC"]).sort_values("Time UTC").reset_index(drop=True)
            chunk["source_file"] = path.name
            if chunk.empty:
                continue

            combined = (
                pd.concat([trailing_context, chunk], ignore_index=True)
                if not trailing_context.empty
                else chunk.copy()
            )
            combined_flags = combined[target_flag].fillna(-1).astype(int)

            for flag in remaining_classes.copy():
                flag_positions = combined.index[combined_flags == flag].tolist()
                if not flag_positions:
                    continue

                span_start_index = flag_positions[0]
                while span_start_index > 0 and int(combined_flags.iloc[span_start_index - 1]) == flag:
                    span_start_index -= 1
                span_end_index = flag_positions[0]
                while span_end_index + 1 < len(combined) and int(combined_flags.iloc[span_end_index + 1]) == flag:
                    span_end_index += 1

                span_midpoint_index = (span_start_index + span_end_index) // 2
                start = max(span_midpoint_index - context_rows_per_class // 2, 0)
                stop = min(start + context_rows_per_class, len(combined))
                if stop - start < context_rows_per_class:
                    start = max(stop - context_rows_per_class, 0)

                context_frames.append(combined.iloc[start:stop].copy())
                remaining_classes.remove(flag)

            trailing_context = combined.iloc[-context_rows_per_class:].copy()

    if not context_frames:
        return pd.DataFrame(columns=use_columns + ["source_file"])
    return pd.concat(context_frames, ignore_index=True).sort_values("Time UTC").reset_index(drop=True)



In [ ]:
# Load a lightweight exploration sample directly from raw CSV files.
raw_csv_paths = sorted(Path(RAW_DATA_DIR).glob("*.csv"))
if not raw_csv_paths:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")

# Conductivity-plug examples should always be selected and coloured by the
# reviewed ML label, not by the raw instrument QAQC columns.
FLAG_EXAMPLE_TARGET = "ml_label" if DATASET_PROFILE_ID == "conductivity_plugs" else TARGET_FLAG
FLAG_EXAMPLE_DISPLAY_NAME = "ml_label" if DATASET_PROFILE_ID == "conductivity_plugs" else "QC flag"
FLAG_EXAMPLE_LABEL_MEANINGS = (
    {
        0: "good",
        1: "conductivity bad plug",
        2: "conductivity bad other",
        3: "non-conductivity failure",
        4: "missing data",
    }
    if DATASET_PROFILE_ID == "conductivity_plugs"
    else None
)
FLAG_EXAMPLE_AVOID_CONTEXT_LABELS = (4,) if DATASET_PROFILE_ID == "conductivity_plugs" else (9,)

orientation_required_columns = ["Time UTC", FLAG_EXAMPLE_TARGET]
if PLOT_MEASUREMENT_COLUMN:
    orientation_required_columns.append(PLOT_MEASUREMENT_COLUMN)
orientation_candidate_paths = filter_csv_paths_with_required_columns(
    raw_csv_paths,
    orientation_required_columns,
)
orientation_source_paths = orientation_candidate_paths if orientation_candidate_paths else raw_csv_paths

orientation_file_limit = min(ORIENTATION_FILE_LIMIT, len(orientation_source_paths))
orientation_selected_paths = select_part_paths(
    orientation_source_paths,
    limit=orientation_file_limit,
    mode=ORIENTATION_FILE_SELECTION_MODE,
)
ORIENTATION_ROWS_PER_FILE = 4000
orientation_columns = list(
    dict.fromkeys(
        [
            "Time UTC",
            FLAG_EXAMPLE_TARGET,
            TARGET_FLAG,
            *OPTIONAL_QC_COLUMNS,
            PLOT_MEASUREMENT_COLUMN,
            PLOT_SECONDARY_COLUMN,
            *MEASUREMENT_COLUMNS,
            "Depth (m)",
        ]
    )
)

exploration_df = load_raw_scalar_sample(
    orientation_selected_paths,
    sample_rows_per_file=ORIENTATION_ROWS_PER_FILE,
    columns=orientation_columns,
)

flag_context_df = load_raw_flag_context_sample(
    orientation_selected_paths,
    target_flag=FLAG_EXAMPLE_TARGET,
    classes=FLAG_EXAMPLE_CLASSES,
    context_rows_per_class=BASE_FLAG_EXAMPLE_POINTS_PER_PANEL,
    columns=orientation_columns,
)

print(
    {
        "orientation_raw_data_dir": RAW_DATA_DIR,
        "orientation_candidate_files": len(orientation_candidate_paths),
        "orientation_selected_files": len(orientation_selected_paths),
        "orientation_rows_loaded": len(exploration_df),
        "flag_context_rows_loaded": len(flag_context_df),
        "FLAG_EXAMPLE_TARGET": FLAG_EXAMPLE_TARGET,
        "FLAG_EXAMPLE_DISPLAY_NAME": FLAG_EXAMPLE_DISPLAY_NAME,
        "FLAG_EXAMPLE_AVOID_CONTEXT_LABELS": FLAG_EXAMPLE_AVOID_CONTEXT_LABELS,
        "ORIENTATION_ROWS_PER_FILE": ORIENTATION_ROWS_PER_FILE,
        "orientation_preview_columns": orientation_columns,
    }
)




In [ ]:
# Look at a few rows so the dataset description is tied to the actual table.
preview_candidates = [
    "Time UTC",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
    "Depth (m)",
    TARGET_FLAG,
    *OPTIONAL_QC_COLUMNS,
]
preview_columns = [column for column in preview_candidates if column in exploration_df.columns]
display(exploration_df[preview_columns].head(8))
print(
    {
        "time_start": exploration_df["Time UTC"].min().isoformat(),
        "time_end": exploration_df["Time UTC"].max().isoformat(),
        "preview_columns": preview_columns,
    }
)


### Seeing The Target Labels In Context

Before training anything, it helps to look at the time series directly.

The plots below show local windows centred on representative target-label values pulled directly from the raw CSV files. This is useful because:

- it connects the abstract label numbers to real sensor behaviour,
- it shows that some problematic labels appear in coherent stretches rather than isolated single rows,
- it reminds you that the model is trying to learn from patterns in time, not from labels in isolation.

If you want to make these panels wider or more annotated, the next config cell exposes:

- `FLAG_EXAMPLE_POINTS_PER_PANEL`
- `FLAG_EXAMPLE_REGION_ALPHA`
- `FLAG_EXAMPLE_SHOW_POINTS`
- `FLAG_EXAMPLE_CLASSES`


In [ ]:
FLAG_EXAMPLE_POINTS_PER_PANEL = 5_000 if DATASET_PROFILE_ID == "conductivity_plugs" else BASE_FLAG_EXAMPLE_POINTS_PER_PANEL
FLAG_EXAMPLE_REGION_ALPHA = 0.18
FLAG_EXAMPLE_SHOW_POINTS = DATASET_PROFILE_ID == "conductivity_plugs"

print(
    {
        "FLAG_EXAMPLE_POINTS_PER_PANEL": FLAG_EXAMPLE_POINTS_PER_PANEL,
        "FLAG_EXAMPLE_REGION_ALPHA": FLAG_EXAMPLE_REGION_ALPHA,
        "FLAG_EXAMPLE_SHOW_POINTS": FLAG_EXAMPLE_SHOW_POINTS,
        "FLAG_EXAMPLE_CLASSES": FLAG_EXAMPLE_CLASSES,
    }
)




In [ ]:
# Plot representative local windows for the main target-label values we see in this dataset.
timeseries_figure, timeseries_examples = plot_flag_examples(
    flag_context_df,
    target_flag=FLAG_EXAMPLE_TARGET,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
    avoid_context_labels=FLAG_EXAMPLE_AVOID_CONTEXT_LABELS,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    points_per_panel=FLAG_EXAMPLE_POINTS_PER_PANEL,
    classes=FLAG_EXAMPLE_CLASSES,
    good_labels=GOOD_LABELS,
    region_alpha=FLAG_EXAMPLE_REGION_ALPHA,
    show_flag_points=FLAG_EXAMPLE_SHOW_POINTS,
)
plt.show()
display(timeseries_examples)




## Part 2 — Data Preparation and Caching

This part explains how to turn raw ONC CSV exports into typed parquet tables that are much easier to reuse in ML experiments.


### 🧹 How We Prepare The Data

This is the part you can reuse most directly on your own data.

No matter which scalar dataset you start from, the same four questions show up:

1. what should count as one training example,
2. which fields are essential,
3. how should the raw values be cleaned and typed,
4. what saved format will be fast to reuse later.

In this notebook the raw preparation is handled by `scripts/prepare_scalar_session1_data.py`. We run that once up front because the original ONC CSV exports are great for provenance, but awkward for repeated ML experiments.


### Step-By-Step: Turning Raw Data Into ML-Ready Data

A good preparation pipeline is usually simpler than it first sounds.

**1. Decide what one example means**

For this notebook, one cleaned row is the basic reusable unit. Later sections can regroup those rows into windows when a model needs more temporal context.

**2. Keep an explicit list of the fields you need**

For scalar time series, that usually means:

- a reliable timestamp,
- one or more measurement columns,
- the target-label column,
- any extra context columns,
- and an identifier such as `source_file`.

**3. Clean and type the raw values**

In our prep script we:

- locate the real ONC header row and skip the metadata lines above it,
- parse `Time UTC` into typed datetimes,
- convert measurement and QC columns into numeric values,
- drop unusable timestamps,
- sort by time,
- and remember which file each row came from.

**4. Save reusable artifacts**

We keep two views of the same cleaned data:

- **row-level parquet**: one row per timestamp, useful for tabular models and sequence building,
- **window-summary parquet**: one row per fixed window, useful when you want short stretches summarised into one feature vector.

That is the core pattern to reuse on your own data: define the unit, clean it carefully, and save it in a form that later experiments can read quickly.


---


### 📦 Build Or Reuse The Parquet Cache

Most of the notebook uses a prepared parquet cache. Rebuilding it is optional when a shared or previously generated cache is already available.

When we do rebuild it, there are two separate outputs:

- `create_session1_row_level_parquet_cache(...)`: reads raw CSV files, cleans and types the rows, aligns companion device streams when needed, and writes row-level parquet parts.
- `create_session1_window_level_parquet_cache(...)`: reads those row-level parquet parts and writes one window-summary parquet file for k-means and short-window exploration.

For ordinary CSVs that only need a simple row-level cache, `session1_intro_utils.csv_files_to_parquet_cache(...)` is the smaller helper. It supports `header="auto"`, `header="first_row"`, or an integer header row such as `header=5`.


In [ ]:
RUN_RAW_PREP = False
FORCE_REBUILD_CACHE = False
PREP_MAX_FILES = None
PREP_SAMPLE_ROWS = DEFAULT_PREP_SAMPLE_ROWS
PREP_WINDOW_SIZE = 256
PREP_MERGE_TOLERANCE_SECONDS = 5
PREP_CACHE_ROOT = str(RUNTIME_CACHE_DIR)
PREP_CACHE_BUNDLE_NAME = CACHE_BUNDLE_NAME
PREP_PRIMARY_DEVICE = PRIMARY_DEVICE
PREP_ISSUE_LABELS = ISSUE_LABELS.copy()
PREP_MEASUREMENT_COLUMNS = MEASUREMENT_COLUMNS.copy()

print(
    {
        "RUN_RAW_PREP": RUN_RAW_PREP,
        "FORCE_REBUILD_CACHE": FORCE_REBUILD_CACHE,
        "PREP_MAX_FILES": PREP_MAX_FILES,
        "PREP_SAMPLE_ROWS": PREP_SAMPLE_ROWS,
        "PREP_WINDOW_SIZE": PREP_WINDOW_SIZE,
        "PREP_MERGE_TOLERANCE_SECONDS": PREP_MERGE_TOLERANCE_SECONDS,
        "PREP_CACHE_ROOT": PREP_CACHE_ROOT,
        "PREP_CACHE_BUNDLE_NAME": PREP_CACHE_BUNDLE_NAME,
        "PREP_PRIMARY_DEVICE": PREP_PRIMARY_DEVICE,
        "PREP_ISSUE_LABELS": PREP_ISSUE_LABELS,
        "PREP_MEASUREMENT_COLUMNS": PREP_MEASUREMENT_COLUMNS,
    }
)


In [ ]:
# The row-level cache keeps one cleaned row per timestamp.
# The window-level cache summarises those rows into fixed-size windows for k-means.
# PREP_CACHE_ROOT is where this notebook writes new cache files during this run.
runtime_cache_path = Path(PREP_CACHE_ROOT)
# Prefer a cache from this run, but fall back to READ_CACHE_DIR if a prepared cache already exists there.
existing_cache_paths = choose_cache_paths(
    [runtime_cache_path, Path(READ_CACHE_DIR)],
    # PREP_CACHE_BUNDLE_NAME names the row/window/metadata files for the active dataset.
    cache_stem=PREP_CACHE_BUNDLE_NAME,
)
cache_metadata_path = existing_cache_paths.metadata_path
missing_cache = not cache_metadata_path.exists()
# RUN_RAW_PREP rebuilds on demand; AUTO_BUILD_MISSING_CACHE keeps skip-ahead profiles usable.
should_run_prep = FORCE_REBUILD_CACHE or RUN_RAW_PREP or (AUTO_BUILD_MISSING_CACHE and missing_cache)

print(
    {
        "should_run_prep": should_run_prep,
        "cache_exists": cache_metadata_path.exists(),
        "auto_build_missing_cache": AUTO_BUILD_MISSING_CACHE,
        "raw_data_dir": RAW_DATA_DIR,
        "cache_bundle_name": PREP_CACHE_BUNDLE_NAME,
        "row_level_step": "raw CSV files -> typed row-level parquet parts",
        "window_level_step": "row-level parquet parts -> fixed-window summary parquet",
    }
)

if should_run_prep:
    row_cache_result = session1_intro_utils.create_session1_row_level_parquet_cache(
        # raw_data_dir: folder containing the source CSV files.
        raw_data_dir=RAW_DATA_DIR,
        # cache_root: output folder where the parquet bundle is written.
        cache_root=runtime_cache_path,
        # cache_bundle_name: shared filename stem for row parts, window file, and metadata JSON.
        cache_bundle_name=PREP_CACHE_BUNDLE_NAME,
        # target_flag: reviewed QC/custom label column used as the ML target later.
        target_flag=TARGET_FLAG,
        # primary_device: dataset-specific stream to treat as the main timestamp grid.
        primary_device=PREP_PRIMARY_DEVICE,
        # measurement_columns: sensor-value columns kept for plots and ML features.
        measurement_columns=PREP_MEASUREMENT_COLUMNS,
        # max_files: optional quick-run limit; None means read every CSV in RAW_DATA_DIR.
        max_files=PREP_MAX_FILES,
        # sample_rows: optional per-file row limit; None means keep all rows from each CSV.
        sample_rows=PREP_SAMPLE_ROWS,
        # merge_tolerance_seconds: maximum timestamp mismatch allowed when aligning companion streams.
        merge_tolerance_seconds=PREP_MERGE_TOLERANCE_SECONDS,
    )
    print(row_cache_result["summary"])

    window_cache_result = session1_intro_utils.create_session1_window_level_parquet_cache(
        # row_cache_result tells the window builder where the cleaned row-level parts were written.
        row_cache_result,
        # target_flag and issue_labels let each window store its issue rate.
        target_flag=TARGET_FLAG,
        issue_labels=PREP_ISSUE_LABELS,
        # window_size is the number of row-level timestamps summarised into one window.
        window_size=PREP_WINDOW_SIZE,
        # The remaining arguments keep the window metadata consistent with the row-level cache.
        sample_rows=PREP_SAMPLE_ROWS,
        merge_tolerance_seconds=PREP_MERGE_TOLERANCE_SECONDS,
        primary_device=PREP_PRIMARY_DEVICE,
        max_files=PREP_MAX_FILES,
    )
    print(window_cache_result["summary"])
elif missing_cache:
    print(
        "No prepared cache was found for this dataset profile. "
        "Point READ_CACHE_DIR at a prebuilt cache, or set RUN_RAW_PREP = True to build a reduced demo cache."
    )
else:
    print("Using the existing prepared cache. Set RUN_RAW_PREP = True if you want to rebuild it.")


### Where The Prep Code Lives

Use `session1_intro_utils.csv_files_to_parquet_cache(...)` when you just need a row-level parquet cache from one or more CSV files.

This workshop uses `scripts/prepare_scalar_session1_data.py` for the full cache because it also handles dataset-specific alignment and the window-summary parquet used later for clustering.

A useful reading order is:

1. `main()` for the overall flow,
2. `read_scalar_csv(...)` for row cleaning and typing,
3. `build_window_features(...)` for the window-summary cache.


---


### ⚡ Why The Parquet Cache Helps

Raw CSV is the source format. Parquet is the reusable analysis format.

This section shows four things:

- which cache bundle is active,
- which columns this notebook will read from the row-level cache,
- how the row-level and window-summary caches differ,
- how read time changes when we read all columns versus only the selected notebook columns.

For small, compact CSV samples, a full parquet read may not beat CSV. The practical benefit is that parquet keeps typed columns and supports **column-pruned reads**: loading only the timestamp, source file, target label, optional QC/context fields, and measurement columns needed by the next analysis step.


In [ ]:
cache_context = session1_intro_utils.show_session1_cache_inspection(
    raw_data_dir=RAW_DATA_DIR,
    runtime_cache_dir=RUNTIME_CACHE_DIR,
    read_cache_dir=READ_CACHE_DIR,
    cache_bundle_name=CACHE_BUNDLE_NAME,
    target_flag=TARGET_FLAG,
    measurement_columns=MEASUREMENT_COLUMNS,
    optional_qc_columns=OPTIONAL_QC_COLUMNS,
    plot_measurement_column=PLOT_MEASUREMENT_COLUMN,
    plot_secondary_column=PLOT_SECONDARY_COLUMN,
)

# Later cells use these resolved cache paths and active column lists.
globals().update(cache_context["notebook_values"])


In [ ]:
cache_read_result = session1_intro_utils.show_session1_cache_read_comparison(
    representative_raw_path=representative_raw_path,
    representative_parquet_path=representative_parquet_path,
    row_use_columns=ROW_USE_COLUMNS,
    sample_rows=RAW_CACHE_READ_SAMPLE_ROWS,
)


### Choosing A Useful Storage Format

Parquet is a strong fit for **typed tabular data** and many **1D time-series** workflows, but it is not the only good option.

A simple rule of thumb is: store the data in a format that matches the natural unit of your experiment.

- **Scalar tables / 1D time series**: parquet is often ideal because you can read only the columns you need.
- **Sequences that will be windowed later**: row-level parquet still works well because you can build windows on demand.
- **Images or 2D products** such as spectrograms: a folder of image files can be perfectly reasonable if each file is already one example.
- **Large 2D or 3D numeric arrays**: formats like NumPy `.npy` / `.npz`, HDF5, Zarr, or NetCDF are often a better fit than CSV.
- **Text examples**: JSONL and parquet are common because each row can hold one document plus labels and metadata.

So the real lesson is not "always use parquet." It is "pick a reusable format that preserves the structure of your examples and is efficient for the experiments you plan to run."


## Part 3 — Understanding and Defining the Reviewed Modelling Dataset

This part turns the prepared parquet cache into the data the models will use.

In this notebook, the **reviewed modelling dataset** means: the rows with usable reviewed labels, after any row-budget cap from `DATA_FRACTION`, plus the model target columns we create from those labels.

The **fixed split** means: the train / validation / test division that stays unchanged while we compare different models and train-only sampling choices.

The code uses the same vocabulary: `reviewed_model_df` is the labelled row dataset used for modelling, and `FIXED_SPLIT_STRATEGY` describes how those rows are divided into train, validation, and test.


### What This Part Is For

Up to this point, the data is **clean and stored efficiently**, but we have not yet turned it into an ML-ready reviewed dataset and evaluation split.

In this part you will:

- inspect the full selected parquet data,
- verify how reviewed flags are distributed through time,
- compare split strategies on the reviewed rows,
- keep validation/test fixed,
- and only then choose how much of the training split to keep.

This is an important transition point. A lot of ML performance and clarity comes from how you define the **dataset and evaluation protocol**, not just from which model you choose afterward.


### Phase 1 — Define The Reviewed Modelling Dataset

In this first phase, we use the prepared parquet cache to understand the reviewed data, compare fixed split strategies, and lock the train / validation / test split before we make any train-only runtime decisions.


---


In [ ]:
# Load the metadata that was written during cache preparation.
persistent_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)
active_cache_paths = choose_cache_paths([runtime_cache_dir, persistent_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

row_cache_dir = active_cache_paths.row_level_dir
window_cache_path = active_cache_paths.window_cache_path
metadata_path = active_cache_paths.metadata_path

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing cache metadata at {metadata_path}. Run scripts/prepare_scalar_session1_data.py first."
    )

metadata = json.loads(metadata_path.read_text())
part_paths = sorted(row_cache_dir.glob("*.parquet"))
if not part_paths:
    raise FileNotFoundError(f"No parquet parts found in {row_cache_dir}")

# Use the full prepared cache by default. DATA_FRACTION controls row budgets later.
selected_paths = part_paths
part_to_source = {
    Path(file_info["row_level_part"]).name: file_info["source_file"]
    for file_info in metadata["processed_files"]
}
part_to_row_count = {
    Path(file_info["row_level_part"]).name: int(file_info["row_count"])
    for file_info in metadata["processed_files"]
}
source_to_row_part = {
    file_info["source_file"]: row_cache_dir / Path(file_info["row_level_part"]).name
    for file_info in metadata["processed_files"]
}
selected_source_files = {part_to_source[path.name] for path in selected_paths}

print(
    {
        "active_cache_dir": str(active_cache_paths.root),
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "persistent_cache_dir": str(persistent_cache_dir),
        "runtime_output_root": str(RUNTIME_OUTPUT_ROOT),
        "model_output_dir": str(MODEL_OUTPUT_DIR),
        "selected_parts": len(selected_paths),
    }
)


In [ ]:
# Summarise the overall cache before loading row samples.
cache_summary = {
    "full_row_count": metadata["row_count"],
    "full_window_count": metadata["window_count"],
    "processed_file_count": metadata["processed_file_count"],
    "issue_fraction": round(float(metadata["issue_fraction"]), 4),
    "task_mode": TASK_MODE,
}
print(cache_summary)

selected_file_summary = pd.DataFrame(metadata["processed_files"])
selected_file_summary = selected_file_summary[selected_file_summary["source_file"].isin(selected_source_files)].reset_index(drop=True)
selected_file_summary = selected_file_summary[["source_file", "row_count", "time_start", "time_end"]].copy()
selected_file_summary["source_file"] = selected_file_summary["source_file"].str.replace(
    r"_\d{8}T\d{6}Z_\d{8}T\d{6}Z-NaN\.csv$",
    "",
    regex=True,
)
display(selected_file_summary.head(5))


#### 🚚 Use The Prepared Cache To Understand The Full Dataset

Parquet gives us fast access to the prepared dataset, so we can inspect selected data directly with only the columns we need.

The flow in this part is:

- inspect the prepared parquet data with only the columns we need,
- compare split strategies on reviewed rows,
- choose the fixed split,
- then choose a train-only subset strategy if you want to compare sampling approaches.

`DATA_FRACTION` controls the row budgets used when the reviewed modelling table and model training subsets are built.


#### 🎚️ Make The Row Budgets Visible

Large reviewed datasets can make live notebook runs slow. This section shows the exact row/window caps used to keep the workshop runnable.

`DATA_FRACTION` is the single top-level size control. Set `USE_DATA_FRACTION_BUDGETS = False` in the next cell to keep all reviewed rows and remove the per-section row/window caps. The advanced notebook has a few extra caps for model-search sections because those can be much more expensive than one baseline fit.

| Cap variable | Where it is used | How it is computed when caps are on | Why it exists |
|---|---|---|---|
| `REVIEWED_MODEL_ROW_LIMIT` | reviewed modelling dataframe | `FULL_REVIEWED_ROW_COUNT * DATA_FRACTION` | build a smaller reviewed dataframe before split comparison |
| `TRAIN_SUBSET_MAX_ROWS` | train-only subset comparison | `TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION`, with a minimum | compare train-subsampling strategies quickly |
| `ISSUE_ROWS_PER_FILE` | `issue_focused` train subset | `ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION`, with a minimum | control how many extra issue rows are added |
| `RF_TRAIN_MAX_ROWS` | Random Forest fit | `RF_TRAIN_BASE_ROWS * DATA_FRACTION`, with a minimum | keep the Random Forest baseline fast enough for a live run |
| `RF_VALIDATION_MAX_ROWS` / `RF_TEST_MAX_ROWS` | Random Forest scoring | `RF_EVAL_BASE_ROWS * DATA_FRACTION`, with a minimum | score representative validation/test rows without loading every row |
| `KMEANS_WINDOW_LIMIT` | k-means rows/windows | `KMEANS_WINDOW_BASE_ROWS * DATA_FRACTION`, with a minimum | keep clustering and example plots readable and fast |
| `RF_SEARCH_MODEL_ROW_LIMIT` | Random Forest search | `SEARCH_MODEL_BASE_ROWS * DATA_FRACTION`, with a minimum | test several RF settings without fitting on the full train split each time |
| `CNN_SEARCH_MODEL_ROW_LIMIT` | CNN search | `SEARCH_MODEL_BASE_ROWS * DATA_FRACTION`, with a minimum | test several CNN settings without building very large tensors |


In [ ]:
# Use every parquet part from the prepared cache. DATA_FRACTION limits rows later; it does not choose files.
SUMMARY_TIME_BIN_COUNT = 24

# Set this to False when you want full-data modelling instead of quicker workshop-sized runs.
USE_DATA_FRACTION_BUDGETS = True
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Count the rows that have one of the labels we will use for modelling.
# This lets us show the reviewed-row cap before we build the modelling dataframe.
target_distribution = {int(label): int(count) for label, count in metadata.get("target_distribution", {}).items()}
FULL_REVIEWED_ROW_COUNT = sum(
    target_distribution.get(int(label), 0)
    for label in [*GOOD_LABELS, *ISSUE_LABELS]
)

# Base caps are the row/window limits used when DATA_FRACTION = 1.0 but caps are still on.
# Minimums prevent very small DATA_FRACTION values from producing unusably tiny examples.
TRAIN_SUBSET_BASE_ROWS = 1_000_000
TRAIN_SUBSET_MIN_ROWS = 10_000
ISSUE_ROWS_PER_FILE_BASE = 12_000
ISSUE_ROWS_PER_FILE_MIN = 1_000
RF_TRAIN_BASE_ROWS = 250_000
RF_TRAIN_MIN_ROWS = 25_000
RF_EVAL_BASE_ROWS = 150_000
RF_EVAL_MIN_ROWS = 15_000
KMEANS_WINDOW_BASE_ROWS = 5_000
KMEANS_WINDOW_MIN_ROWS = 2_000
SEARCH_MODEL_BASE_ROWS = 500_000
SEARCH_MODEL_MIN_ROWS = 200_000

if USE_DATA_FRACTION_BUDGETS and DATA_FRACTION < 0.999:
    BUDGET_MODE = "DATA_FRACTION row/window caps"
    BUDGET_DATA_FRACTION = DATA_FRACTION
    # This is the main dataset-size control: keep this fraction of usable reviewed rows.
    REVIEWED_MODEL_ROW_LIMIT = max(1, int(round(FULL_REVIEWED_ROW_COUNT * DATA_FRACTION)))
    # The remaining caps keep specific model/demo sections fast and memory-safe.
    TRAIN_SUBSET_MAX_ROWS = max(TRAIN_SUBSET_MIN_ROWS, int(TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION))
    ISSUE_ROWS_PER_FILE = max(ISSUE_ROWS_PER_FILE_MIN, int(ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION))
    RF_TRAIN_MAX_ROWS = max(RF_TRAIN_MIN_ROWS, int(RF_TRAIN_BASE_ROWS * DATA_FRACTION))
    RF_VALIDATION_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * DATA_FRACTION))
    RF_TEST_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * DATA_FRACTION))
    KMEANS_WINDOW_LIMIT = max(KMEANS_WINDOW_MIN_ROWS, int(KMEANS_WINDOW_BASE_ROWS * DATA_FRACTION))
    RF_SEARCH_MODEL_ROW_LIMIT = max(SEARCH_MODEL_MIN_ROWS, int(SEARCH_MODEL_BASE_ROWS * DATA_FRACTION))
    CNN_SEARCH_MODEL_ROW_LIMIT = max(SEARCH_MODEL_MIN_ROWS, int(SEARCH_MODEL_BASE_ROWS * DATA_FRACTION))
else:
    BUDGET_MODE = "full reviewed data / no row caps"
    BUDGET_DATA_FRACTION = 1.0
    REVIEWED_MODEL_ROW_LIMIT = FULL_REVIEWED_ROW_COUNT
    TRAIN_SUBSET_MAX_ROWS = None
    ISSUE_ROWS_PER_FILE = ISSUE_ROWS_PER_FILE_BASE
    RF_TRAIN_MAX_ROWS = None
    RF_VALIDATION_MAX_ROWS = None
    RF_TEST_MAX_ROWS = None
    KMEANS_WINDOW_LIMIT = None
    RF_SEARCH_MODEL_ROW_LIMIT = None
    CNN_SEARCH_MODEL_ROW_LIMIT = None

print(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "USE_DATA_FRACTION_BUDGETS": USE_DATA_FRACTION_BUDGETS,
        "BUDGET_MODE": BUDGET_MODE,
        "FULL_REVIEWED_ROW_COUNT": FULL_REVIEWED_ROW_COUNT,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_VALIDATION_MAX_ROWS": RF_VALIDATION_MAX_ROWS,
        "RF_TEST_MAX_ROWS": RF_TEST_MAX_ROWS,
        "SUMMARY_TIME_BIN_COUNT": SUMMARY_TIME_BIN_COUNT,
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
        "RF_SEARCH_MODEL_ROW_LIMIT": RF_SEARCH_MODEL_ROW_LIMIT,
        "CNN_SEARCH_MODEL_ROW_LIMIT": CNN_SEARCH_MODEL_ROW_LIMIT,
    }
)


**Helper: validation adequacy**

Before training anything, we want to know whether validation contains enough flagged data to make model selection meaningful.

The helper below summarises that question for any candidate split.


In [ ]:
def summarize_issue_adequacy(
    split_frames: dict[str, pd.DataFrame],
    *,
    label_column: str,
    issue_labels: list[int] | tuple[int, ...],
    min_issue_rows: int,
    min_issue_share_pct: float,
    min_rows_per_issue_label: int,
) -> pd.DataFrame:
    adequacy_rows = []
    normalized_issue_labels = [int(label) for label in issue_labels]

    for split_name, frame in split_frames.items():
        labels = pd.to_numeric(frame[label_column], errors="coerce").dropna().astype(int)
        total_rows = int(len(labels))
        label_counts = labels.value_counts().sort_index()
        issue_counts = label_counts.reindex(normalized_issue_labels, fill_value=0).astype(int)
        issue_rows = int(issue_counts.sum())
        issue_share_pct = round(100 * issue_rows / total_rows, 2) if total_rows else 0.0
        per_label_ok = all(int(issue_counts.get(label, 0)) >= min_rows_per_issue_label for label in normalized_issue_labels)

        adequacy_rows.append(
            {
                "split": split_name,
                "rows": total_rows,
                "issue_rows": issue_rows,
                "issue_share_pct": issue_share_pct,
                "meets_issue_row_floor": issue_rows >= min_issue_rows,
                "meets_issue_share_floor": issue_share_pct >= min_issue_share_pct,
                "meets_per_issue_label_floor": per_label_ok,
                "adequate_for_model_selection": (
                    issue_rows >= min_issue_rows
                    and issue_share_pct >= min_issue_share_pct
                    and per_label_ok
                ),
                **{f"flag_{label}_count": int(issue_counts.get(label, 0)) for label in normalized_issue_labels},
            }
        )

    return pd.DataFrame(adequacy_rows).set_index("split")


def build_split_strategy_tables(
    strategy_metric_rows: list[dict[str, object]],
    strategy_adequacy_frames: list[pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_frame = pd.DataFrame(strategy_metric_rows).set_index("strategy").copy()
    summary_frame = summary_frame.rename(
        columns={
            "rows": "reviewed_rows_total",
            "train_rows": "train_rows",
            "validation_rows": "validation_rows",
            "test_rows": "test_rows",
            "validation_issue_rows": "validation_issue_rows",
            "validation_issue_share_pct": "validation_issue_share_pct",
            "validation_adequate": "validation_ok",
            "max_pairwise_total_variation": "max_split_gap",
            "mean_pairwise_total_variation": "mean_split_gap",
        }
    )
    if "validation_ok" in summary_frame.columns:
        summary_frame["validation_ok"] = summary_frame["validation_ok"].map({True: "yes", False: "no"})

    detail_frame = pd.concat(strategy_adequacy_frames, ignore_index=True).copy()
    if detail_frame.empty:
        return summary_frame, detail_frame

    detail_frame["adequate_for_model_selection"] = detail_frame["adequate_for_model_selection"].map(
        {True: "yes", False: "no"}
    )
    detail_frame = detail_frame.rename(
        columns={
            "adequate_for_model_selection": "adequate",
        }
    )
    pivot_frame = (
        detail_frame.set_index(["strategy", "split"])[["rows", "issue_rows", "issue_share_pct", "adequate"]]
        .unstack("split")
        .swaplevel(0, 1, axis=1)
        .sort_index(axis=1, level=0)
    )
    pivot_frame.columns = [f"{split}_{metric}" for split, metric in pivot_frame.columns]
    pivot_frame = pivot_frame.rename(
        columns={
            "train_rows": "train_rows",
            "train_issue_rows": "train_issue_rows",
            "train_issue_share_pct": "train_issue_share_pct",
            "train_adequate": "train_ok",
            "validation_rows": "validation_rows",
            "validation_issue_rows": "validation_issue_rows",
            "validation_issue_share_pct": "validation_issue_share_pct",
            "validation_adequate": "validation_ok",
            "test_rows": "test_rows",
            "test_issue_rows": "test_issue_rows",
            "test_issue_share_pct": "test_issue_share_pct",
            "test_adequate": "test_ok",
        }
    )
    return summary_frame, pivot_frame


In [ ]:
# Load only the columns needed for label accounting and split design.
# We do not need the full feature table yet.
overview_columns = list(dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG]))
dataset_label_df = load_full_row_level_frame(selected_paths, columns=overview_columns)
dataset_label_df[TARGET_FLAG] = pd.to_numeric(dataset_label_df[TARGET_FLAG], errors="coerce")
if "source_file" in dataset_label_df.columns:
    dataset_label_df["source_file"] = dataset_label_df["source_file"].astype("category")

if KMEANS_FEATURE_MODE == "window_summary":
    # Load cached window summaries for clustering when averaging and variability are the useful signal.
    window_df = pd.read_parquet(window_cache_path, columns=WINDOW_USE_COLUMNS)
    window_df["window_start"] = pd.to_datetime(window_df["window_start"], utc=True)
    window_df["window_end"] = pd.to_datetime(window_df["window_end"], utc=True)
    window_df = (
        window_df[window_df["source_file"].isin(selected_source_files)]
        .sort_values("window_start")
        .reset_index(drop=True)
    )
    window_limit = KMEANS_WINDOW_LIMIT
    window_df = evenly_spaced_take(window_df, window_limit)
elif KMEANS_FEATURE_MODE == "row_level":
    # Row-level k-means builds its feature frame later from the reviewed modelling rows.
    window_df = pd.DataFrame()
    window_limit = None
else:
    raise ValueError(f"Unsupported KMEANS_FEATURE_MODE: {KMEANS_FEATURE_MODE}")

print(
    {
        "selected_parts": len(selected_paths),
        "selected_rows": len(dataset_label_df),
        "loaded_windows": len(window_df),
        "data_fraction": DATA_FRACTION,
        "kmeans_feature_mode": KMEANS_FEATURE_MODE,
        "overview_columns_loaded": overview_columns,
        "window_columns_loaded": len(WINDOW_USE_COLUMNS) if KMEANS_FEATURE_MODE == "window_summary" else None,
        "window_limit": window_limit,
    }
)


In [ ]:
reviewed_model_accounting = session1_intro_utils.show_reviewed_model_row_accounting(
    dataset_label_df,
    target_flag=TARGET_FLAG,
    task_mode=TASK_MODE,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,  # Explicitly computed in the visible row-budget cell above.
    data_fraction=BUDGET_DATA_FRACTION,
    flag_palette=DEFAULT_FLAG_PALETTE,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
)

# reviewed_label_df keeps every usable reviewed row.
# reviewed_model_df applies DATA_FRACTION and is the dataframe used for split comparison below.
reviewed_label_df = reviewed_model_accounting["reviewed_label_df"]
reviewed_model_df = reviewed_model_accounting["reviewed_model_df"]
active_labels = reviewed_model_accounting["active_labels"]
model_good_labels = reviewed_model_accounting["model_good_labels"]
model_issue_labels = reviewed_model_accounting["model_issue_labels"]
selected_target_counts = reviewed_model_accounting["selected_target_counts"]
selected_reviewed_counts = reviewed_model_accounting["selected_reviewed_counts"]
reviewed_model_counts = reviewed_model_accounting["reviewed_model_counts"]
row_accounting_summary = reviewed_model_accounting["row_summary"]
REVIEWED_MODEL_ROW_LIMIT = reviewed_model_accounting["summary"]["reviewed_model_row_limit"]


#### Verify How The Flags Change Across Time

Before choosing a split strategy, look at **where the reviewed modelling labels live on the time axis**.

This is one of the most important dataset checks you can do before any machine learning:

- if issue labels only appear in one short period, a strict contiguous split may put almost all of them into only one split,
- if the label mix changes gradually across time, that may reflect a real regime shift rather than a sampling mistake,
- if different labels appear in different time periods, you should expect the split strategy to change the difficulty of the task.

This is why dataset preparation is never just "convert files and start training." You also need to **verify the label distribution, the time coverage, and the assumptions behind your split** before trusting any model result.

The plots below are not model outputs. They are a dataset sanity check on the reviewed modelling rows before we define train / validation / test.


In [ ]:
temporal_summary_bundle = session1_intro_utils.show_temporal_flag_summary(
    reviewed_model_df,
    target_flag=TARGET_FLAG,
    selected_path_count=len(selected_paths),
    temporal_summary_bin_count=SUMMARY_TIME_BIN_COUNT,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
)
temporal_bin_count = temporal_summary_bundle["bin_count"]
temporal_labels = temporal_summary_bundle["labels"]
temporal_counts = temporal_summary_bundle["counts"]
temporal_shares = temporal_summary_bundle["shares"]
temporal_issue_only_shares = temporal_summary_bundle["issue_only_shares"]
temporal_summary = temporal_summary_bundle["summary"]

print(
    {
        "temporal_bin_count": temporal_bin_count,
        "time_bins_shown": len(temporal_summary),
        "issue_labels": ISSUE_LABELS,
        "max_issue_share_pct": float(temporal_summary["issue_share_pct"].max()) if len(temporal_summary) else 0.0,
        "min_issue_share_pct": float(temporal_summary["issue_share_pct"].min()) if len(temporal_summary) else 0.0,
    }
)


#### Split Strategy Comparison Settings

These settings control the split-strategy comparison in the next cells.

The train / validation / test split changes the **question** you are asking:

- `global_contiguous`: one early / middle / late cut across the whole time axis. Use this when you want the strictest deployment-style test of temporal generalisation.
- `per_source_contiguous`: make an early / middle / late cut inside each source file, then combine the pieces. Use this when files represent distinct deployments, maintenance periods, or acquisition sessions that should each contribute to every split.
- `interleaved_blocks`: keep short local time blocks intact, but distribute those blocks across train / validation / test. Use this when interesting events are sparse and a single long cut would leave one split with almost none of them.

One important rule: **never artificially balance validation or test**. Those splits should reflect the real distribution that falls out of the chosen strategy.

So the goal here is not to inflate issue labels in validation. The goal is to choose a split that is both scientifically meaningful **and** leaves validation with enough naturally occurring issue examples to support model selection.


In [ ]:
SUPPORTED_SPLIT_STRATEGIES = (
    "global_contiguous",
    "per_source_contiguous",
    "interleaved_blocks",
)

AUTO_SELECT_SPLIT_BLOCK_ROWS = True
SPLIT_BLOCK_ROWS = 1024
SPLIT_BLOCK_ROW_CANDIDATES = (512, 1024, 2048, 4096)
VALIDATION_MIN_ISSUE_ROWS = 100
VALIDATION_MIN_ISSUE_SHARE_PCT = 1.0
VALIDATION_MIN_ROWS_PER_ISSUE_LABEL = 20

show_setup_json(
    {
        "AUTO_SELECT_SPLIT_BLOCK_ROWS": AUTO_SELECT_SPLIT_BLOCK_ROWS,
        "SPLIT_BLOCK_ROWS": SPLIT_BLOCK_ROWS,
        "SPLIT_BLOCK_ROW_CANDIDATES": SPLIT_BLOCK_ROW_CANDIDATES,
        "VALIDATION_MIN_ISSUE_ROWS": VALIDATION_MIN_ISSUE_ROWS,
        "VALIDATION_MIN_ISSUE_SHARE_PCT": VALIDATION_MIN_ISSUE_SHARE_PCT,
        "VALIDATION_MIN_ROWS_PER_ISSUE_LABEL": VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
    }
)


#### Compare Split Strategies With Tables

This comparison uses `reviewed_model_df`: the usable reviewed rows after applying the optional `DATA_FRACTION` row budget.

That keeps the split comparison aligned with the fixed split that the model sections will actually use.

The tables below show two things:

- a compact one-row summary for each strategy,
- and a side-by-side train / validation / test breakdown so you can compare issue coverage without scrolling through a long stacked table.


In [ ]:
comparison_labels = [
    label
    for label in sorted(set(model_good_labels).union(model_issue_labels))
    if label in set(reviewed_model_df["model_target"].dropna().astype(int).unique().tolist())
]
present_issue_labels = [label for label in model_issue_labels if label in comparison_labels]

block_scan_frame = scan_interleaved_block_rows(
    reviewed_model_df,
    label_column="model_target",
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    candidate_block_rows=SPLIT_BLOCK_ROW_CANDIDATES,
)
ACTIVE_SPLIT_BLOCK_ROWS = (
    int(block_scan_frame.iloc[0]["block_rows"])
    if AUTO_SELECT_SPLIT_BLOCK_ROWS and len(block_scan_frame)
    else int(SPLIT_BLOCK_ROWS)
)

if len(block_scan_frame):
    display(block_scan_frame)

split_strategy_labels = {
    "global_contiguous": "Global contiguous",
    "per_source_contiguous": "Per-source contiguous",
    "interleaved_blocks": "Interleaved blocks",
    "episode_aware": "Episode-aware blocks",
}

strategy_metric_rows = []
strategy_adequacy_frames = []
strategy_frames_by_name = {}

for strategy_name in ["global_contiguous", "per_source_contiguous", "interleaved_blocks"]:
    strategy_frames = split_frame_by_strategy(
        reviewed_model_df,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        strategy=strategy_name,
        block_rows=ACTIVE_SPLIT_BLOCK_ROWS,
    )
    strategy_frames_by_name[strategy_name] = strategy_frames
    strategy_counts, strategy_shares = summarize_split_distributions(
        strategy_frames,
        label_column="model_target",
    )
    adequacy_frame = summarize_issue_adequacy(
        strategy_frames,
        label_column="model_target",
        issue_labels=present_issue_labels,
        min_issue_rows=VALIDATION_MIN_ISSUE_ROWS,
        min_issue_share_pct=VALIDATION_MIN_ISSUE_SHARE_PCT,
        min_rows_per_issue_label=VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
    )
    adequacy_frame = adequacy_frame.reset_index()
    adequacy_frame.insert(0, "strategy", split_strategy_labels[strategy_name])
    strategy_adequacy_frames.append(adequacy_frame)

    gap_metrics = compute_split_share_gap(strategy_shares)
    validation_row = adequacy_frame[adequacy_frame["split"] == "validation"].iloc[0]
    strategy_metric_rows.append(
        {
            "strategy": split_strategy_labels[strategy_name],
            "rows": int(strategy_counts.sum().sum()),
            "train_rows": int(strategy_counts["train"].sum()),
            "validation_rows": int(strategy_counts["validation"].sum()),
            "test_rows": int(strategy_counts["test"].sum()),
            "validation_issue_rows": int(validation_row["issue_rows"]),
            "validation_issue_share_pct": float(validation_row["issue_share_pct"]),
            "validation_adequate": bool(validation_row["adequate_for_model_selection"]),
            **gap_metrics,
        }
    )

strategy_metric_frame, strategy_split_detail = build_split_strategy_tables(
    strategy_metric_rows,
    strategy_adequacy_frames,
)
display(
    strategy_metric_frame.style.format(
        {
            "reviewed_rows_total": "{:,.0f}",
            "train_rows": "{:,.0f}",
            "validation_rows": "{:,.0f}",
            "test_rows": "{:,.0f}",
            "validation_issue_rows": "{:,.0f}",
            "validation_issue_share_pct": "{:.2f}",
            "max_split_gap": "{:.4f}",
            "mean_split_gap": "{:.4f}",
        }
    )
)
display(
    strategy_split_detail.style.format(
        {
            "train_rows": "{:,.0f}",
            "train_issue_rows": "{:,.0f}",
            "train_issue_share_pct": "{:.2f}",
            "validation_rows": "{:,.0f}",
            "validation_issue_rows": "{:,.0f}",
            "validation_issue_share_pct": "{:.2f}",
            "test_rows": "{:,.0f}",
            "test_issue_rows": "{:,.0f}",
            "test_issue_share_pct": "{:.2f}",
        }
    )
)


**Visual comparison**

Now that you have seen the label drift through time, we can ask a more focused question: **how does the split itself change the balance?**

We compare three strategies on the same reviewed modelling dataset:

- **Global contiguous**: the earliest rows go to train, the middle rows go to validation, and the latest rows go to test.
- **Per-source contiguous**: each source file is split early / middle / late, then the per-file pieces are combined.
- **Interleaved blocks**: short local time blocks stay intact, but the blocks are distributed across train / validation / test in a repeating schedule.

The timeline plot shows where each split lands across the selected time period covered by the reviewed modelling rows. The composition plot then compares the label balance for those same split assignments. The choice cell comes immediately after that comparison.


In [ ]:
strategy_timeline_bundle = show_split_strategy_timeline(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    figure_title="Where each split strategy places train / validation / test over time",
)
strategy_timeline_summary = strategy_timeline_bundle["summary"]

strategy_plot_bundle = show_split_strategy_comparison(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    issue_labels=present_issue_labels,
    label_column="model_target",
    flag_palette=DEFAULT_FLAG_PALETTE,
    legend_title="model target",
    figure_title="How the split strategy changes label balance on the reviewed modelling dataset",
)
strategy_shares_by_name = strategy_plot_bundle["shares"]
strategy_issue_only_shares_by_name = strategy_plot_bundle["issue_only_shares"]

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "reviewed_model_rows": len(reviewed_model_df),
        "ACTIVE_SPLIT_BLOCK_ROWS": ACTIVE_SPLIT_BLOCK_ROWS,
        "AUTO_SELECT_SPLIT_BLOCK_ROWS": AUTO_SELECT_SPLIT_BLOCK_ROWS,
        "present_issue_labels": present_issue_labels,
        "validation_is_rebalanced": False,
    }
)


**Questions to ask before you lock the split**

Before choosing a fixed split, pause and look for structure that a simple label-balance table cannot show on its own:

1. Are the issue flags isolated points, or do they cluster into longer episodes?
2. Could one real bad-data event be getting cut across train, validation, and test?
3. If the model uses windows or lagged context, would nearby timestamps leak information across split boundaries?
4. Does this split answer the question you care about: generalize to nearby conditions, or generalize to a completely new bad episode?

Keep those questions in mind as you pick the split strategy below. The advanced notebook comes back to them with a stronger episode-aware fixed split.


#### Choose And Build The Fixed Split

This block chooses the fixed train / validation / test split that every later model section will reuse.

The editable choices stay visible below. The utility helpers handle the repetitive split comparison, dataframe construction, target creation, and plotting.


In [ ]:
# SPLIT_STRATEGY is the basic split we compare against the episode-aware option.
# Options: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
SPLIT_STRATEGY = "global_contiguous"
if SPLIT_STRATEGY not in SUPPORTED_SPLIT_STRATEGIES:
    raise ValueError(f"Unsupported split strategy: {SPLIT_STRATEGY}. Choose from {SUPPORTED_SPLIT_STRATEGIES}.")

# USE_EPISODE_AWARE_FIXED_SPLIT controls the final split used by the model sections.
# True: keep issue episodes and their context together. False: use SPLIT_STRATEGY directly.
USE_EPISODE_AWARE_FIXED_SPLIT = True

# Episode-aware controls used only when USE_EPISODE_AWARE_FIXED_SPLIT is True.
# EPISODE_CLEAN_BLOCK_ROWS: block size for ordinary clean/background stretches.
# EPISODE_CONTEXT_ROWS: clean rows kept around each issue episode.
# EPISODE_MERGE_GAP_ROWS: maximum gap for merging nearby issue stretches into one episode.
# EPISODE_PURGE_GAP_ROWS: buffer rows removed near split boundaries to reduce leakage.
EPISODE_CLEAN_BLOCK_ROWS = ACTIVE_SPLIT_BLOCK_ROWS
EPISODE_CONTEXT_ROWS = 512
EPISODE_MERGE_GAP_ROWS = 128
EPISODE_PURGE_GAP_ROWS = 256

# REVIEWED_MODEL_ROW_LIMIT was computed from DATA_FRACTION and the usable reviewed row count above.
# With DATA_FRACTION = 1.0, this equals the full usable reviewed count.

# These are the final split settings passed to the build helper below.
FIXED_SPLIT_STRATEGY = "episode_aware" if USE_EPISODE_AWARE_FIXED_SPLIT else SPLIT_STRATEGY
FIXED_SPLIT_BLOCK_ROWS = EPISODE_CLEAN_BLOCK_ROWS if USE_EPISODE_AWARE_FIXED_SPLIT else ACTIVE_SPLIT_BLOCK_ROWS

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "SPLIT_STRATEGY": SPLIT_STRATEGY,
        "USE_EPISODE_AWARE_FIXED_SPLIT": USE_EPISODE_AWARE_FIXED_SPLIT,
        "FIXED_SPLIT_STRATEGY": FIXED_SPLIT_STRATEGY,
        "FIXED_SPLIT_BLOCK_ROWS": FIXED_SPLIT_BLOCK_ROWS,
        "EPISODE_CONTEXT_ROWS": EPISODE_CONTEXT_ROWS,
        "EPISODE_MERGE_GAP_ROWS": EPISODE_MERGE_GAP_ROWS,
        "EPISODE_PURGE_GAP_ROWS": EPISODE_PURGE_GAP_ROWS,
        "validation_is_rebalanced": False,
    }
)


In [ ]:
# Compare the selected basic split with the episode-aware split before building the final frames.
advanced_split_comparison = session1_intro_utils.show_episode_aware_split_comparison(
    # reviewed_model_df: reviewed rows after the DATA_FRACTION budget, used for split diagnostics.
    reviewed_label_df=reviewed_model_df,
    # base_strategy: the basic split selected above for comparison.
    base_strategy=SPLIT_STRATEGY,
    base_strategy_label=split_strategy_labels.get(SPLIT_STRATEGY, SPLIT_STRATEGY),
    # train_fraction / validation_fraction: test receives the remaining fraction.
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    # Block and episode controls define how local time chunks are assigned.
    active_split_block_rows=ACTIVE_SPLIT_BLOCK_ROWS,
    episode_clean_block_rows=EPISODE_CLEAN_BLOCK_ROWS,
    episode_context_rows=EPISODE_CONTEXT_ROWS,
    episode_merge_gap_rows=EPISODE_MERGE_GAP_ROWS,
    episode_purge_gap_rows=EPISODE_PURGE_GAP_ROWS,
    # Label inputs define which classes are plotted and counted as issues.
    target_flag="model_target",
    issue_labels=present_issue_labels,
    comparison_labels=comparison_labels,
    flag_palette=DEFAULT_FLAG_PALETTE,
    # Validation thresholds are diagnostics only; validation/test rows are not rebalanced.
    validation_min_issue_rows=VALIDATION_MIN_ISSUE_ROWS,
    validation_min_issue_share_pct=VALIDATION_MIN_ISSUE_SHARE_PCT,
    validation_min_rows_per_issue_label=VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
)

advanced_split_candidates = advanced_split_comparison["split_candidates"]
advanced_metric_frame = advanced_split_comparison["metric_frame"]
advanced_split_detail = advanced_split_comparison["split_detail"]


#### Build The Fixed Split

The next helper loads the reviewed rows, creates `issue` and `model_target`, applies the selected split, and restores the dataframe names used by the model sections.


In [ ]:
reviewed_split_bundle = session1_intro_utils.build_reviewed_modelling_split(
    # selected_paths: row-level parquet parts selected from the active cache earlier in Part 3.
    selected_paths=selected_paths,
    # target_flag: the reviewed QC/custom label column we want to learn from.
    target_flag=TARGET_FLAG,
    # task_mode: "multiclass" keeps label values; "binary" converts to issue vs not-issue.
    task_mode=TASK_MODE,
    # good_labels / issue_labels: define which reviewed labels become usable modelling rows.
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    # model_row_limit: optional row cap derived from DATA_FRACTION for faster live runs.
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    # train_fraction / validation_fraction: test receives the remaining fraction.
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    # split_strategy / split_block_rows: final settings from the control cell above.
    split_strategy=FIXED_SPLIT_STRATEGY,
    split_block_rows=FIXED_SPLIT_BLOCK_ROWS,
    # episode_* controls only affect split_strategy="episode_aware"; other strategies ignore them.
    episode_context_rows=EPISODE_CONTEXT_ROWS,
    episode_merge_gap_rows=EPISODE_MERGE_GAP_ROWS,
    purge_gap_rows=EPISODE_PURGE_GAP_ROWS,
    # measurement_columns: kept with the returned metadata for later model sections.
    measurement_columns=MEASUREMENT_COLUMNS,
)

# Restore the standard dataframe names used by the ML sections:
# reviewed_model_df, train_full_df, valid_df, test_df, model_good_labels, model_issue_labels, etc.
globals().update(reviewed_split_bundle["notebook_values"])
session1_intro_utils.show_reviewed_modelling_split_build(reviewed_split_bundle)


#### Review The Fixed Split

The review helper shows the split sizes, issue coverage, time span, full label mix, issue-only label mix, and the advanced validation issue-coverage diagnostic.


In [ ]:
fixed_split_review = session1_intro_utils.show_fixed_split_review(
    # split_frames: the exact train / validation / test dataframes all model sections reuse.
    {"train": train_full_df, "validation": valid_df, "test": test_df},
    # issue_labels: which model_target values count as issue classes in the issue-only plot.
    issue_labels=model_issue_labels,
    # split_strategy_label: human-readable title for the plots and tables.
    split_strategy_label=split_strategy_labels.get(FIXED_SPLIT_STRATEGY, FIXED_SPLIT_STRATEGY),
    # flag_palette: consistent colours for label values across the notebook.
    flag_palette=DEFAULT_FLAG_PALETTE,
    # The next three inputs are diagnostics only; they do not change the split.
    min_issue_rows=VALIDATION_MIN_ISSUE_ROWS,
    min_issue_share_pct=VALIDATION_MIN_ISSUE_SHARE_PCT,
    min_rows_per_issue_label=VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
)

# Keep the displayed review objects available in case later cells or participants want to inspect them.
split_overview = fixed_split_review["overview"]
split_counts = fixed_split_review["counts"]
split_shares = fixed_split_review["shares"]
split_issue_only_shares = fixed_split_review["issue_only_shares"]
split_adequacy = fixed_split_review.get("adequacy")
validation_adequacy = fixed_split_review.get("validation_adequacy")


### Phase 2 — Design The Train-Only Subset

The reviewed modelling dataset is fixed now. From this point on, validation and test stay unchanged. The next decisions only affect **which training rows** we hand to the models.


#### Training Subset Comparison Settings

The fixed split is fixed now. `DATA_FRACTION` controls the train-only row budget used in the next block, while `TRAIN_SUBSET_STRATEGY` later chooses *which* training rows are kept.

`BALANCED_REVIEWED_TARGET_ISSUE_SHARE` only matters for `balanced_reviewed`. It is a target, not a guarantee: if the full training split does not contain enough issue rows, the helper keeps all available issue rows and fills the remaining row budget with good rows.

Validation and test stay untouched so the comparison stays fair.


In [ ]:
SUPPORTED_TRAIN_SUBSET_STRATEGIES = ("full_train", "time_spread", "balanced_reviewed", "issue_focused")
# TRAIN_SUBSET_MAX_ROWS and ISSUE_ROWS_PER_FILE were set in the visible row-budget cell above.
# Set TRAIN_SUBSET_MAX_ROWS = None here if you want this section to use the full train split.
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25
if not 0 < BALANCED_REVIEWED_TARGET_ISSUE_SHARE < 1:
    raise ValueError("BALANCED_REVIEWED_TARGET_ISSUE_SHARE must be in the interval (0, 1).")

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "ISSUE_ROWS_PER_FILE": ISSUE_ROWS_PER_FILE,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    }
)


#### Compare Train-Only Subset Strategies

These next cells stay focused on the training set only. Validation and test do not change here.

First we summarise the candidate train-only subsets. Then we visualise how the target mix changes inside train.


In [ ]:
train_subset_preview_columns = ["source_file", "Time UTC", "model_target"]
train_subset_preview_source = train_full_df[train_subset_preview_columns].copy()
train_subset_rows_limit = (
    len(train_full_df)
    if TRAIN_SUBSET_MAX_ROWS is None
    else min(int(TRAIN_SUBSET_MAX_ROWS), len(train_full_df))
)

available_train_issue_rows = int(train_subset_preview_source["model_target"].isin(model_issue_labels).sum())
max_possible_balanced_issue_share = min(available_train_issue_rows, train_subset_rows_limit) / max(train_subset_rows_limit, 1)

train_subset_preview_frames = {"full_train": train_subset_preview_source}
for subset_strategy in ("time_spread", "balanced_reviewed", "issue_focused"):
    train_subset_preview_frames[subset_strategy] = sample_frame_by_strategy(
        train_subset_preview_source,
        rows_limit=train_subset_rows_limit,
        sample_strategy=subset_strategy,
        target_flag="model_target",
        good_labels=GOOD_LABELS,
        issue_labels=model_issue_labels,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )

train_subset_labels = {
    "full_train": "full train",
    "time_spread": "time-spread subset",
    "balanced_reviewed": (
        f"balanced-reviewed subset ({BALANCED_REVIEWED_TARGET_ISSUE_SHARE:.0%} target; "
        f"max {max_possible_balanced_issue_share:.1%})"
    ),
    "issue_focused": "issue-focused subset",
}

train_subset_overview_rows = []
for subset_name, subset_frame in train_subset_preview_frames.items():
    subset_counts = subset_frame["model_target"].value_counts().sort_index()
    issue_rows = int(subset_counts.reindex(model_issue_labels, fill_value=0).sum())
    train_subset_overview_rows.append(
        {
            "subset": train_subset_labels[subset_name],
            "rows": len(subset_frame),
            "issue_rows": issue_rows,
            "issue_share_pct": round(100 * issue_rows / max(len(subset_frame), 1), 2),
            "target_issue_share_pct": (
                round(100 * BALANCED_REVIEWED_TARGET_ISSUE_SHARE, 2)
                if subset_name == "balanced_reviewed"
                else None
            ),
            "max_possible_issue_share_pct": round(
                100 * min(available_train_issue_rows, len(subset_frame)) / max(len(subset_frame), 1),
                2,
            ),
            "time_start": subset_frame["Time UTC"].min().isoformat(),
            "time_end": subset_frame["Time UTC"].max().isoformat(),
        }
    )
train_subset_overview = pd.DataFrame(train_subset_overview_rows).set_index("subset")

train_budget_counts = pd.DataFrame(
    {
        subset_name: subset_frame["model_target"].value_counts().sort_index()
        for subset_name, subset_frame in train_subset_preview_frames.items()
    }
).fillna(0).astype(int)
train_budget_shares = train_budget_counts.div(train_budget_counts.sum(axis=0), axis=1).fillna(0.0)
display(
    train_subset_overview.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
            "target_issue_share_pct": "{:.2f}",
            "max_possible_issue_share_pct": "{:.2f}",
        },
        na_rep="-",
    )
)


In [ ]:
train_budget_plot = train_budget_shares.T.copy()
train_budget_plot.index = [
    f"{train_subset_labels[subset_name]} (n={int(train_budget_counts[subset_name].sum()):,})"
    for subset_name in train_budget_plot.index
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in train_budget_plot.columns]

fig, ax = plt.subplots(figsize=(11.8, 4.6))
train_budget_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.6)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in train")
ax.set_ylabel("")
ax.set_title("How the train-only subset strategy changes the training set")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="target_label", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()


#### Choose The Train-Only Subset

Pick the train-only subset strategy you want the models to fit on. This is the main knob to change when you want to see how train-set construction affects the models.

If you choose `balanced_reviewed`, keep an eye on `BALANCED_REVIEWED_TARGET_ISSUE_SHARE`. Raising it can give the model more issue examples until all available issue rows are already included; beyond that point the maximum possible issue share is limited by the training data.

Validation and test stay fixed. This choice only affects the training rows that are handed to the models.


In [ ]:
# Change this directly to compare how different train-only subset choices affect the models.
# Options: "full_train", "time_spread", "balanced_reviewed", "issue_focused"
TRAIN_SUBSET_STRATEGY = "time_spread"
if TRAIN_SUBSET_STRATEGY not in SUPPORTED_TRAIN_SUBSET_STRATEGIES:
    raise ValueError(
        f"Unsupported TRAIN_SUBSET_STRATEGY: {TRAIN_SUBSET_STRATEGY}. Choose from {SUPPORTED_TRAIN_SUBSET_STRATEGIES}."
    )

train_df = (
    train_full_df
    if TRAIN_SUBSET_STRATEGY == "full_train"
    else sample_frame_by_strategy(
        train_full_df,
        rows_limit=TRAIN_SUBSET_MAX_ROWS,
        sample_strategy=TRAIN_SUBSET_STRATEGY,
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
)

show_setup_json(
    {
        "TRAIN_SUBSET_STRATEGY": TRAIN_SUBSET_STRATEGY,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
    }
)


## Part 4 — Random Forest

We start with a strong tabular baseline: one row becomes one supervised example, and the forest learns to separate QC classes from engineered numeric features.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### 🌲 Supervised Learning: Random Forest

A **Random Forest** is an ensemble of many decision trees.

Here is the basic idea:

1. make many slightly different training samples from the data,
2. train one decision tree on each sample,
3. let the trees vote on the final class.

Each tree asks simple if/then questions such as:

- is conductivity above some threshold?
- did temperature change sharply?
- is the measurement happening at a particular time of day?

In this notebook, the Random Forest does **not** train on the raw reviewed modelling table directly.
Right before fitting the forest, we turn each reviewed row into a tabular feature vector that includes:

- the selected measurement columns themselves,
- `abs_delta` features that measure how sharply each measurement changed from the previous row,
- and simple clock features such as hour, minute, and day of year.

That feature-building step is specific to the Random Forest baseline. The CNN and transformer sections later do **not** use this same tabular feature table.

Why this is a good Session 1 model:

- it works well on numeric tabular data,
- it usually needs less feature scaling ceremony than many other models,
- it gives us interpretable outputs such as feature importance.

Limits to keep in mind:

- it does not naturally understand long sequential context,
- it only learns from the features we explicitly give it,
- rare classes can still be difficult.


![Random Forest bagging illustration](../assets/session1/random_forest_bagging_illustration.png)

Diagram idea: bootstrap samples are drawn from the original dataset, separate trees are trained, and their predictions are aggregated into one model decision.

Image credit: Harry585, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Random Forest Bagging Illustration.png](https://commons.wikimedia.org/wiki/File:Random_Forest_Bagging_Illustration.png)


### Random Forest Settings

These settings control the baseline forest we train below.

Main variables:

- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree. `None` means grow until other stopping rules apply.
- `min_samples_leaf`: minimum number of samples allowed in a leaf.
- `min_samples_split`: minimum number of samples needed to split an internal node.
- `max_features`: how many features each split is allowed to consider. Common choices are `"sqrt"`, `"log2"`, or `None`.
- `class_weight`: how strongly to compensate for imbalance. Common choices are `None`, `"balanced"`, and `"balanced_subsample"`.
- `trees_per_step`: how many trees to add each time we print progress.

Forests do **not** train epoch-by-epoch like neural networks. To make progress visible, we grow the forest in chunks using `warm_start=True` and print the validation F1 after each chunk.

One practical note: when we use `warm_start=True`, this notebook converts `"balanced"` or `"balanced_subsample"` into one fixed class-weight dictionary computed from `y_train`. That avoids a scikit-learn warning and keeps the weighting consistent across growth steps.


In [ ]:
RF_CONFIG = {
    "imputer_strategy": "median",
    "n_estimators": 200,
    "trees_per_step": 25,
    "max_depth": None,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "verbose": 0,
}

display(pd.Series(RF_CONFIG, name="value").rename_axis("rf_parameter").to_frame())


#### Random Forest Row Budget

The Random Forest uses the same `DATA_FRACTION` budget as the rest of the notebook. The next cell applies the derived caps and prints the actual row counts used for train, validation, and test.


In [ ]:
# These row caps were set in the visible row-budget cell above.
# Set any of them to None here if you want the random forest section to use the full split.

RF_TRAIN_CAP_STRATEGY = (
    TRAIN_SUBSET_STRATEGY
    if TRAIN_SUBSET_STRATEGY != "full_train"
    else "time_spread"
)

rf_train_fit_df = (
    sample_frame_by_strategy(
        train_df,
        rows_limit=RF_TRAIN_MAX_ROWS,
        sample_strategy=RF_TRAIN_CAP_STRATEGY,
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    if RF_TRAIN_MAX_ROWS is not None and len(train_df) > RF_TRAIN_MAX_ROWS
    else train_df.copy()
)
rf_valid_eval_df = (
    evenly_spaced_take(valid_df.sort_values("Time UTC").reset_index(drop=True), RF_VALIDATION_MAX_ROWS)
    if RF_VALIDATION_MAX_ROWS is not None and len(valid_df) > RF_VALIDATION_MAX_ROWS
    else valid_df.copy()
)
rf_test_eval_df = (
    evenly_spaced_take(test_df.sort_values("Time UTC").reset_index(drop=True), RF_TEST_MAX_ROWS)
    if RF_TEST_MAX_ROWS is not None and len(test_df) > RF_TEST_MAX_ROWS
    else test_df.copy()
)

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_VALIDATION_MAX_ROWS": RF_VALIDATION_MAX_ROWS,
        "RF_TEST_MAX_ROWS": RF_TEST_MAX_ROWS,
        "rf_train_cap_strategy": RF_TRAIN_CAP_STRATEGY,
        "rf_train_rows_used": len(rf_train_fit_df),
        "rf_validation_rows_used": len(rf_valid_eval_df),
        "rf_test_rows_used": len(rf_test_eval_df),
    }
)


In [ ]:
import gc

rf_split_rows = materialize_reviewed_split_frames(
    selected_paths,
    {
        "train": rf_train_fit_df,
        "validation": rf_valid_eval_df,
        "test": rf_test_eval_df,
    },
    columns=ROW_USE_COLUMNS,
    target_flag=TARGET_FLAG,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
)
rf_train_rows_df = rf_split_rows["train"]
rf_valid_rows_df = rf_split_rows["validation"]
rf_test_rows_df = rf_split_rows["test"]

# Build the Random Forest feature table here. This is the step where the shared
# helper adds the tabular baseline features:
# - the selected measurement columns,
# - per-column `abs_delta` change features,
# - and simple UTC clock features.
rf_train_fit_df, feature_columns, _ = build_model_frame(
    rf_train_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_valid_eval_df, _, _ = build_model_frame(
    rf_valid_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_test_eval_df, _, _ = build_model_frame(
    rf_test_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)

# The raw materialized split frames are no longer needed once the feature tables exist.
# Releasing them keeps high-fraction runs from carrying duplicate copies of the data.
for _rf_name in ["rf_split_rows", "rf_train_rows_df", "rf_valid_rows_df", "rf_test_rows_df"]:
    globals().pop(_rf_name, None)
gc.collect()

# Step 1: fit the imputer once on training data and reuse it for validation/test.
rf_imputer = SimpleImputer(strategy=RF_CONFIG.get("imputer_strategy", "median"))
X_train_rf = rf_imputer.fit_transform(rf_train_fit_df[feature_columns]).astype(np.float32, copy=False)
X_valid_rf = rf_imputer.transform(rf_valid_eval_df[feature_columns]).astype(np.float32, copy=False)
y_train_rf = rf_train_fit_df["model_target"].reset_index(drop=True)
y_valid_rf = rf_valid_eval_df["model_target"].reset_index(drop=True)
y_test_rf = rf_test_eval_df["model_target"].reset_index(drop=True)

# Step 2: compute one stable class-weight dictionary for the whole training split.
requested_class_weight = RF_CONFIG.get("class_weight")
if requested_class_weight in {"balanced", "balanced_subsample"}:
    rf_classes = np.array(sorted(pd.Series(y_train_rf).dropna().unique()))
    rf_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=rf_classes,
        y=y_train_rf,
    )
    effective_class_weight = {
        int(label) if isinstance(label, (np.integer, int)) else label: float(weight)
        for label, weight in zip(rf_classes.tolist(), rf_weight_values.tolist())
    }
else:
    effective_class_weight = requested_class_weight

print(
    {
        "requested_class_weight": requested_class_weight,
        "effective_class_weight": effective_class_weight,
    }
)

# Step 3: build the forest itself. We use warm_start so we can grow it in chunks and print progress.
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["trees_per_step"],
    warm_start=True,
    max_depth=RF_CONFIG["max_depth"],
    min_samples_leaf=RF_CONFIG["min_samples_leaf"],
    min_samples_split=RF_CONFIG["min_samples_split"],
    max_features=RF_CONFIG["max_features"],
    class_weight=effective_class_weight,
    n_jobs=-1,
    random_state=SEED,
    verbose=RF_CONFIG["verbose"],
)

rf_progress_rows = []
total_trees = RF_CONFIG["n_estimators"]
trees_per_step = RF_CONFIG["trees_per_step"]
growth_schedule = list(range(trees_per_step, total_trees + 1, trees_per_step))
if not growth_schedule:
    growth_schedule = [total_trees]
elif growth_schedule[-1] != total_trees:
    growth_schedule.append(total_trees)

for tree_count in growth_schedule:
    rf_model.set_params(n_estimators=tree_count)
    rf_model.fit(X_train_rf, y_train_rf)
    current_valid_predictions = rf_model.predict(X_valid_rf)
    current_valid_f1 = f1_score(
        y_valid_rf,
        current_valid_predictions,
        average=report_average(task_mode),
        zero_division=0,
    )
    rf_progress_rows.append({"trees_built": tree_count, "validation_f1": float(current_valid_f1)})
    print(f"Built {tree_count:>3} trees | validation F1 = {current_valid_f1:.4f}")

# Step 4: wrap the fitted pieces into a reusable sklearn pipeline object.
rf_pipeline = Pipeline(
    steps=[
        ("imputer", rf_imputer),
        ("model", rf_model),
    ]
)

X_test_rf = rf_imputer.transform(rf_test_eval_df[feature_columns]).astype(np.float32, copy=False)
valid_predictions = rf_model.predict(X_valid_rf)
test_predictions = rf_model.predict(X_test_rf)
labels = sorted(pd.unique(pd.concat([y_train_rf, y_valid_rf, y_test_rf])))

display(pd.DataFrame(rf_progress_rows))

with RF_MODEL_PATH.open("wb") as handle:
    pickle.dump(
        {
            "pipeline": rf_pipeline,
            "feature_columns": feature_columns,
            "labels": labels,
            "task_mode": task_mode,
        },
        handle,
    )

print(
    {
        "saved_random_forest_model": str(RF_MODEL_PATH),
        "rf_train_rows_used": int(len(rf_train_fit_df)),
        "rf_validation_rows_used": int(len(rf_valid_eval_df)),
        "rf_test_rows_used": int(len(rf_test_eval_df)),
    }
)

# The fitted pipeline keeps the trained model and imputer. The arrays and
# temporary fit frames below are only needed while fitting and predicting.
for _rf_name in [
    "X_train_rf",
    "X_valid_rf",
    "X_test_rf",
    "current_valid_predictions",
    "rf_train_fit_df",
    "rf_valid_eval_df",
    "rf_model",
    "rf_imputer",
    "y_train_rf",
]:
    globals().pop(_rf_name, None)
gc.collect()


In [ ]:
# Report validation and test metrics separately for the RF evaluation slices.
metric_rows = []
for split_name, y_true, y_pred in [
    ("validation", y_valid_rf, valid_predictions),
    ("test", y_test_rf, test_predictions),
]:
    split_f1 = f1_score(y_true, y_pred, average=report_average(task_mode), zero_division=0)
    metric_rows.append({"split": split_name, "f1": round(float(split_f1), 4)})
    print(f"{split_name.title()} macro/binary F1: {split_f1:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

display(pd.DataFrame(metric_rows))

# Plot normalised confusion matrices for both validation and test.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid_rf,
    valid_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[0],
)
axes[0].set_title(f"Validation confusion matrix for {target_name}")

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    test_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[1],
)
axes[1].set_title(f"Test confusion matrix for {target_name}")
fig.suptitle("How to read these plots: each row is a true class, and darker diagonal cells are better.", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance tells us which columns the forest used most strongly.
feature_importances = pd.Series(
    rf_pipeline.named_steps["model"].feature_importances_,
    index=feature_columns,
).sort_values(ascending=False)

top_importances = feature_importances.head(12).sort_values()
top_importances.plot(kind="barh", figsize=(8, 6), title="Top Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
plt.close()
display(feature_importances.head(12).rename("importance").to_frame())

# Show a few test-set mistakes to ground the discussion in real timestamps.
error_preview_columns = [
    "Time UTC",
    TARGET_FLAG,
    "issue",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
]
error_preview_columns.extend(
    [
        column
        for column in MEASUREMENT_COLUMNS
        if column not in error_preview_columns
    ][:1]
)
error_preview_columns = [column for column in error_preview_columns if column in rf_test_eval_df.columns]
error_frame = rf_test_eval_df[error_preview_columns].copy()
error_frame["prediction"] = test_predictions
error_frame["correct"] = error_frame["prediction"] == y_test_rf.to_numpy()
error_examples = error_frame.loc[~error_frame["correct"]].head(12)
print({"test_errors_shown": len(error_examples), "test_error_rate": round(float((~error_frame["correct"]).mean()), 4)})
display(error_examples)

# The test prediction arrays are no longer needed after the mistake preview.
for _rf_name in [
    "rf_test_eval_df",
    "y_test_rf",
    "test_predictions",
    "error_frame",
    "error_examples",
    "feature_importances",
    "top_importances",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Date-Range Demo: Predict QC Flags with the Random Forest

The test-set metrics above summarise performance across the whole held-out split. This mini-workflow asks a more concrete question:

**What does the Random Forest predict on one specific interval of time?**

Use UTC strings such as `"2025-09-10 00:00:00Z"` if you want to override the default range.


In [ ]:
RF_RANGE_START = None
RF_RANGE_END = None
RF_AUTO_SELECT_TEST_RANGE = True
RF_MAX_POINTS_TO_PLOT = 800

print(
    {
        "RF_RANGE_START": RF_RANGE_START,
        "RF_RANGE_END": RF_RANGE_END,
        "RF_AUTO_SELECT_TEST_RANGE": RF_AUTO_SELECT_TEST_RANGE,
        "RF_MAX_POINTS_TO_PLOT": RF_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
import gc

rf_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=RF_RANGE_START,
    end=RF_RANGE_END,
    auto_select=RF_AUTO_SELECT_TEST_RANGE,
    max_points=RF_MAX_POINTS_TO_PLOT,
)

rf_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=rf_range_selection["start"],
    end=rf_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)

if rf_interval_rows.empty:
    print("No row-level data was found in the requested Random Forest range.")
else:
    rf_interval_model_df, _, _ = build_model_frame(
        rf_interval_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        model_row_limit=None,
    )
    rf_interval_model_df = rf_interval_model_df[
        (rf_interval_model_df["Time UTC"] >= rf_range_selection["start"])
        & (rf_interval_model_df["Time UTC"] <= rf_range_selection["end"])
    ].reset_index(drop=True)

    if rf_interval_model_df.empty:
        print("The selected Random Forest range did not contain usable labelled rows after preparation.")
    else:
        rf_interval_predictions = rf_pipeline.predict(rf_interval_model_df[feature_columns])
        rf_interval_origin = infer_interval_origin(
            rf_range_selection["start"],
            rf_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        rf_interval_metrics = compute_interval_classification_metrics(
            rf_interval_model_df["model_target"],
            rf_interval_predictions,
            labels=labels,
            average=report_average(task_mode),
            target_names=[str(label) for label in labels],
        )
        rf_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
        rf_true_intervals = build_labeled_intervals(
            rf_interval_model_df,
            time_column="Time UTC",
            label_column="model_target",
        )
        rf_pred_frame = rf_interval_model_df[["Time UTC"]].copy()
        rf_pred_frame["predicted_label"] = rf_interval_predictions
        rf_pred_intervals = build_labeled_intervals(
            rf_pred_frame,
            time_column="Time UTC",
            label_column="predicted_label",
        )

        print(
            {
                "selection_mode": rf_range_selection["selection_mode"],
                "selected_priority_flag": rf_range_selection["selected_label"],
                "interval_origin": rf_interval_origin,
                "range_start": rf_range_selection["start"].isoformat(),
                "range_end": rf_range_selection["end"].isoformat(),
                "rows_in_interval": int(len(rf_interval_model_df)),
                "interval_f1": rf_interval_metrics["f1"],
            }
        )
        print(rf_interval_metrics["report_text"])

        display(
            pd.DataFrame(
                {
                    "true_count": pd.Series(rf_interval_model_df["model_target"]).value_counts().sort_index(),
                    "predicted_count": pd.Series(rf_interval_predictions).value_counts().sort_index(),
                }
            ).fillna(0).astype(int)
        )

        rf_demo_figure = plot_time_series_with_bands(
            rf_interval_model_df,
            band_specs=[
                {"title": f"True {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": rf_true_intervals, "palette": rf_plot_palette},
                {"title": f"RF {FLAG_EXAMPLE_DISPLAY_NAME} predictions", "intervals": rf_pred_intervals, "palette": rf_plot_palette},
            ],
            measurement_column=PLOT_MEASUREMENT_COLUMN,
            secondary_column=PLOT_SECONDARY_COLUMN,
            max_points=RF_MAX_POINTS_TO_PLOT,
            title="Random Forest predictions on a selected time range",
            label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
            legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
        )
        plt.show()
        plt.close(rf_demo_figure)

        rf_cm_fig, rf_cm_ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix=rf_interval_metrics["confusion_matrix"],
            display_labels=rf_interval_metrics["display_labels"],
        ).plot(ax=rf_cm_ax, cmap="Blues", colorbar=False)
        rf_cm_ax.set_title("Random Forest confusion matrix on the selected range")
        plt.tight_layout()
        plt.show()
        plt.close(rf_cm_fig)

# Random Forest interval intermediates can be rebuilt by rerunning this cell.
for _rf_name in [
    "rf_interval_rows",
    "rf_interval_model_df",
    "rf_interval_predictions",
    "rf_interval_metrics",
    "rf_true_intervals",
    "rf_pred_frame",
    "rf_pred_intervals",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Reflection Questions: Random Forest

1. Which input features seem most responsible for the mistakes in this interval: raw measurements, change features, or time-of-day features?
2. Do the errors line up with the class imbalance we saw earlier, or do they suggest a different modelling problem?
3. If you wanted to improve this interval specifically, would you change the features, the target, or the forest settings first?

Add your own notes or follow-up questions here:

- 
- 
- 


### Random Forest: Tune The Forest, Then Rethink The Features

This section stays entirely inside the Random Forest part of the workflow.

The lesson is deliberate:

1. start with a reasonable baseline forest,
2. tune the forest hyperparameters on a validation split,
3. then ask the more important question: are we feeding the model the right information?

In many tabular ML problems, feature preparation and target definition matter more than squeezing a few extra points from `n_estimators` or `max_depth`.


In [ ]:
RF_SEARCH_SPACE = {
    "n_estimators": [250, 400],
    "max_depth": [18, None],
    "min_samples_leaf": [1],
    "min_samples_split": [2],
    "max_features": ["sqrt"],
    "class_weight": ["balanced_subsample"],
}

# RF_SEARCH_MODEL_ROW_LIMIT was set in the visible row-budget cell above.
# Set RF_SEARCH_MODEL_ROW_LIMIT = None here if you want search trials to use the full train split.
BEST_RF_SEARCH_CONFIG = None

print({"RUN_HYPERPARAMETER_SEARCH": RUN_HYPERPARAMETER_SEARCH})
print({"DATA_FRACTION": DATA_FRACTION, "rf_search_model_row_limit": RF_SEARCH_MODEL_ROW_LIMIT})


In [ ]:
import gc

if RUN_HYPERPARAMETER_SEARCH:
    rf_search_train_selection_df = sample_frame_by_strategy(
        train_full_df,
        rows_limit=RF_SEARCH_MODEL_ROW_LIMIT,
        sample_strategy=TRAIN_SUBSET_STRATEGY if TRAIN_SUBSET_STRATEGY != "full_train" else "time_spread",
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    rf_search_valid_selection_df = evenly_spaced_take(
        valid_df.sort_values("Time UTC").reset_index(drop=True),
        RF_VALIDATION_MAX_ROWS,
    )
    rf_search_test_selection_df = evenly_spaced_take(
        test_df.sort_values("Time UTC").reset_index(drop=True),
        RF_TEST_MAX_ROWS,
    )
    rf_search_labels = sorted(reviewed_model_df["model_target"].dropna().astype(int).unique().tolist())

    rf_search_split_rows = materialize_reviewed_split_frames(
        selected_paths,
        {
            "train": rf_search_train_selection_df,
            "validation": rf_search_valid_selection_df,
            "test": rf_search_test_selection_df,
        },
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    rf_search_train_df, feature_columns, _ = build_model_frame(
        rf_search_split_rows["train"],
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    rf_search_valid_df, _, _ = build_model_frame(
        rf_search_split_rows["validation"],
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    globals().pop("rf_search_split_rows", None)
    gc.collect()

    rf_search_results, _, _ = run_rf_search(
        rf_search_train_df,
        rf_search_valid_df,
        feature_columns,
        labels=rf_search_labels,
        task_mode=task_mode,
        seed=SEED,
        search_space=RF_SEARCH_SPACE,
    )
    display(rf_search_results.head(10))

    for _rf_name in ["rf_search_train_df", "rf_search_valid_df"]:
        globals().pop(_rf_name, None)
    gc.collect()

    rf_candidate_keys = [
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "min_samples_split",
        "max_features",
        "class_weight",
    ]
    rf_transfer_train_selection_df = (
        sample_frame_by_strategy(
            train_df,
            rows_limit=RF_TRAIN_MAX_ROWS,
            sample_strategy=RF_TRAIN_CAP_STRATEGY,
            target_flag=TARGET_FLAG,
            good_labels=GOOD_LABELS,
            issue_labels=ISSUE_LABELS,
            issue_rows=ISSUE_ROWS_PER_FILE,
            balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
        )
        if RF_TRAIN_MAX_ROWS is not None and len(train_df) > RF_TRAIN_MAX_ROWS
        else train_df.copy()
    )
    rf_transfer_split_rows = materialize_reviewed_split_frames(
        selected_paths,
        {
            "train": rf_transfer_train_selection_df,
            "validation": rf_search_valid_selection_df,
            "test": rf_search_test_selection_df,
        },
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    rf_transfer_train_df, feature_columns, _ = build_model_frame(
        rf_transfer_split_rows["train"],
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    rf_transfer_valid_df, _, _ = build_model_frame(
        rf_transfer_split_rows["validation"],
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    rf_transfer_test_df, _, _ = build_model_frame(
        rf_transfer_split_rows["test"],
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    globals().pop("rf_transfer_split_rows", None)
    gc.collect()

    rf_transfer_rows = []
    for _, candidate_row in rf_search_results.head(2).iterrows():
        candidate_config = {}
        for key in rf_candidate_keys:
            value = candidate_row[key]
            if key in {"n_estimators", "min_samples_leaf", "min_samples_split"}:
                candidate_config[key] = int(value)
            elif key == "max_depth":
                candidate_config[key] = None if pd.isna(value) else int(value)
            else:
                candidate_config[key] = value
        candidate_model = fit_random_forest(
            rf_transfer_train_df,
            feature_columns,
            seed=SEED,
            config=candidate_config,
        )
        candidate_valid = evaluate_classifier(
            candidate_model,
            rf_transfer_valid_df,
            feature_columns,
            labels=rf_search_labels,
            task_mode=task_mode,
        )
        candidate_test = evaluate_classifier(
            candidate_model,
            rf_transfer_test_df,
            feature_columns,
            labels=rf_search_labels,
            task_mode=task_mode,
        )
        rf_transfer_rows.append(
            {
                "config": candidate_config,
                "validation_f1": candidate_valid["f1"],
                "test_f1": candidate_test["f1"],
            }
        )
        del candidate_model
        gc.collect()

    rf_transfer_frame = pd.DataFrame(rf_transfer_rows).sort_values("validation_f1", ascending=False).reset_index(drop=True)
    BEST_RF_SEARCH_CONFIG = rf_transfer_frame.iloc[0]["config"]
    display(rf_transfer_frame)

    for _rf_name in [
        "rf_search_results",
        "rf_search_train_selection_df",
        "rf_search_valid_selection_df",
        "rf_search_test_selection_df",
        "rf_transfer_train_selection_df",
        "rf_transfer_train_df",
        "rf_transfer_valid_df",
        "rf_transfer_test_df",
        "rf_transfer_frame",
        "rf_transfer_rows",
    ]:
        globals().pop(_rf_name, None)
    gc.collect()
else:
    print("RF hyperparameter search skipped. Keeping the baseline forest settings.")

print({"BEST_RF_SEARCH_CONFIG": BEST_RF_SEARCH_CONFIG})


### Beyond Random Forest: Better Features, Better Tree Families, Better Targets

The Random Forest section above teaches the baseline workflow. The next question is how to improve it without jumping straight to a neural network.

A useful way to think about tree-model improvement is:

1. **Add more context**. If a single row is not enough, give the model lags and rolling summaries.
2. **Try a stronger tree family**. Random Forest is not the only bagged-tree option; ExtraTrees can work better on noisy tabular data.
3. **Revisit the target itself**. If labels are ambiguous, collapsing some classes may produce a model that is both easier to learn and closer to the operational decision.

The cells below use the local experiments we just ran to show this process explicitly.


In [ ]:
# Build context features on a small, explicit training sample.
# This demonstrates the feature-engineering idea without carrying the full row cache in memory.
context_selection_df = evenly_spaced_take(
    train_df.sort_values("Time UTC").reset_index(drop=True),
    min(50_000, len(train_df)),
)
context_rows_df = load_selected_row_level_frame(
    selected_paths,
    context_selection_df,
    columns=ROW_USE_COLUMNS,
)
context_model_df, feature_columns, _ = build_model_frame(
    context_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
advanced_model_df, context_feature_columns = add_temporal_context_features(
    context_model_df,
    measurement_columns=measurement_columns,
)
lag_examples = [name for name in context_feature_columns if " lag_" in name][:12]
rolling_examples = [name for name in context_feature_columns if " roll_" in name][:12]
context_feature_examples = pd.DataFrame(
    [
        *[{"feature_group": "lag", "feature_name": name} for name in lag_examples],
        *[{"feature_group": "rolling", "feature_name": name} for name in rolling_examples],
    ]
)

print(
    {
        "context_sample_rows": len(advanced_model_df),
        "base_feature_count": len(feature_columns),
        "new_context_feature_count": len(context_feature_columns),
        "total_feature_count_if_used": len(feature_columns) + len(context_feature_columns),
    }
)
display(context_feature_examples)

for _context_name in ["context_selection_df", "context_rows_df", "context_model_df", "advanced_model_df"]:
    globals().pop(_context_name, None)
gc.collect()


### Tree Family Comparison

We compared:

- the baseline Random Forest,
- Random Forest plus temporal context features,
- ExtraTrees plus the same context features,
- and a gradient-boosted tree baseline.

The key point is not just *which model won*. It is *why*:

- context features gave the tree models access to short-term history,
- ExtraTrees handled the noisy tabular signal better than the baseline forest,
- and we checked performance on the same time-based splits rather than trusting one lucky number.


In [ ]:
tree_search_path = ARTIFACT_DIR / "tree_model_comparison_search.csv"
tree_full_path = ARTIFACT_DIR / "tree_model_comparison_full.csv"

if tree_search_path.exists() and tree_full_path.exists():
    tree_search_results = pd.read_csv(tree_search_path)
    tree_full_results = pd.read_csv(tree_full_path)

    display(
        tree_search_results[
            [
                "name",
                "family",
                "feature_variant",
                "validation_f1",
                "test_f1",
                "validation_class_4_recall",
                "test_class_4_recall",
            ]
        ]
    )
    display(
        tree_full_results[
            [
                "name",
                "family",
                "feature_variant",
                "validation_f1",
                "test_f1",
                "validation_class_4_recall",
                "test_class_4_recall",
            ]
        ]
    )
else:
    print(
        "Advanced tree comparison artifacts are not included in this repo. "
        "This section uses offline comparison results when they are available."
    )


### Target Design: Should We Collapse Ambiguous Flags?

Another important modelling decision is whether the original label set is the right one.

For this dataset, `3` and `4` are both problem cases, while `9` means the value is missing. Depending on the operational question, it can be more useful to predict:

- the original `1 / 3 / 4 / 9` labels,
- a collapsed `1 / 34 / 9` target,
- or a binary `good vs issue` target.

This is not just a modelling trick. It is a problem-definition question. A simpler target can be much easier to learn and may align better with the real decision you need to automate.


In [ ]:
strategy_summary_path = ARTIFACT_DIR / "extratrees_strategy_summary.json"
strategy_results_path = ARTIFACT_DIR / "extratrees_target_strategy_comparison.csv"
collapsed_metrics_path = ARTIFACT_DIR / "collapsed_target_per_class_metrics.md"

if strategy_results_path.exists():
    strategy_results = pd.read_csv(strategy_results_path)
    display(strategy_results)
else:
    print(
        "Target-strategy comparison artifacts are not included in this repo. "
        "This section uses offline comparison results when they are available."
    )

per_class_tables = []
for csv_name, strategy_name, split_name in [
    ("collapsed_1_34_9_validation_per_class_metrics.csv", "collapsed_1_34_9", "validation"),
    ("collapsed_1_34_9_test_per_class_metrics.csv", "collapsed_1_34_9", "test"),
    ("binary_issue_validation_per_class_metrics.csv", "binary_issue", "validation"),
    ("binary_issue_test_per_class_metrics.csv", "binary_issue", "test"),
]:
    csv_path = ARTIFACT_DIR / csv_name
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame.insert(0, "strategy", strategy_name)
        frame.insert(1, "split", split_name)
        per_class_tables.append(frame)

if per_class_tables:
    display(pd.concat(per_class_tables, ignore_index=True))
else:
    print(
        "Per-class collapsed-target metric artifacts are not included in this repo."
    )


### What We Learned From The Advanced Tree Experiments

The main lessons from these local experiments were:

- **More data helped**, but only modestly for the full 4-class problem.
- **Temporal context features helped more** because the QC issues often occur in short coherent runs.
- **ExtraTrees outperformed the baseline Random Forest** in this setting.
- **Collapsing ambiguous labels helped dramatically**, especially for `1 / 34 / 9` and binary `issue` detection.

This is exactly the kind of modelling mindset we want in Session 2:

- do not assume the first feature set is enough,
- do not assume the first model family is the best,
- and do not assume the original label definition is the most useful one.


## Part 5 — k-means

Next we switch to an unsupervised lens: instead of predicting flags directly, we group windows with similar summary behaviour and interpret those clusters.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### 🧭 Unsupervised Learning: k-means On Window Features

Supervised learning uses labels. Unsupervised learning does **not**.

Here we use **k-means**, which groups rows or windows into clusters based on similarity in feature space. The cluster IDs themselves do not carry meaning ahead of time. We interpret them *after* fitting by looking at:

- how many examples ended up in each cluster,
- the mean issue rate inside each cluster,
- where the cluster sits in feature space,
- and what the underlying time-series examples look like.

This is useful when you want to surface interesting operating regimes or suspicious periods even before a label model is mature.

One subtle point: the cluster numbers and colours are arbitrary. They only become meaningful after we interpret them using the plots and summary statistics below.


### 🔎 k-means Settings

Main variables:

- `n_clusters`: how many clusters we ask the algorithm to find.
- `n_init`: how many random initializations to try. `"auto"` is a good default in current scikit-learn.
- `KMEANS_FEATURE_MODE`: inherited from the dataset profile. Some datasets cluster best on window summaries, while spike-driven ones may cluster better on row-level features.

The number of cached windows loaded for clustering is derived from `DATA_FRACTION`. If you want to experiment with k-means itself, the most useful first change is usually `n_clusters`.


In [ ]:
KMEANS_CONFIG = {
    "n_clusters": 5,
    "n_init": "auto",
}
# KMEANS_WINDOW_LIMIT was set in the visible row-budget cell above.
# Set KMEANS_WINDOW_LIMIT = None here if you want to cluster every selected window/row.
KMEANS_EXAMPLES_PER_CLUSTER = 1
KMEANS_EXAMPLE_CONTEXT_POINTS = 7500
KMEANS_EXAMPLE_HIGHLIGHT_ALPHA = 0.22

display(pd.Series(KMEANS_CONFIG, name="value").rename_axis("kmeans_parameter").to_frame())
print(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
        "KMEANS_EXAMPLES_PER_CLUSTER": KMEANS_EXAMPLES_PER_CLUSTER,
        "KMEANS_EXAMPLE_CONTEXT_POINTS": KMEANS_EXAMPLE_CONTEXT_POINTS,
        "KMEANS_EXAMPLE_HIGHLIGHT_ALPHA": KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    }
)


In [ ]:
if KMEANS_FEATURE_MODE == "window_summary":
    cluster_source_df = window_df
    kmeans_feature_columns = [
        column
        for column in cluster_source_df.columns
        if column.endswith("_mean") or column.endswith("_std")
    ]
    cluster_x_column = f"{PLOT_MEASUREMENT_COLUMN}_mean"
    cluster_y_column = f"{PLOT_SECONDARY_COLUMN}_mean"
    cluster_axis_label_suffix = "mean"
else:
    # Row-level k-means should use the same visible row budget as window-summary k-means.
    # Without this sample, row-level feature mode would materialise the full reviewed dataset.
    cluster_selection_df = evenly_spaced_take(
        reviewed_model_df[["Time UTC", "source_file"]].copy(),
        KMEANS_WINDOW_LIMIT,
        time_column="Time UTC",
    )
    cluster_source_rows = load_selected_row_level_frame(
        selected_paths,
        cluster_selection_df,
        columns=ROW_USE_COLUMNS,
    )
    cluster_source_df, feature_columns, _ = build_model_frame(
        cluster_source_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    cluster_source_df["issue"] = cluster_source_df["issue"].astype(int)
    cluster_source_df["window_start"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["window_end"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["issue_rate"] = cluster_source_df["issue"].astype(float)
    kmeans_feature_columns = [column for column in feature_columns if column in cluster_source_df.columns]
    cluster_x_column = PLOT_MEASUREMENT_COLUMN
    cluster_y_column = PLOT_SECONDARY_COLUMN
    cluster_axis_label_suffix = "value"

if cluster_x_column not in cluster_source_df.columns or cluster_y_column not in cluster_source_df.columns:
    raise ValueError(
        f"k-means plotting columns are missing for profile {DATASET_PROFILE_ID}: "
        f"{cluster_x_column!r}, {cluster_y_column!r}"
    )

window_df, cluster_summary = fit_kmeans(
    cluster_source_df,
    n_clusters=KMEANS_CONFIG["n_clusters"],
    seed=SEED,
    n_init=KMEANS_CONFIG["n_init"],
    feature_mode=KMEANS_FEATURE_MODE,
    feature_columns=kmeans_feature_columns,
)
display(cluster_summary.round({"mean_issue_rate": 3, "max_issue_rate": 3, "avg_distance": 3}))


In [ ]:
# Plot a readable sample of clustered points so the scatter does not become an unreadable blob.
plot_window_df = evenly_spaced_take(window_df.sort_values("window_start"), 5000)
cluster_ids = sorted(cluster_summary.index.tolist())
cluster_colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_ids)))
cluster_palette = {cluster_id: cluster_colors[idx] for idx, cluster_id in enumerate(cluster_ids)}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
cluster_centers = (
    plot_window_df.groupby("cluster")[[cluster_x_column, cluster_y_column]]
    .mean()
    .reindex(cluster_ids)
)

legend_handles = []
for cluster_id in cluster_ids:
    subset = plot_window_df[plot_window_df["cluster"] == cluster_id]
    axes[0].scatter(
        subset[cluster_x_column],
        subset[cluster_y_column],
        s=18,
        alpha=0.55,
        color=cluster_palette[cluster_id],
        edgecolors="none",
    )
    axes[0].scatter(
        cluster_centers.loc[cluster_id, cluster_x_column],
        cluster_centers.loc[cluster_id, cluster_y_column],
        s=220,
        marker="*",
        color=cluster_palette[cluster_id],
        edgecolors="black",
        linewidths=0.8,
    )
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=cluster_palette[cluster_id],
            markersize=8,
            label=(
                f"Cluster {cluster_id} | n={int(cluster_summary.loc[cluster_id, 'window_count'])} | "
                f"mean issue={cluster_summary.loc[cluster_id, 'mean_issue_rate']:.2f}"
            ),
        )
    )

axes[0].set_title(f"k-means clusters in {KMEANS_FEATURE_MODE.replace('_', ' ')} feature space")
axes[0].set_xlabel(f"{PLOT_MEASUREMENT_COLUMN} ({cluster_axis_label_suffix})")
axes[0].set_ylabel(f"{PLOT_SECONDARY_COLUMN} ({cluster_axis_label_suffix})")
axes[0].grid(alpha=0.25)
axes[0].legend(handles=legend_handles, title="Cluster legend", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

bar_positions = np.arange(len(cluster_ids))
bar_values = [cluster_summary.loc[cluster_id, "mean_issue_rate"] for cluster_id in cluster_ids]
bar_colors = [cluster_palette[cluster_id] for cluster_id in cluster_ids]
axes[1].bar(bar_positions, bar_values, color=bar_colors, edgecolor="black", linewidth=0.6)
axes[1].set_xticks(bar_positions)
axes[1].set_xticklabels([f"Cluster {cluster_id}" for cluster_id in cluster_ids], rotation=30)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Mean issue rate")
axes[1].set_title("Average issue rate by cluster")
for idx, cluster_id in enumerate(cluster_ids):
    axes[1].text(
        idx,
        bar_values[idx] + 0.02,
        f"n={int(cluster_summary.loc[cluster_id, 'window_count'])}\n{bar_values[idx]:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

interesting_windows = window_df.sort_values(
    ["issue_rate", "distance_to_centroid"],
    ascending=[False, False],
).head(10)
interesting_windows = interesting_windows[
    ["window_start", "window_end", "source_file", "cluster", "issue_rate", "distance_to_centroid"]
].copy()
interesting_windows["source_file"] = interesting_windows["source_file"].map(clean_source_file_label)
display(interesting_windows.round({"issue_rate": 3, "distance_to_centroid": 3}))


### Looking At Real Time-Series Windows From Each Cluster

The scatter plot shows where clusters sit in feature space, but it does not show what the underlying sensor behaviour actually looked like.

The next plot closes that loop. For each cluster, we pick a representative window that sits close to its centroid, then:

- show a wider time-series context from the original row-level parquet,
- show the true target-label regions from that underlying data,
- highlight the exact window used in k-means,
- mark the datapoints inside that highlighted window.

This makes it much easier to interpret clusters as real operating regimes rather than abstract colored dots.


In [ ]:
cluster_example_figure, cluster_example_windows = plot_cluster_window_examples(
    window_df,
    source_to_row_part=source_to_row_part,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    target_flag=TARGET_FLAG,
    good_labels=GOOD_LABELS,
    examples_per_cluster=KMEANS_EXAMPLES_PER_CLUSTER,
    context_points=KMEANS_EXAMPLE_CONTEXT_POINTS,
    highlight_alpha=KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
)
plt.show()
display(cluster_example_windows)


### Date-Range Demo: See Which Clusters Appear in a Specific Interval

k-means does not predict target labels directly. Instead, it groups short windows with similar summary behaviour.

This demo lets us ask: **what kinds of windows show up inside one selected time range, and how do their issue rates differ?**


In [ ]:
KMEANS_RANGE_START = "2026-02-9T12:56:27.707000+00:00"
KMEANS_RANGE_END = "2026-02-11T12:56:27.707000+00:00"
KMEANS_AUTO_SELECT_TEST_RANGE = True
KMEANS_MAX_POINTS_TO_PLOT = 800

print(
    {
        "KMEANS_RANGE_START": KMEANS_RANGE_START,
        "KMEANS_RANGE_END": KMEANS_RANGE_END,
        "KMEANS_AUTO_SELECT_TEST_RANGE": KMEANS_AUTO_SELECT_TEST_RANGE,
        "KMEANS_MAX_POINTS_TO_PLOT": KMEANS_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
kmeans_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=KMEANS_RANGE_START,
    end=KMEANS_RANGE_END,
    auto_select=KMEANS_AUTO_SELECT_TEST_RANGE,
    max_points=KMEANS_MAX_POINTS_TO_PLOT,
)

kmeans_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=kmeans_range_selection["start"],
    end=kmeans_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)
kmeans_interval_windows = window_df[
    (window_df["window_start"] <= kmeans_range_selection["end"])
    & (window_df["window_end"] >= kmeans_range_selection["start"])
].copy()

if kmeans_interval_rows.empty or kmeans_interval_windows.empty:
    print("No overlapping k-means windows were found in the requested range.")
else:
    kmeans_interval_origin = infer_interval_origin(
        kmeans_range_selection["start"],
        kmeans_range_selection["end"],
        {"train": train_df, "validation": valid_df, "test": test_df},
    )
    kmeans_interval_intervals = merge_adjacent_intervals(
        kmeans_interval_windows.rename(
            columns={"window_start": "start", "window_end": "end", "cluster": "label"}
        )[["start", "end", "label"]]
    )
    kmeans_cluster_counts = (
        kmeans_interval_windows.groupby("cluster")
        .agg(
            window_count=("cluster", "size"),
            mean_issue_rate=("issue_rate", "mean"),
            max_issue_rate=("issue_rate", "max"),
        )
        .sort_index()
    )

    print(
        {
            "selection_mode": kmeans_range_selection["selection_mode"],
            "selected_priority_flag": kmeans_range_selection["selected_label"],
            "interval_origin": kmeans_interval_origin,
            "range_start": kmeans_range_selection["start"].isoformat(),
            "range_end": kmeans_range_selection["end"].isoformat(),
            "row_points_in_interval": int(len(kmeans_interval_rows)),
            "window_count_in_interval": int(len(kmeans_interval_windows)),
        }
    )
    display(kmeans_cluster_counts.round(3))

    kmeans_demo_figure = plot_time_series_with_bands(
        kmeans_interval_rows,
        band_specs=[
            {
                "title": "k-means clusters",
                "intervals": kmeans_interval_intervals,
                "palette": cluster_palette,
            }
        ],
        measurement_column=PLOT_MEASUREMENT_COLUMN,
        secondary_column=PLOT_SECONDARY_COLUMN,
        max_points=KMEANS_MAX_POINTS_TO_PLOT,
        title="k-means cluster assignments on a selected time range",
        legend_title="cluster",
    )
    plt.show()


### Reflection Questions: k-means

1. Do these cluster spans look like real operating regimes, or do some clusters still seem hard to interpret physically?
2. If you changed `n_clusters`, which regions in this interval do you expect would split apart or merge together?
3. How closely do the cluster spans line up with target-label changes, and what does that say about the usefulness of unsupervised learning here?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 6 — 1D CNN

Now we keep short stretches of signal together as one training example and see how a sequence model behaves when it learns local time patterns directly.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### Sequential Model: 1D CNN

A **convolutional neural network (CNN)** applies small learnable filters across an ordered signal. For images that order is two-dimensional. In this notebook, the order is **time**.

The key idea is:

1. a short filter looks at a local pattern,
2. the same filter slides across the whole sequence,
3. later layers combine those detected patterns into a final prediction.

For this scalar QC task, a 1D CNN can learn patterns such as:

- sudden spikes or drops,
- flat segments,
- repeated local shapes across several sensor channels.

Why this is useful here:

- Random Forest treats each row mostly as a tabular snapshot,
- CNN keeps a short stretch of signal together as one example,
- the training loop also gives us a chance to teach batching, validation checkpoints, and early stopping.

The CNN section runs by default in this notebook and follows standard training practice:

- train / validation / test split,
- class-aware loss weighting,
- best-checkpoint saving based on validation F1,
- early stopping,
- mini-batch training through `DataLoader`,
- pinned-memory and multi-worker loading when the local machine supports it.

Useful references: [PyTorch tutorial on defining neural networks](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial), [PyTorch DataLoader docs](https://pytorch.org/docs/stable/data.html), [PyTorch performance tuning guide](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)


![Convolutional network diagram](../assets/session1/convolutional_network.png)

Diagram idea: small filters slide across the input, produce feature maps, and later layers combine those maps into a final prediction.

In this notebook the same logic is applied to **1D time windows** rather than 2D images.

Image credit: Aphex34, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Convolutional Network.png](https://commons.wikimedia.org/wiki/File:Convolutional_Network.png)


### 🎛️ CNN Settings

These settings control the baseline 1D CNN and its training loop.

Model-shape variables:

- `window_size`: number of time steps in each training window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `conv_channels`: number of filters in each convolution layer.
- `dropout`: dropout probability used for regularisation.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Optimisation variables:

- `epochs`: maximum number of passes through the training data.
- `batch_size`: number of windows per optimisation step.
- `learning_rate`: optimiser step size.
- `weight_decay`: L2-style regularisation for the optimiser.
- `patience` and `min_delta`: early-stopping controls.
- `lr_decay_factor` and `lr_patience`: learning-rate scheduler controls.
- `gradient_clip_norm`: cap on gradient size to stabilize training.

DataLoader variables:

- `num_workers`: worker processes for loading batches.
- `pin_memory`: can speed up host-to-GPU transfer.
- `persistent_workers`: keeps workers alive between epochs.
- `prefetch_factor`: how many batches each worker prepares ahead of time.

Dataset-aware default:

- the turbidity and conductivity-plug profiles start in `"per_timestep"` mode because short local events are easy to lose inside mixed windows,
- the other profiles start in `"window"` mode to keep the first baseline simpler.


In [ ]:
# CNN_CONFIG controls the learning problem, model size, and training loop.
# Start with these defaults, then change one or two values at a time so results stay interpretable.
CNN_CONFIG = {
    # Number of consecutive timestamps given to the CNN for one example.
    "window_size": 128,
    # "window" predicts one label for the whole window; "per_timestep" predicts one label per row.
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "epochs": 10,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "dropout": 0.2,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    # Each number is the number of filters in a Conv1D layer. More filters can learn richer patterns but cost more memory.
    "conv_channels": [32, 64],
    "label_reduction": "worst",
}

# DataLoader settings control how batches are moved into PyTorch.
# num_workers=0 is slower but safest inside shared notebook environments.
_loader_torch = globals().get("torch", None)
CNN_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(CNN_CONFIG, name="value").rename_axis("cnn_parameter").to_frame())
display(pd.Series(CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Utility For Window Labels

The CNN and transformer sections now support two output styles:

- `"window"`: one prediction for the whole window,
- `"per_timestep"`: one prediction for every point inside the window.

When you choose `"window"`, we need one short rule for turning several row-level target labels inside a window into one label. The helper below keeps that reduction in one place so the later cells stay easier to read.


In [ ]:
# Reduce row-level labels to one window-level label for sequence models.
def reduce_window_target(
    values: np.ndarray,
    mode: str,
    severity_order: tuple[int, ...] = (1, 3, 4, 9),
) -> int:
    labels = [int(value) for value in values if pd.notna(value)]
    if not labels:
        return severity_order[0]
    effective_order = tuple(sorted(set(severity_order).union(labels)))
    severity_rank = {label: index for index, label in enumerate(effective_order)}
    if mode == "worst":
        return max(labels, key=lambda label: severity_rank.get(label, -1))
    counts = pd.Series(labels).value_counts()
    tied_labels = counts[counts == counts.max()].index.tolist()
    return max(tied_labels, key=lambda label: severity_rank.get(int(label), -1))


### CNN Step 1: Turn Rows into Windows

A CNN expects a fixed-size tensor, not an arbitrary-length table.

In this step we:

- collect the measurement columns we want to model,
- group consecutive rows into fixed windows,
- either reduce many row-level target labels into one window label or keep one target per timestamp,
- split the windows with the active train / validation / test strategy,
- normalise each sensor channel using only the training split.

One important detail: the baseline CNN does **not** take the full Random Forest feature table as input. It sees windows of the selected measurement columns only. That means it gets more temporal context than the RF, but less hand-engineered context.

This is a good place to pause and ask: what information do we lose when we compress several row labels into one window label, and when is that simplification actually helpful?


In [ ]:
# Run this cell when you are ready to work through the CNN section.
# Earlier notebook parts do not need PyTorch, so we import it here.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
    CNN_READY = True
    print({
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except ImportError as exc:
    PYTORCH_READY = False
    CNN_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)


In [ ]:
if not CNN_READY:
    CNN_RUN = False
    print("CNN training cell skipped.")
else:
    # Load the row-level measurements for the already chosen train/validation/test split.
    # This keeps the CNN evaluation aligned with the Random Forest fixed split.
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    # Pull the settings into short variable names because they are used throughout the CNN cells.
    WINDOW_SIZE = CNN_CONFIG["window_size"]
    CNN_OUTPUT_MODE = CNN_CONFIG["output_mode"]
    # Convert rows into fixed-length windows.
    # The helper returns X arrays shaped like (windows, time, channels) and y labels for each split.
    cnn_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=CNN_OUTPUT_MODE,
        window_size=WINDOW_SIZE,
        label_reduction=CNN_CONFIG["label_reduction"],
    )
    # class_labels maps model output indexes back to the original QC labels.
    class_labels = cnn_bundle.class_labels

    # A classifier needs at least two classes and at least one complete window in every split.
    if task_mode == "multiclass" and len(class_labels) < 2:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "windows_total": int(len(cnn_bundle.X_train) + len(cnn_bundle.X_valid) + len(cnn_bundle.X_test)),
                "active_labels": class_labels,
                "skip_reason": "Need at least two active classes to train the CNN in multiclass mode.",
            }
        )
    elif len(cnn_bundle.X_train) == 0 or len(cnn_bundle.X_valid) == 0 or len(cnn_bundle.X_test) == 0:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        # Move to channel-first layout expected by Conv1D.
        X_train = np.transpose(cnn_bundle.X_train, (0, 2, 1))
        X_valid = np.transpose(cnn_bundle.X_valid, (0, 2, 1))
        X_test = np.transpose(cnn_bundle.X_test, (0, 2, 1))
        # Keep the labels beside the transposed feature arrays; labels are not normalised.
        y_train = cnn_bundle.y_train
        y_valid = cnn_bundle.y_valid
        y_test = cnn_bundle.y_test

        # Normalise each sensor channel using only the training split.
        # Validation and test use the training mean/std so evaluation does not peek at held-out data.
        channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train = (X_train - channel_mean) / channel_std
        X_valid = (X_valid - channel_mean) / channel_std
        X_test = (X_test - channel_mean) / channel_std

        CNN_RUN = True
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "fixed_split_block_rows": FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(WINDOW_SIZE),
                "channels": int(len(measurement_columns)),
            }
        )


### CNN Step 2: Build the Model and the DataLoaders

Here we create four pieces:

- the neural network itself,
- the loss function,
- the optimiser and learning-rate scheduler,
- the `DataLoader` objects that stream mini-batches during training.

The output toggle changes the final prediction head:

- in `"window"` mode, the CNN pools across time and emits one label for the whole window,
- in `"per_timestep"` mode, the CNN keeps the time axis and emits one label per point.

Notice that the `DataLoader` keeps the full dataset in CPU memory and feeds the GPU one batch at a time. That is usually much more efficient than trying to move every window onto the GPU all at once.


In [ ]:
if not CNN_RUN:
    print("CNN model setup skipped.")
else:
    # Fix PyTorch randomness so reruns are easier to compare.
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    # TinyQCNet is a compact 1D CNN: it scans across time and learns short temporal patterns.
    # Input tensors have shape (batch, sensor_channels, time_steps).
    class TinyQCNet(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            # The feature extractor applies convolution filters over neighbouring timestamps.
            self.features = nn.Sequential(
                nn.Conv1d(channels, CNN_CONFIG["conv_channels"][0], kernel_size=7, padding=3),
                nn.ReLU(),
                nn.Conv1d(CNN_CONFIG["conv_channels"][0], CNN_CONFIG["conv_channels"][1], kernel_size=5, padding=2),
                nn.ReLU(),
                nn.Dropout(CNN_CONFIG["dropout"]),
            )
            if CNN_OUTPUT_MODE == "window":
                # Pool across time when the model should emit one label for the whole window.
                self.pool = nn.AdaptiveAvgPool1d(1)
                self.head = nn.Linear(CNN_CONFIG["conv_channels"][1], output_dim)
            else:
                # A 1x1 convolution keeps the time axis and emits one prediction per timestamp.
                self.head = nn.Conv1d(CNN_CONFIG["conv_channels"][1], output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # First transform the raw sensor channels into learned temporal features.
            x = self.features(x)
            if CNN_OUTPUT_MODE == "window":
                x = self.pool(x).squeeze(-1)
                return self.head(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(1)
            return logits.transpose(1, 2)

    # Choose a loss function that matches the target type.
    # Multiclass uses one integer class per example; binary uses a single logit and class weighting.
    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        # Weight rare classes more heavily so the model does not learn to ignore issue labels.
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        # pos_weight gives the issue class more influence when issues are rare.
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    # TensorDataset pairs each input window with its target label for DataLoader batching.
    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    # DataLoader options are collected once so train/validation/test loaders stay consistent.
    loader_kwargs = {}
    num_workers = int(CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(CNN_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(CNN_LOADER_CONFIG["prefetch_factor"])
    if CNN_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    # The notebook no longer needs the large NumPy arrays after TensorDatasets are built.
    # Deleting them reduces memory pressure before training starts.
    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(
        train_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=True,
        **loader_kwargs,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )

    # Create the model and optimiser. AdamW is a good default for neural networks with weight decay.
    model = TinyQCNet(channels=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNN_CONFIG["learning_rate"],
        weight_decay=CNN_CONFIG["weight_decay"],
    )
    # Reduce the learning rate when validation F1 stops improving.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=CNN_CONFIG["lr_decay_factor"],
        patience=CNN_CONFIG["lr_patience"],
    )

    # These variables track the best validation checkpoint and support early stopping.
    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    # One shared epoch function handles both training and evaluation.
    # training=True enables gradients and optimiser steps; training=False only scores batches.
    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                # The logits shape differs for window-level vs per-timestep predictions,
                # so multiclass loss receives the class dimension in the place PyTorch expects.
                if task_mode == "multiclass":
                    if CNN_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    # Clear old gradients, backpropagate this batch, optionally clip, then update weights.
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            # Store flattened predictions/targets so F1 can be computed across the whole split.
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": CNN_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )


### CNN Step 3: Train with Validation Monitoring

This cell runs the optimisation loop.

We are following standard training practice:

- train on the training split,
- score the model on the validation split after each epoch,
- save the best checkpoint seen so far,
- stop early if the validation metric stops improving.

Watch the printed validation F1 as the run progresses. When `output_mode="window"`, that F1 is computed per window. When `output_mode="per_timestep"`, it is computed across all timestamps in the validation windows.


In [ ]:
if not CNN_RUN:
    print("CNN fit skipped.")
else:
    # Train for a small number of epochs and evaluate validation performance after each pass.
    for epoch in range(1, CNN_CONFIG["epochs"] + 1):
        # The training pass updates model weights; predictions are not needed here.
        train_loss, _, _ = run_epoch(train_loader, training=True)
        # The validation pass does not update weights; it estimates generalisation during training.
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        # Validation F1 drives the learning-rate scheduler and the best-checkpoint decision.
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        # Save a checkpoint whenever validation F1 improves by at least min_delta.
        if valid_f1 > best_metric + CNN_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": WINDOW_SIZE,
                    "output_mode": CNN_OUTPUT_MODE,
                    "config": {**CNN_CONFIG, **CNN_LOADER_CONFIG},
                },
                CNN_MODEL_PATH,
            )
        else:
            # Early stopping protects the notebook from spending epochs on a stalled model.
            patience_counter += 1
            if patience_counter >= CNN_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### CNN Step 4: Reload the Best Checkpoint and Evaluate

Training loss alone is not enough. In this final step we:

- reload the weights that gave the best validation F1,
- run the held-out test split once,
- inspect the classification report and confusion matrix.

This keeps the evaluation honest and makes it clear that the validation set guided model selection, while the test set is reserved for final reporting.


In [ ]:
if not CNN_RUN:
    print("CNN evaluation skipped.")
else:
    # Reload the best validation checkpoint before touching the test split.
    if best_state is None:
        raise RuntimeError("CNN training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    # Test results are reported once, after model selection is finished.
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": CNN_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(CNN_MODEL_PATH),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )

    # Build human-readable report labels for either multiclass or binary mode.
    if task_mode == "multiclass":
        report_labels = list(range(len(class_labels)))
        report_names = [str(label) for label in class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    # Plot learning curves and a normalised confusion matrix to show both training behaviour and errors.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("CNN test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See CNN Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
# Leave start/end as None to auto-select an interesting test interval.
# Set explicit timestamps here when you want to inspect a specific deployment period.
CNN_RANGE_START = None
CNN_RANGE_END = None
CNN_AUTO_SELECT_TEST_RANGE = True
CNN_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "CNN_RANGE_START": CNN_RANGE_START,
        "CNN_RANGE_END": CNN_RANGE_END,
        "CNN_AUTO_SELECT_TEST_RANGE": CNN_AUTO_SELECT_TEST_RANGE,
        "CNN_MAX_POINTS_TO_PLOT": CNN_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
cnn_interval_metrics = None

# Keep this guard because `model`, `device`, `channel_mean`, and `channel_std`
# are created only when the CNN training cells have run successfully.
if not CNN_RUN:
    cnn_demo_result = show_cnn_interval_demo(cnn_run=False)
else:
    cnn_demo_result = show_cnn_interval_demo(
        cnn_run=CNN_RUN,
        # CNN_CONFIG controls the prediction shape and runtime:
        # - output_mode="window" gives one label per window;
        #   output_mode="per_timestep" gives one label per timestamp.
        # - window_size changes how much history each CNN example sees.
        # - batch_size changes how many examples are predicted at once.
        cnn_config=CNN_CONFIG,
        # These objects come from the CNN training cells above.
        model=model,
        device=device,
        channel_mean=channel_mean,
        channel_std=channel_std,
        # The helper auto-selects from `test_df` by default so this visual check
        # usually stays on held-out data. Set CNN_RANGE_START/CNN_RANGE_END in
        # the previous cell when you want to inspect a specific time period.
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        range_start=CNN_RANGE_START,
        range_end=CNN_RANGE_END,
        auto_select_test_range=CNN_AUTO_SELECT_TEST_RANGE,
        max_points_to_plot=CNN_MAX_POINTS_TO_PLOT,
        # These inputs tell the helper how to reload the selected raw rows and
        # rebuild the same model-ready columns used during training.
        metadata=metadata,
        row_cache_dir=ROW_CACHE_DIR,
        row_use_columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        class_labels=class_labels,
        # Plot labels should match the active dataset. For conductivity-plug
        # runs this means custom `ml_label` meanings rather than generic QC flags.
        plot_measurement_column=PLOT_MEASUREMENT_COLUMN,
        plot_secondary_column=PLOT_SECONDARY_COLUMN,
        flag_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
        label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
        flag_palette=DEFAULT_FLAG_PALETTE,
    )
    cnn_interval_metrics = cnn_demo_result["interval_metrics"]


### Reflection Questions: Baseline CNN

1. Which output mode is active here: one label per window or one label per timestamp? How does that change what each mistake means?
2. If `output_mode="window"`, does compressing a full window into one label hide short events? If `output_mode="per_timestep"`, are the predicted boundaries sharp enough to be useful?
3. Would you improve this result first by changing `window_size`, the window label rule, the input features, or the CNN architecture/training settings?

Add your own notes or follow-up questions here:

- 
- 
- 


### CNN: Tune The Network, Then Revisit The Problem Definition

We start by tuning the baseline window-classification CNN on the same target used above. That gives us a fair neural-network analogue to the Random Forest search.

Then we step back and ask a broader question:

- if feature engineering and target design mattered so much for trees, do they matter for neural networks too?

The answer turns out to be yes. The later CNN cells show that architecture knobs alone were not the whole story.


In [ ]:
# This small grid search tries a few CNN sizes and learning rates.
# Keep it compact because each trial trains a neural network.
CNN_SEARCH_SPACE = {
    "window_size": [128, 256],
    "label_reduction": ["worst"],
    "epochs": [12],
    "batch_size": [128],
    "learning_rate": [5e-4, 1e-3],
    "weight_decay": [1e-4],
    "patience": [4],
    "min_delta": [0.001],
    "dropout": [0.15],
    "conv_channels": [[32, 64], [64, 128]],
    "lr_decay_factor": [0.5],
    "lr_patience": [1],
    "gradient_clip_norm": [1.0],
    "num_workers": [0],
    "pin_memory": [True],
    "persistent_workers": [False],
    "prefetch_factor": [2],
}

# CNN_SEARCH_MODEL_ROW_LIMIT was set in the visible row-budget cell above.
# Set CNN_SEARCH_MODEL_ROW_LIMIT = None here if you want CNN search trials to use the full train split.
BEST_CNN_SEARCH_CONFIG = None

print({"RUN_HYPERPARAMETER_SEARCH": RUN_HYPERPARAMETER_SEARCH})
print({"DATA_FRACTION": DATA_FRACTION, "cnn_search_model_row_limit": CNN_SEARCH_MODEL_ROW_LIMIT})


In [ ]:
import gc

if RUN_HYPERPARAMETER_SEARCH:
    # Search on a representative training subset, while keeping capped validation/test slices fixed for comparison.
    cnn_search_train_selection_df = sample_frame_by_strategy(
        train_full_df,
        rows_limit=CNN_SEARCH_MODEL_ROW_LIMIT,
        sample_strategy=TRAIN_SUBSET_STRATEGY if TRAIN_SUBSET_STRATEGY != "full_train" else "time_spread",
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    cnn_search_valid_selection_df = evenly_spaced_take(
        valid_df.sort_values("Time UTC").reset_index(drop=True),
        RF_VALIDATION_MAX_ROWS,
    )
    cnn_search_test_selection_df = evenly_spaced_take(
        test_df.sort_values("Time UTC").reset_index(drop=True),
        RF_TEST_MAX_ROWS,
    )
    cnn_search_split_frames = materialize_reviewed_split_frames(
        selected_paths,
        {
            "train": cnn_search_train_selection_df,
            "validation": cnn_search_valid_selection_df,
            "test": cnn_search_test_selection_df,
        },
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    # The utility trains each candidate config and returns the best validation result.
    cnn_search_results, best_cnn_config, best_cnn_result = run_cnn_search_from_frames(
        cnn_search_split_frames,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        seed=SEED,
        search_space=CNN_SEARCH_SPACE,
        checkpoint_dir=ARTIFACT_DIR / "notebook_cnn_search",
    )
    display(cnn_search_results.head(10))
    # Store the winner so later cells can reuse or compare against it.
    BEST_CNN_SEARCH_CONFIG = best_cnn_config
    display(
        pd.DataFrame(
            [
                {
                    "model": "Baseline 1D CNN search winner",
                    "best_validation_f1": best_cnn_result["best_validation_f1"],
                    "search_rows": len(cnn_search_split_frames["train"]),
                    "config": BEST_CNN_SEARCH_CONFIG,
                }
            ]
        )
    )

    for _cnn_name in [
        "cnn_search_train_selection_df",
        "cnn_search_valid_selection_df",
        "cnn_search_test_selection_df",
        "cnn_search_split_frames",
        "cnn_search_results",
    ]:
        globals().pop(_cnn_name, None)
    gc.collect()
else:
    print("CNN hyperparameter search skipped. Keeping the baseline CNN settings.")

print({"BEST_CNN_SEARCH_CONFIG": BEST_CNN_SEARCH_CONFIG})


### CNN: Does The Same Lesson Hold?

We can ask the same question for neural networks that we asked for Random Forest:

- does hyperparameter tuning help?
- or do target definition and input representation matter even more?

The answer from the larger local GPU study was: **the same lesson still holds**.

CNN hyperparameters matter, but the strongest gains came from:

1. choosing a target that matched the operational problem better,
2. giving the network more informative inputs,
3. and then tuning the neural-network settings on top of that.


In [ ]:
# Load saved CNN study outputs if they exist, so the notebook can summarise larger offline runs.
cnn_result_frames = []
for csv_name, study_name in [
    ("cnn_optimization_final.csv", "final_screen"),
    ("cnn_optimization_finalists.csv", "large_retrain"),
    ("cnn_targeted_large_results.csv", "targeted_large_retrain"),
]:
    csv_path = ARTIFACT_DIR / csv_name
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame["study_name"] = study_name
        cnn_result_frames.append(frame)

if cnn_result_frames:
    # Combine all available study CSVs and rank by held-out test F1.
    cnn_results = pd.concat(cnn_result_frames, ignore_index=True, sort=False)
    cnn_results = cnn_results.sort_values(["test_f1", "validation_f1"], ascending=False).reset_index(drop=True)
    display(
        cnn_results[
            [
                "study_name",
                "strategy",
                "feature_set",
                "model_mode",
                "row_count",
                "window_size",
                "conv_channels",
                "validation_f1",
                "test_f1",
            ]
        ].head(12)
    )

    # These recommendations summarise which CNN variants were most reliable in the offline studies.
    robust_rows = [
        {
            "recommended_choice": "Best robust collapsed-target CNN",
            "target_strategy": "collapsed_1_34_9",
            "feature_set": "measurements_plus_deltas",
            "model_mode": "sequence",
            "window_size": 256,
            "why": "Best larger-sample CNN result; deltas capture sudden local changes, and 256 steps gives more temporal context.",
        },
        {
            "recommended_choice": "Best robust binary CNN",
            "target_strategy": "binary_issue",
            "feature_set": "measurements_plus_deltas_plus_rolling_time",
            "model_mode": "sequence",
            "window_size": 128,
            "why": "Strong for good-versus-issue decisions, but still below the tree baseline.",
        },
    ]
    display(pd.DataFrame(robust_rows))
else:
    print("CNN optimisation artifacts not found. Run the local CNN study scripts first.")

# Compare the best CNN studies with the tree baseline when that artifact is available.
comparison_path = ARTIFACT_DIR / "extratrees_target_strategy_comparison.csv"
if cnn_result_frames and comparison_path.exists():
    tree_results = pd.read_csv(comparison_path)
    comparison_rows = [
        {
            "target_strategy": "collapsed_1_34_9",
            "best_cnn_test_f1": 0.9932348935330678,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "collapsed_1_34_9", "test_f1"].iloc[0]
            ),
        },
        {
            "target_strategy": "binary_issue",
            "best_cnn_test_f1": 0.9855123210924848,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "binary_issue", "test_f1"].iloc[0]
            ),
        },
        {
            "target_strategy": "multiclass_1_3_4_9",
            "best_cnn_test_f1": 0.6952206110852922,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "multiclass_1_3_4_9", "test_f1"].iloc[0]
            ),
        },
    ]
    comparison_frame = pd.DataFrame(comparison_rows)
    comparison_frame["cnn_minus_tree"] = comparison_frame["best_cnn_test_f1"] - comparison_frame["best_tree_test_f1"]
    display(comparison_frame)


#### Why The Advanced CNN Defaults Below Use The Collapsed Target

The defaults in the next section are based on the most robust local GPU result we found, not just the highest one-off validation score.

Those defaults make three deliberate choices:

- **Target strategy:** dataset-aware by default
  For conductivity and turbidity, the advanced section starts from `collapsed_1_34_9`. For oxygen, it starts from `collapsed_12_34_9` so that `1/2` and `3/4` can travel together instead of reusing the CTD collapse blindly. For conductivity plugs, it starts from `raw_multiclass` because the `ml_label` classes follow a different scheme.
- **Input set:** `measurements_plus_deltas`
  Raw values tell us where the instrument is. Absolute deltas tell us when it changes abruptly. That combination was more reliable than raw values alone.
- **Model mode:** `sequence`
  QC is naturally a timestamp-level task, so per-timestamp prediction is a better conceptual match than forcing one label onto a whole window.

These are not magic settings. They are simply the choices that held up best when we reran the strongest candidates on larger samples.


### Advanced CNN Variant: Predict One QC Flag Per Timestamp

The baseline CNN above does **window classification**: one input window, one label.

That is simple and useful as a baseline, but it throws away some detail because many row-level target labels are compressed into one window label.

This advanced variant does **sequence labelling** instead:

- input: one fixed window of time-series data
- output: one QC prediction for each time step inside that window

This is often a better match for the real ONC QC task, because the operational question is usually closer to "what is the flag at this timestamp?" than "what is the single label for this whole chunk?"


#### Sequence-Labelling CNN Settings

This model uses almost the same ingredients as the baseline CNN, but the final layer produces a prediction at **every position in the window**.

Important differences from the earlier CNN:

- `sequence_target_strategy` lets us change the problem definition itself.
- `sequence_feature_set` lets us decide what information the CNN receives.
- `window_size` still controls context, but now every row in the window gets its own prediction.
- evaluation is done per timestamp, not per window.

The defaults here mirror the strongest **robust** CNN run we found for the current problem framing. For conductivity and turbidity that means collapsed `1 / 34 / 9`; for oxygen it starts from `12 / 34 / 9`; for conductivity plugs it stays in raw multiclass mode until you decide whether a custom collapse is more helpful.


In [ ]:
# The sequence-labelling CNN predicts a QC label at each timestamp instead of one label per window.
SEQUENCE_CNN_TARGET_STRATEGY = DEFAULT_SEQUENCE_TARGET_STRATEGY
SEQUENCE_CNN_FEATURE_SET = "measurements_plus_deltas"

# These settings use a slightly larger CNN because per-timestep prediction is a harder task.
SEQUENCE_CNN_CONFIG = {
    "window_size": 256,
    "epochs": 15,
    "batch_size": 192,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.0,
    "dropout": 0.15,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    "conv_channels": [64, 128, 256],
}

# Loader settings mirror the baseline CNN settings for easier comparison.
SEQUENCE_CNN_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": True,
    "persistent_workers": False,
    "prefetch_factor": 2,
}

SEQUENCE_CNN_MODEL_PATH = ARTIFACT_DIR / "best_sequence_label_cnn_checkpoint.pt"

# Per-timestamp windows are much heavier than one-label-per-window CNN data.
# These caps still follow the visible DATA_FRACTION row/window caps above, but
# keep this advanced section inside the 16 GB workshop job size.
SEQUENCE_CNN_TRAIN_MAX_ROWS = min(TRAIN_SUBSET_MAX_ROWS if TRAIN_SUBSET_MAX_ROWS is not None else 80_000, 80_000)
SEQUENCE_CNN_VALIDATION_MAX_ROWS = min(
    RF_VALIDATION_MAX_ROWS if RF_VALIDATION_MAX_ROWS is not None else 40_000,
    40_000,
)
SEQUENCE_CNN_TEST_MAX_ROWS = min(
    RF_TEST_MAX_ROWS if RF_TEST_MAX_ROWS is not None else 40_000,
    40_000,
)

print(
    {
        "sequence_target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
        "sequence_feature_set": SEQUENCE_CNN_FEATURE_SET,
        "sequence_train_max_rows": SEQUENCE_CNN_TRAIN_MAX_ROWS,
        "sequence_validation_max_rows": SEQUENCE_CNN_VALIDATION_MAX_ROWS,
        "sequence_test_max_rows": SEQUENCE_CNN_TEST_MAX_ROWS,
    }
)
display(pd.Series(SEQUENCE_CNN_CONFIG, name="value").rename_axis("sequence_cnn_parameter").to_frame())
display(pd.Series(SEQUENCE_CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Step 1: Build The Target And Feature Set We Actually Want To Learn

For the baseline CNN, we reduced one whole window to a single label. Here we keep the label sequence itself, so a `256`-step input window has `256` targets.

We also make two explicit choices that came from the larger optimisation study:

- apply a target strategy before training,
- and decide which engineered inputs we want the CNN to see.


In [ ]:
import gc

# The baseline CNN section has already finished by this point. Releasing its
# tensors/loaders before building per-timestep windows keeps 16 GB FIR jobs stable.
for _cnn_name in [
    "cnn_bundle",
    "X_train", "X_valid", "X_test", "y_train", "y_valid", "y_test",
    "train_targets_tensor", "valid_targets_tensor", "test_targets_tensor",
    "train_dataset", "valid_dataset", "test_dataset",
    "train_loader", "valid_loader", "test_loader",
    "cnn_model", "optimizer", "scheduler", "best_state", "loss_fn",
    "sequence_split_frames_materialized",
    "channel_mean", "channel_std",
]:
    globals().pop(_cnn_name, None)
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if not CNN_READY:
    SEQUENCE_CNN_RUN = False
    print("Sequence-labelling CNN skipped because PyTorch is not available.")
else:
    # Per-timestep labels create many more target values than the baseline CNN.
    # Build this section from bounded split selections instead of reusing the
    # full row-level frames from the baseline CNN.
    sequence_train_selection_df = sample_frame_by_strategy(
        train_full_df,
        rows_limit=SEQUENCE_CNN_TRAIN_MAX_ROWS,
        sample_strategy=TRAIN_SUBSET_STRATEGY if TRAIN_SUBSET_STRATEGY != "full_train" else "time_spread",
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    sequence_validation_selection_df = evenly_spaced_take(
        valid_df.sort_values("Time UTC").reset_index(drop=True),
        SEQUENCE_CNN_VALIDATION_MAX_ROWS,
    )
    sequence_test_selection_df = evenly_spaced_take(
        test_df.sort_values("Time UTC").reset_index(drop=True),
        SEQUENCE_CNN_TEST_MAX_ROWS,
    )
    sequence_source_frames = materialize_reviewed_split_frames(
        selected_paths,
        {
            "train": sequence_train_selection_df,
            "validation": sequence_validation_selection_df,
            "test": sequence_test_selection_df,
        },
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )

    def build_sequence_feature_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
        feature_df = frame.sort_values("Time UTC").reset_index(drop=True).copy()
        # Absolute deltas help the CNN notice sudden changes between adjacent readings.
        for column in measurement_columns:
            feature_df[f"{column} abs_delta"] = feature_df[column].diff().abs().fillna(0.0)

        # Rolling context features give each timestamp a small amount of local history.
        context_source = feature_df[["Time UTC", *measurement_columns]].copy()
        context_frame, context_columns = add_temporal_context_features(
            context_source,
            measurement_columns=measurement_columns,
            lag_steps=(),
            rolling_windows=(5,),
        )
        for column in context_columns:
            feature_df[column] = context_frame[column]

        # Cyclical time features avoid artificial jumps at midnight or year boundaries.
        radians_hour = 2.0 * np.pi * feature_df["Time UTC"].dt.hour / 24.0
        radians_doy = 2.0 * np.pi * feature_df["Time UTC"].dt.dayofyear / 366.0
        feature_df["hour_sin"] = np.sin(radians_hour)
        feature_df["hour_cos"] = np.cos(radians_hour)
        feature_df["doy_sin"] = np.sin(radians_doy)
        feature_df["doy_cos"] = np.cos(radians_doy)

        target_df, _, _ = apply_target_strategy(
            feature_df.dropna(subset=[TARGET_FLAG]).copy(),
            TARGET_FLAG,
            SEQUENCE_CNN_TARGET_STRATEGY,
            issue_labels=ISSUE_LABELS,
        )
        if "model_target" in target_df.columns:
            target_df = target_df.drop(columns=["model_target"])
        target_df = target_df.rename(columns={"strategy_target": "model_target"})
        return target_df, context_columns

    sequence_split_frames_full = {}
    sequence_context_columns = []
    for split_name, source_frame in sequence_source_frames.items():
        sequence_split_frames_full[split_name], split_context_columns = build_sequence_feature_frame(source_frame)
        if not sequence_context_columns:
            sequence_context_columns = split_context_columns

    # Feature sets let us compare raw measurements against richer temporal representations.
    sequence_feature_sets = {
        "measurements": measurement_columns,
        "measurements_plus_deltas": measurement_columns + [f"{column} abs_delta" for column in measurement_columns],
        "measurements_plus_deltas_plus_rolling_time": measurement_columns
        + [f"{column} abs_delta" for column in measurement_columns]
        + sequence_context_columns
        + ["hour_sin", "hour_cos", "doy_sin", "doy_cos"],
    }

    if SEQUENCE_CNN_FEATURE_SET not in sequence_feature_sets:
        raise ValueError(
            f"Unsupported sequence feature set: {SEQUENCE_CNN_FEATURE_SET}. Choose from {list(sequence_feature_sets)}."
        )

    _, sequence_class_labels, sequence_average = apply_target_strategy(
        sequence_source_frames["train"].dropna(subset=[TARGET_FLAG]).copy(),
        TARGET_FLAG,
        SEQUENCE_CNN_TARGET_STRATEGY,
        issue_labels=ISSUE_LABELS,
    )
    sequence_input_columns = sequence_feature_sets[SEQUENCE_CNN_FEATURE_SET]
    sequence_split_frames = {
        "train": sequence_split_frames_full["train"],
        "validation": sequence_split_frames_full["validation"],
        "test": sequence_split_frames_full["test"],
    }

    sequence_window_size = SEQUENCE_CNN_CONFIG["window_size"]
    sequence_task_mode = "binary" if sequence_average == "binary" else "multiclass"
    # Build per-timestep windows: X is a sequence of features, y is a sequence of labels.
    sequence_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames,
        measurement_columns=sequence_input_columns,
        target_column="model_target",
        task_mode=sequence_task_mode,
        output_mode="per_timestep",
        window_size=sequence_window_size,
        label_reduction="worst",
    )

    sequence_class_labels = sequence_bundle.class_labels
    # As with the baseline CNN, each split needs complete windows and enough labels to learn.
    if sequence_task_mode == "multiclass" and len(sequence_class_labels) < 2:
        SEQUENCE_CNN_RUN = False
        print(
            {
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "skip_reason": "Need at least two active classes to train the sequence-labelling CNN in multiclass mode.",
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
                "active_labels": sequence_class_labels,
            }
        )
    elif len(sequence_bundle.X_train) == 0 or len(sequence_bundle.X_valid) == 0 or len(sequence_bundle.X_test) == 0:
        SEQUENCE_CNN_RUN = False
        print(
            {
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
            }
        )
    else:
        # Conv1D expects channel-first arrays: (windows, channels, time).
        X_train_seq = np.transpose(sequence_bundle.X_train, (0, 2, 1))
        X_valid_seq = np.transpose(sequence_bundle.X_valid, (0, 2, 1))
        X_test_seq = np.transpose(sequence_bundle.X_test, (0, 2, 1))
        y_train_seq = sequence_bundle.y_train
        y_valid_seq = sequence_bundle.y_valid
        y_test_seq = sequence_bundle.y_test

        # Normalise from the training split only, then apply the same scaling to validation/test.
        channel_mean = X_train_seq.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train_seq.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train_seq = (X_train_seq - channel_mean) / channel_std
        X_valid_seq = (X_valid_seq - channel_mean) / channel_std
        X_test_seq = (X_test_seq - channel_mean) / channel_std

        SEQUENCE_CNN_RUN = True
        print(
            {
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "fixed_split_block_rows": FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "rows_used_train": int(len(sequence_source_frames["train"])),
                "rows_used_validation": int(len(sequence_source_frames["validation"])),
                "rows_used_test": int(len(sequence_source_frames["test"])),
                "windows_total": int(len(X_train_seq) + len(X_valid_seq) + len(X_test_seq)),
                "train_windows": int(len(X_train_seq)),
                "validation_windows": int(len(X_valid_seq)),
                "test_windows": int(len(X_test_seq)),
                "window_size": int(sequence_window_size),
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
                "channel_count": int(len(sequence_input_columns)),
                "sequence_task_mode": sequence_task_mode,
            }
        )

    for _sequence_name in [
        "sequence_train_selection_df",
        "sequence_validation_selection_df",
        "sequence_test_selection_df",
        "sequence_source_frames",
        "sequence_split_frames_full",
        "sequence_split_frames",
        "sequence_bundle",
    ]:
        globals().pop(_sequence_name, None)
    gc.collect()


### Step 2: Build A CNN That Emits One Logit Sequence

The architecture is still a 1D CNN, but the classifier head is now a `1x1` convolution. That lets the network output one class distribution per time step.

In the robust local run, three choices mattered:

- a deeper channel stack (`[64, 128, 256]`) gave the network enough capacity,
- the `1x1` head preserved one prediction per timestamp,
- and class-weighted losses helped the model pay attention to the rarer QC states.


In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN model setup skipped.")
else:
    # Seed PyTorch and choose GPU when available.
    torch.manual_seed(SEED)
    sequence_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if sequence_device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    # This CNN preserves the time axis: every input timestamp receives an output logit.
    class SequenceLabelCNN(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            # Build a stack of Conv1D blocks with batch norm and GELU activations.
            blocks = []
            in_channels = channels
            kernel_sizes = [7, 5, 3]
            for index, out_channels in enumerate(SEQUENCE_CNN_CONFIG["conv_channels"]):
                kernel_size = kernel_sizes[min(index, len(kernel_sizes) - 1)]
                padding = kernel_size // 2
                blocks.extend(
                    [
                        nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding),
                        nn.BatchNorm1d(out_channels),
                        nn.GELU(),
                        nn.Dropout(SEQUENCE_CNN_CONFIG["dropout"]),
                    ]
                )
                in_channels = out_channels
            self.features = nn.Sequential(*blocks)
            # A 1x1 head converts learned features at each timestamp into class logits.
            self.head = nn.Conv1d(in_channels, output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.head(self.features(x))

    # Pick the target tensor dtype and loss function for multiclass vs binary targets.
    if sequence_task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train_seq).long()
        valid_targets_tensor = torch.from_numpy(y_valid_seq).long()
        test_targets_tensor = torch.from_numpy(y_test_seq).long()
        # Flattened class counts weight rare timestamp labels during training.
        class_counts = np.bincount(y_train_seq.reshape(-1), minlength=len(sequence_class_labels)).clip(min=1)
        class_weights = y_train_seq.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=sequence_device))
        output_dim = len(sequence_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train_seq.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid_seq.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test_seq.astype(np.float32))
        # Binary mode uses pos_weight for rare issue timestamps.
        positive_count = max(float(y_train_seq.sum()), 1.0)
        negative_count = max(float(y_train_seq.size - y_train_seq.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=sequence_device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    # DataLoaders batch windows while preserving the full target sequence for each window.
    train_dataset = TensorDataset(torch.from_numpy(X_train_seq).float(), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(X_valid_seq).float(), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(X_test_seq).float(), test_targets_tensor)

    sequence_loader_kwargs = {}
    num_workers = int(SEQUENCE_CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        sequence_loader_kwargs["num_workers"] = num_workers
        sequence_loader_kwargs["persistent_workers"] = bool(SEQUENCE_CNN_LOADER_CONFIG["persistent_workers"])
        sequence_loader_kwargs["prefetch_factor"] = int(SEQUENCE_CNN_LOADER_CONFIG["prefetch_factor"])
    if SEQUENCE_CNN_LOADER_CONFIG["pin_memory"]:
        sequence_loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(sequence_loader_kwargs.get("pin_memory", False) and sequence_device.type == "cuda")

    train_loader_seq = DataLoader(train_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=True, **sequence_loader_kwargs)
    valid_loader_seq = DataLoader(valid_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=False, **sequence_loader_kwargs)
    test_loader_seq = DataLoader(test_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=False, **sequence_loader_kwargs)

    # Create the model, optimiser, and validation-driven learning-rate scheduler.
    sequence_model = SequenceLabelCNN(channels=len(sequence_input_columns), output_dim=output_dim).to(sequence_device)
    sequence_optimizer = torch.optim.AdamW(
        sequence_model.parameters(),
        lr=SEQUENCE_CNN_CONFIG["learning_rate"],
        weight_decay=SEQUENCE_CNN_CONFIG["weight_decay"],
    )
    sequence_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        sequence_optimizer,
        mode="max",
        factor=SEQUENCE_CNN_CONFIG["lr_decay_factor"],
        patience=SEQUENCE_CNN_CONFIG["lr_patience"],
    )

    # Track the best validation checkpoint separately from the final epoch.
    sequence_best_metric = -np.inf
    sequence_best_epoch = 0
    sequence_patience_counter = 0
    sequence_history = []
    sequence_best_state = None

    # One epoch returns loss plus flattened timestamp predictions for F1 scoring.
    def run_sequence_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        sequence_model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(sequence_device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(sequence_device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = sequence_model(batch_x)
                # CrossEntropyLoss expects logits shaped (batch, classes, time).
                if sequence_task_mode == "multiclass":
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = logits.argmax(dim=1)
                else:
                    logits = logits.squeeze(1)
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    # Standard PyTorch training step: zero gradients, backpropagate, clip, update.
                    sequence_optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if SEQUENCE_CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(sequence_model.parameters(), SEQUENCE_CNN_CONFIG["gradient_clip_norm"])
                    sequence_optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(sequence_device),
            "train_batches": len(train_loader_seq),
            "validation_batches": len(valid_loader_seq),
            "test_batches": len(test_loader_seq),
            "feature_count": len(sequence_input_columns),
            "feature_columns_preview": sequence_input_columns[:8],
        }
    )


### Step 3: Train And Monitor Per-Timestamp Performance

The loop is familiar, but now the validation metric is computed across **all timestamps in the validation windows** rather than one prediction per window.

This is also where we make the usual training choices explicit:

- monitor validation F1 after every epoch,
- keep the best checkpoint,
- stop early when validation no longer improves.


In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN fit skipped.")
else:
    # Train and monitor validation F1 after every epoch.
    for epoch in range(1, SEQUENCE_CNN_CONFIG["epochs"] + 1):
        # Training updates weights; validation only measures performance.
        train_loss, _, _ = run_sequence_epoch(train_loader_seq, training=True)
        valid_loss, valid_preds, valid_targets = run_sequence_epoch(valid_loader_seq, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(sequence_task_mode), zero_division=0)
        # Validation F1 controls both learning-rate decay and checkpoint selection.
        sequence_scheduler.step(valid_f1)
        sequence_history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": sequence_optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        # Save the best model rather than assuming the final epoch is best.
        if valid_f1 > sequence_best_metric + SEQUENCE_CNN_CONFIG["min_delta"]:
            sequence_best_metric = valid_f1
            sequence_best_epoch = epoch
            sequence_patience_counter = 0
            sequence_best_state = copy.deepcopy(sequence_model.state_dict())
            torch.save(
                {
                    "model_state_dict": sequence_best_state,
                    "task_mode": sequence_task_mode,
                    "class_labels": sequence_class_labels,
                    "feature_columns": sequence_input_columns,
                    "window_size": sequence_window_size,
                    "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                    "config": {**SEQUENCE_CNN_CONFIG, **SEQUENCE_CNN_LOADER_CONFIG},
                },
                SEQUENCE_CNN_MODEL_PATH,
            )
        else:
            # Stop when validation F1 has not improved for several epochs.
            sequence_patience_counter += 1
            if sequence_patience_counter >= SEQUENCE_CNN_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### Step 4: Evaluate Per-Timestamp Predictions

This report answers a different question from the earlier CNN section. It tells us how well the model predicts the target label at each timestamp inside the window, not just the single "worst" label for the whole window.

At the end we also compare the CNN result to the saved tree baseline for the same target strategy, so we can see whether the added sequence modelling actually paid off.


In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN evaluation skipped.")
else:
    # Evaluate the best validation checkpoint on the untouched test windows.
    if sequence_best_state is None:
        raise RuntimeError("Sequence-labelling CNN training did not produce a valid checkpoint")

    sequence_model.load_state_dict(sequence_best_state)
    test_loss, test_preds, test_targets = run_sequence_epoch(test_loader_seq, training=False)
    sequence_test_f1 = f1_score(
        test_targets,
        test_preds,
        average=report_average(sequence_task_mode),
        zero_division=0,
    )

    print(pd.DataFrame(sequence_history).tail())
    print(
        {
            "timestamps_in_test_windows": int(len(test_targets)),
            "device": str(sequence_device),
            "best_validation_f1": float(sequence_best_metric),
            "best_epoch": sequence_best_epoch,
            "test_loss": float(test_loss),
            "test_f1": float(sequence_test_f1),
            "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
            "feature_set": SEQUENCE_CNN_FEATURE_SET,
            "saved_model": str(SEQUENCE_CNN_MODEL_PATH),
        }
    )

    # Build report labels in the same output space the model learned.
    if sequence_task_mode == "multiclass":
        report_labels = list(range(len(sequence_class_labels)))
        report_names = [str(label) for label in sequence_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    # Plot optimisation history and normalised confusion matrix for timestamp-level errors.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(sequence_history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Sequence-labelling CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Per-timestamp CNN confusion matrix")
    plt.tight_layout()
    plt.show()

    # If the tree comparison artifact exists, show whether this CNN improved on the same target strategy.
    strategy_results_path = ARTIFACT_DIR / "extratrees_target_strategy_comparison.csv"
    if strategy_results_path.exists():
        strategy_results = pd.read_csv(strategy_results_path)
        matching_tree_rows = strategy_results[strategy_results["strategy"] == SEQUENCE_CNN_TARGET_STRATEGY]
        if not matching_tree_rows.empty:
            tree_test_f1 = float(matching_tree_rows["test_f1"].iloc[0])
            print(
                {
                    "cnn_test_f1": float(sequence_test_f1),
                    "tree_test_f1_for_same_target": tree_test_f1,
                    "cnn_minus_tree": float(sequence_test_f1 - tree_test_f1),
                }
            )


#### Date-Range Demo: Per-Timestamp CNN Predictions on a Specific Interval

This advanced CNN predicts one QC class per timestamp, so this is the first neural-network demo in the notebook that lines up directly with the row-level QC task.

The selected interval below shows true and predicted timestamp-level flags side by side.


In [ ]:
# Leave these as None to auto-select a test interval, or set explicit UTC timestamps.
SEQUENCE_CNN_RANGE_START = None
SEQUENCE_CNN_RANGE_END = None
SEQUENCE_CNN_AUTO_SELECT_TEST_RANGE = True
SEQUENCE_CNN_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "SEQUENCE_CNN_RANGE_START": SEQUENCE_CNN_RANGE_START,
        "SEQUENCE_CNN_RANGE_END": SEQUENCE_CNN_RANGE_END,
        "SEQUENCE_CNN_AUTO_SELECT_TEST_RANGE": SEQUENCE_CNN_AUTO_SELECT_TEST_RANGE,
        "SEQUENCE_CNN_MAX_POINTS_TO_PLOT": SEQUENCE_CNN_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
# Select a held-out time range for a qualitative prediction plot.
sequence_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=SEQUENCE_CNN_RANGE_START,
    end=SEQUENCE_CNN_RANGE_END,
    auto_select=SEQUENCE_CNN_AUTO_SELECT_TEST_RANGE,
    max_points=SEQUENCE_CNN_MAX_POINTS_TO_PLOT,
)

# Load only the row-level data needed for the selected interval.
sequence_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=sequence_range_selection["start"],
    end=sequence_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)

if sequence_interval_rows.empty:
    print("No row-level data was found in the requested sequence-labelling CNN range.")
else:
    # Rebuild the same engineered features used during sequence-CNN training.
    sequence_interval_feature_df = sequence_interval_rows.copy()
    # Delta features highlight abrupt local changes in the selected interval.
    for column in measurement_columns:
        sequence_interval_feature_df[f"{column} abs_delta"] = sequence_interval_feature_df[column].diff().abs().fillna(0.0)

    # Rolling context features must be recreated before the model can score the interval.
    sequence_context_source = sequence_interval_feature_df[["Time UTC", *measurement_columns]].copy()
    sequence_context_frame, _ = add_temporal_context_features(
        sequence_context_source,
        measurement_columns=measurement_columns,
        lag_steps=(),
        rolling_windows=(5,),
    )
    for column in sequence_context_columns:
        sequence_interval_feature_df[column] = sequence_context_frame[column]

    # Recreate cyclical time features with the same formula used in training.
    radians_hour = 2.0 * np.pi * sequence_interval_feature_df["Time UTC"].dt.hour / 24.0
    radians_doy = 2.0 * np.pi * sequence_interval_feature_df["Time UTC"].dt.dayofyear / 366.0
    sequence_interval_feature_df["hour_sin"] = np.sin(radians_hour)
    sequence_interval_feature_df["hour_cos"] = np.cos(radians_hour)
    sequence_interval_feature_df["doy_sin"] = np.sin(radians_doy)
    sequence_interval_feature_df["doy_cos"] = np.cos(radians_doy)

    # Apply the same target strategy so plotted labels match the trained model output.
    sequence_interval_target_df, _, _ = apply_target_strategy(
        sequence_interval_feature_df.dropna(subset=[TARGET_FLAG]).copy(),
        TARGET_FLAG,
        SEQUENCE_CNN_TARGET_STRATEGY,
        issue_labels=ISSUE_LABELS,
    )
    sequence_interval_target_df = sequence_interval_target_df.rename(columns={"strategy_target": "model_target"})
    sequence_interval_target_df = sequence_interval_target_df[
        (sequence_interval_target_df["Time UTC"] >= sequence_range_selection["start"])
        & (sequence_interval_target_df["Time UTC"] <= sequence_range_selection["end"])
    ].reset_index(drop=True)

    # Convert the interval into complete windows before running the CNN.
    sequence_interval_bundle = build_sequence_label_interval_data(
        sequence_interval_target_df,
        feature_columns=sequence_input_columns,
        target_column="model_target",
        window_size=sequence_window_size,
    )

    if len(sequence_interval_bundle["raw_sequences"]) == 0:
        print("The selected interval is shorter than one full sequence-labelling window after preprocessing, so the demo is skipped.")
    else:
        # Use training-split normalisation stored in channel_mean/channel_std during prediction.
        sequence_predicted_labels = predict_sequence_label_cnn(
            sequence_model,
            sequence_interval_bundle["raw_sequences"],
            task_mode=sequence_task_mode,
            class_labels=sequence_class_labels,
            device=str(sequence_device),
            channel_mean=channel_mean,
            channel_std=channel_std,
            batch_size=SEQUENCE_CNN_CONFIG["batch_size"],
        )

        # Flatten window outputs so metrics and interval bands are calculated per timestamp.
        sequence_flat_predictions = sequence_predicted_labels.reshape(-1)
        sequence_flat_targets = sequence_interval_bundle["raw_targets"].reshape(-1)
        sequence_flat_times = pd.to_datetime(sequence_interval_bundle["raw_times"].reshape(-1), utc=True)
        sequence_plot_frame = pd.DataFrame(
            {
                "Time UTC": sequence_flat_times,
                "true_label": sequence_flat_targets,
                "predicted_label": sequence_flat_predictions,
            }
        )
        sequence_interval_origin = infer_interval_origin(
            sequence_range_selection["start"],
            sequence_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        sequence_interval_metrics = compute_interval_classification_metrics(
            sequence_flat_targets,
            sequence_flat_predictions,
            labels=sequence_class_labels,
            average=report_average(sequence_task_mode),
            target_names=[str(label) for label in sequence_class_labels],
        )
        sequence_plot_palette = DEFAULT_FLAG_PALETTE if sequence_task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
        # Convert adjacent labels into time intervals to make the overlay plot readable.
        sequence_true_intervals = build_labeled_intervals(
            sequence_plot_frame,
            time_column="Time UTC",
            label_column="true_label",
        )
        sequence_pred_intervals = build_labeled_intervals(
            sequence_plot_frame,
            time_column="Time UTC",
            label_column="predicted_label",
        )

        print(
            {
                "selection_mode": sequence_range_selection["selection_mode"],
                "selected_priority_flag": sequence_range_selection["selected_label"],
                "interval_origin": sequence_interval_origin,
                "range_start": sequence_range_selection["start"].isoformat(),
                "range_end": sequence_range_selection["end"].isoformat(),
                "timestamps_in_interval": int(len(sequence_plot_frame)),
                "interval_f1": sequence_interval_metrics["f1"],
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
            }
        )
        print(sequence_interval_metrics["report_text"])
        display(
            pd.DataFrame(
                {
                    "true_count": pd.Series(sequence_flat_targets).value_counts().sort_index(),
                    "predicted_count": pd.Series(sequence_flat_predictions).value_counts().sort_index(),
                }
            ).fillna(0).astype(int)
        )

        sequence_demo_figure = plot_time_series_with_bands(
            sequence_interval_target_df,
            band_specs=[
                {"title": f"True timestamp {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": sequence_true_intervals, "palette": sequence_plot_palette},
                {"title": f"Sequence CNN {FLAG_EXAMPLE_DISPLAY_NAME} prediction", "intervals": sequence_pred_intervals, "palette": sequence_plot_palette},
            ],
            measurement_column=PLOT_MEASUREMENT_COLUMN,
            secondary_column=PLOT_SECONDARY_COLUMN,
            max_points=SEQUENCE_CNN_MAX_POINTS_TO_PLOT,
            title="Sequence-labelling CNN predictions on a selected time range",
            label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
            legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
        )
        plt.show()

        sequence_cm_fig, sequence_cm_ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix=sequence_interval_metrics["confusion_matrix"],
            display_labels=sequence_interval_metrics["display_labels"],
        ).plot(ax=sequence_cm_ax, cmap="Blues", colorbar=False)
        sequence_cm_ax.set_title("Sequence-labelling CNN confusion matrix on the selected range")
        plt.tight_layout()
        plt.show()


### Reflection Questions: Sequence-labelling CNN

1. Where does predicting one flag per timestamp clearly help more than the earlier window-classification CNN?
2. Do the remaining mistakes look like a feature-set problem, a target-strategy problem, or a pure optimisation problem?
3. If you had to choose one next experiment for this model, would you change the input features, the target strategy, or the sequence length?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 7 — Transformer

The transformer keeps the same supervised task but changes how the model uses context inside each sequence window.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### Sequential Model: Small Transformer Encoder

If you have used ChatGPT or seen a text-to-image model, you have already interacted with a **transformer**.

Transformers were introduced in 2017 in *Attention Is All You Need*, and they quickly became the dominant architecture for sequence modelling. They are most famous in language, but the underlying math is not tied to text. The same ideas are useful for scalar sensor streams, audio, and video.

A simple reason they matter is that they replaced the older "read one step, then the next step" style used by RNNs and LSTMs.

- **RNN/LSTM:** process sequences step by step, which makes training slower and can make long-range context fade.
- **Transformer:** processes the full window together and learns which positions should pay attention to which other positions.

That core operation is called **self-attention**. In plain language, each timestep asks: *which other timesteps in this window are most relevant to understanding me right now?*

For ONC-style data, that is appealing whenever a local measurement only makes sense in a wider context, such as linking a short anomaly to something that happened much earlier or later in the same window.

In this notebook we use a small **encoder-only transformer** for classification. We are not generating text, so we only keep the encoder side and attach a classifier on top.

If you want a fuller explanation after this section, here are the most useful references:

- [ONC Confluence: Transformers overview](https://internal.oceannetworks.ca/x/mwDcDg)
- [3blue1brown: Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


![Transformer model architecture](../assets/session1/transformer_model_architecture.png)

This is the big-picture map. The original transformer has:

- an **encoder stack** on the left,
- a **decoder stack** on the right.

In this notebook we only use the **encoder stack** and attach a classifier on top. So when you look at the rest of the section, focus mostly on the left half of this picture rather than trying to memorize the whole diagram.

Image credit: Arz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:The-Transformer-model-architecture.png](https://commons.wikimedia.org/wiki/File:The-Transformer-model-architecture.png)


![Detailed encoder self-attention diagram](../assets/session1/transformer_encoder_self_attention_detailed.png)

This picture zooms in on the core idea of **self-attention**.

You do not need to memorize every symbol here. The important story is:

- each position creates a few learned views of itself,
- those views are compared against other positions,
- the model turns those comparisons into attention weights,
- then it blends information from the whole window into a new representation.

In plain language: a timestep can decide which other timesteps are worth listening to.

If you want the clearest visual explanation of this idea, the 3blue1brown video linked above is the best companion resource for this section.

Image credit: dvgodoy, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Encoder self-attention, detailed diagram.png](https://commons.wikimedia.org/wiki/File:Encoder_self-attention,_detailed_diagram.png)


![Positional encoding heatmap](../assets/session1/transformer_positional_encoding.png)

One natural question is: if attention can compare all positions to all other positions, how does the model know **where** each timestep sits in the sequence?

That is the job of **positional encoding**.

This heatmap shows that each position gets its own pattern added to the input representation. That gives the transformer access to order information, so it can tell the difference between "this happened early in the window" and "this happened later".

Image credit: Mdbartosz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Positional encoding.png](https://commons.wikimedia.org/wiki/File:Positional_encoding.png)


### Transformer Settings

Main model variables:

- `window_size`: number of time steps in each window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `d_model`: internal embedding size for each position.
- `nhead`: number of attention heads. This must divide `d_model`.
- `num_layers`: number of stacked encoder layers.
- `dim_feedforward`: hidden size of the feed-forward sublayer inside each encoder block.
- `dropout`: dropout used inside the model.
- `pooling`: how we reduce the sequence to one vector for classification. This only matters when `output_mode="window"`.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Training variables:

- `epochs`, `batch_size`, `learning_rate`, `weight_decay`
- `patience`, `min_delta`
- `lr_decay_factor`, `lr_patience`
- `gradient_clip_norm`

Practical note:

- larger `window_size`, `d_model`, or `num_layers` can improve context capacity,
- but they also increase memory use and training time.
- `num_workers > 0` can also increase memory use a lot on Jupyter kernels, because worker
  processes may copy parts of the parent process. Keep it at `0` unless you know the
  environment has headroom.
- `pin_memory` mainly helps when you are actually training on a CUDA device.

Dataset-aware default:

- the turbidity profile starts in `"per_timestep"` mode because the most interesting events are short and windows are often mixed,
- the other profiles start in `"window"` mode so you can begin with one simpler sequence-classification baseline.

A simple mental model for the main knobs:

- `window_size` controls how much time context the model can see,
- `d_model` controls how much representational space each timestep gets,
- `nhead` controls how many different attention patterns the model can learn in parallel,
- `num_layers` controls how many times the sequence is reprocessed with attention.


In [ ]:
TRANSFORMER_CONFIG = {
    "window_size": 128,
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.2,
    "pooling": "mean",
    "label_reduction": "worst",
    "epochs": 8,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
}

_loader_torch = globals().get("torch", None)
TRANSFORMER_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(TRANSFORMER_CONFIG, name="value").rename_axis("transformer_parameter").to_frame())
display(pd.Series(TRANSFORMER_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Transformer Step 1: Prepare Windowed Sequence Data

The transformer uses the same fixed split as the CNN, but it keeps the sequence in its original time-major layout.

In other words:

- CNN sees `[batch, channels, time]`,
- transformer sees `[batch, time, features]`.

Just like the CNN, the transformer here uses windows of the selected measurement columns rather than the full Random Forest feature table. So it gets temporal context directly from the sequence, not from handcrafted calendar or delta features.

The `output_mode` toggle also matters here:

- `"window"` trains one prediction for the whole window,
- `"per_timestep"` trains one prediction at every position in the window.

That makes this step a nice comparison point between two sequence models trained on the same scalar QC problem.


### Transformer Step 0: Release Neural Model Memory And Load PyTorch

If you ran the CNN section just before this one, the notebook may still be holding large tensors, `DataLoader` objects, and model state.

This cleanup cell frees those objects before the transformer starts. It also imports PyTorch here so the transformer section can be run even if you skipped the CNN cells.


In [ ]:
import gc

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
except ImportError as exc:
    PYTORCH_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)

# Clear large sequence-model objects left behind by earlier sections before
# building transformer windows. This is especially important on shared
# Jupyter kernels where CPU RAM is limited.
released_names = []
for name in [
    "cnn_bundle",
    "sequence_split_frames_materialized",
    "train_dataset",
    "valid_dataset",
    "test_dataset",
    "train_loader",
    "valid_loader",
    "test_loader",
    "train_targets_tensor",
    "valid_targets_tensor",
    "test_targets_tensor",
    "X_train",
    "X_valid",
    "X_test",
    "y_train",
    "y_valid",
    "y_test",
    "model",
    "optimizer",
    "scheduler",
    "loss_fn",
    "best_state",
    "history",
]:
    if name in globals():
        del globals()[name]
        released_names.append(name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    {
        "released_objects": released_names,
        "released_count": len(released_names),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }
)


In [ ]:
if not globals().get("PYTORCH_READY", False):
    TRANSFORMER_RUN = False
    print("Transformer training cell skipped because PyTorch is not available in the current notebook state.")
else:
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    transformer_window_size = TRANSFORMER_CONFIG["window_size"]
    TRANSFORMER_OUTPUT_MODE = TRANSFORMER_CONFIG["output_mode"]
    transformer_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=TRANSFORMER_OUTPUT_MODE,
        window_size=transformer_window_size,
        label_reduction=TRANSFORMER_CONFIG["label_reduction"],
    )
    transformer_class_labels = transformer_bundle.class_labels

    if task_mode == "multiclass" and len(transformer_class_labels) < 2:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "windows_total": int(len(transformer_bundle.X_train) + len(transformer_bundle.X_valid) + len(transformer_bundle.X_test)),
                "active_labels": transformer_class_labels,
                "skip_reason": "Need at least two active classes to train the transformer in multiclass mode.",
            }
        )
    elif len(transformer_bundle.X_train) == 0 or len(transformer_bundle.X_valid) == 0 or len(transformer_bundle.X_test) == 0:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        X_train = transformer_bundle.X_train
        X_valid = transformer_bundle.X_valid
        X_test = transformer_bundle.X_test
        y_train = transformer_bundle.y_train
        y_valid = transformer_bundle.y_valid
        y_test = transformer_bundle.y_test

        feature_mean = X_train.mean(axis=(0, 1), keepdims=True)
        feature_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
        X_train = ((X_train - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_valid = ((X_valid - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_test = ((X_test - feature_mean) / feature_std).astype(np.float32, copy=False)

        del transformer_bundle, sequence_split_frames_materialized
        import gc
        gc.collect()

        TRANSFORMER_RUN = True
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "fixed_split_block_rows": FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(transformer_window_size),
                "features_per_step": int(len(measurement_columns)),
            }
        )


### Transformer Step 2: Build Attention Blocks, Loss, and DataLoaders

The transformer needs a few ingredients that are specific to attention models:

- an input projection that maps raw sensor features into `d_model`,
- a positional embedding so the model knows where each time step sits in the window,
- stacked encoder blocks with multi-head attention,
- and, in window mode, a pooling rule that turns the whole window into one classification vector.

The output toggle changes the last step:

- in `"window"` mode, we pool the encoded sequence and classify once,
- in `"per_timestep"` mode, we classify every encoded position directly.

Everything else should feel familiar from the CNN section: loss, optimiser, scheduler, and `DataLoader` setup.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer model setup skipped.")
else:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class TinyQCTransformer(nn.Module):
        def __init__(self, input_dim: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            self.input_projection = nn.Linear(input_dim, TRANSFORMER_CONFIG["d_model"])
            self.position_embedding = nn.Parameter(
                torch.zeros(1, TRANSFORMER_CONFIG["window_size"], TRANSFORMER_CONFIG["d_model"])
            )
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=TRANSFORMER_CONFIG["d_model"],
                nhead=TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=TRANSFORMER_CONFIG["num_layers"])
            self.norm = nn.LayerNorm(TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(TRANSFORMER_CONFIG["d_model"], output_dim)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            if TRANSFORMER_OUTPUT_MODE == "window":
                if TRANSFORMER_CONFIG["pooling"] == "last":
                    pooled = x[:, -1, :]
                else:
                    pooled = x.mean(dim=1)
                pooled = self.norm(pooled)
                return self.head(pooled)
            x = self.norm(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(-1)
            return logits

    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(transformer_class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(transformer_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    loader_kwargs = {}
    num_workers = int(TRANSFORMER_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(TRANSFORMER_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(TRANSFORMER_LOADER_CONFIG["prefetch_factor"])
    if TRANSFORMER_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(train_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=True, **loader_kwargs)
    valid_loader = DataLoader(valid_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)

    model = TinyQCTransformer(input_dim=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=TRANSFORMER_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=TRANSFORMER_CONFIG["lr_decay_factor"],
        patience=TRANSFORMER_CONFIG["lr_patience"],
    )

    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                if task_mode == "multiclass":
                    if TRANSFORMER_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), TRANSFORMER_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": TRANSFORMER_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": TRANSFORMER_LOADER_CONFIG,
        }
    )


### Transformer Step 3: Train with Attention-Based Sequence Modelling

The training loop is almost the same as the CNN loop, which is useful pedagogically: once the data pipeline and validation workflow are clear, we can swap in a different model family without changing the whole experimental process.

Just like the CNN section, the reported validation F1 is per window in `"window"` mode and per timestamp in `"per_timestep"` mode.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer fit skipped.")
else:
    for epoch in range(1, TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss, _, _ = run_epoch(train_loader, training=True)
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        if valid_f1 > best_metric + TRANSFORMER_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": transformer_class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": transformer_window_size,
                    "output_mode": TRANSFORMER_OUTPUT_MODE,
                    "config": {**TRANSFORMER_CONFIG, **TRANSFORMER_LOADER_CONFIG},
                },
                TRANSFORMER_MODEL_PATH,
            )
        else:
            patience_counter += 1
            if patience_counter >= TRANSFORMER_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### Transformer Step 4: Evaluate the Best Attention Model

Just like the CNN section, we finish by restoring the best validation checkpoint and scoring the held-out test split exactly once.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer evaluation skipped.")
else:
    if best_state is None:
        raise RuntimeError("Transformer training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": TRANSFORMER_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(TRANSFORMER_MODEL_PATH),
        }
    )

    if task_mode == "multiclass":
        report_labels = list(range(len(transformer_class_labels)))
        report_names = [str(label) for label in transformer_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Transformer loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Transformer test confusion matrix")
    plt.tight_layout()
    plt.show()


### Transformer Variant: Forecast Good Data, Then Flag Large Errors

The supervised transformer above learns QC labels directly. A different anomaly-detection pattern is to train only on good data, ask the model to predict the next sensor value from recent history, and treat large prediction errors as suspicious.

This is useful when labels are sparse or when you want to catch failure modes that were not well represented in the reviewed labels. The model is not learning what each bad flag looks like. It is learning what normal behaviour usually looks like.

The training setup is:

1. build contiguous windows where the previous `lookback` rows predict the next `target_column` value,
2. train on good-only windows so the model learns normal behaviour,
3. validate on good-only windows for early stopping,
4. score validation/test windows that include both good and issue labels,
5. flag windows whose standardised next-value prediction error is above a good-validation threshold.

`include_target_history` controls whether the model sees previous values of the same column it is trying to predict. If `target_column` is `cond_value_ctd`, then `include_target_history=True` lets the model use previous conductivity values to predict the next conductivity value.

`include_target_history=False` makes the model **exogenous** for the target: it predicts the target from the other sensor/context columns, not from the target's own recent history. This can help for persistent failures such as conductivity plugs, because bad conductivity can be easy to predict from previous bad conductivity. Exogenous prediction asks a cleaner question: *given the other sensors, what should conductivity have been?*

| Parameter | What it changes |
|---|---|
| `target_column` | measurement value to predict one step ahead |
| `lookback` | number of previous rows used as transformer context |
| `include_target_history` | `True` uses the target's own recent history; `False` uses only other columns as exogenous predictors |
| `max_gap_seconds` | drops windows that jump across large time gaps |
| `max_train_source_rows` / `max_eval_source_rows` | caps labelled rows loaded before windows are built |
| `max_train_windows` / `max_eval_windows` | caps final transformer examples after valid windows are found |
| `loss` | `mse` penalizes large errors strongly; `huber` is more robust to occasional large deviations |
| `threshold_quantile` | good-validation error percentile used as the anomaly cutoff |

This section follows the same broad idea used in recent transformer anomaly-detection and forecasting work:

- [Anomaly Transformer](https://arxiv.org/abs/2110.02642) frames unsupervised time-series anomaly detection around how abnormal points relate to the rest of the series.
- [TranAD](https://arxiv.org/abs/2201.07284) uses transformer sequence encoders for multivariate time-series anomaly detection and diagnosis.
- [PatchTST](https://arxiv.org/abs/2211.14730) is a strong example of transformer-based time-series forecasting, especially the idea that forecasting can be a useful self-supervised signal.

The implementation below keeps the idea simple: predict the next value of one measurement column, set an error threshold from good validation windows, then inspect which validation/test points exceed that threshold.


In [ ]:
# Participant choice: keep this False for an exogenous detector, or True to let
# the model use the target's own recent history. Conductivity plugs default to
# False because persistent bad conductivity can otherwise become easy to predict.
FORECAST_INCLUDE_TARGET_HISTORY = DATASET_PROFILE_ID != "conductivity_plugs"

# Huber is a robust regression loss. MSE is also useful when large prediction
# errors should be penalized more aggressively.
FORECAST_LOSS = "huber" if DATASET_PROFILE_ID == "conductivity_plugs" else "mse"

# The threshold is set from good-validation errors. Lower values catch more
# potential anomalies; higher values are more conservative.
FORECAST_THRESHOLD_QUANTILE = 0.95 if DATASET_PROFILE_ID == "conductivity_plugs" else 0.99

FORECAST_TRANSFORMER_CONFIG = {
    # Predict the next value of this column. For conductivity plugs this is cond_value_ctd.
    "target_column": PLOT_MEASUREMENT_COLUMN,
    # Use recent rows as context. Larger values see longer trends but cost more memory.
    "lookback": 96,
    # Windows are sampled after contiguous candidate windows are built, so sequence order is preserved.
    "max_train_windows": 100_000 if DATASET_PROFILE_ID == "conductivity_plugs" else 20_000,
    "max_eval_windows": 20_000,
    # These cap the labelled source rows loaded before windows are built.
    # The helper keeps contiguous blocks so a 96-row history is still a real local sequence.
    "max_train_source_rows": 300_000 if DATASET_PROFILE_ID == "conductivity_plugs" else 120_000,
    "max_eval_source_rows": 120_000,
    # Larger blocks preserve longer local stretches but may load more rows than needed.
    "context_block_rows": 4096,
    # Reject windows that cross large time gaps. This keeps "next row" close to "next moment".
    "max_gap_seconds": 5.0,
    # True: use previous target values as features. False: use only exogenous/context features.
    "include_target_history": FORECAST_INCLUDE_TARGET_HISTORY,
    # d_model is the transformer embedding size: larger can learn richer patterns but costs more memory.
    "d_model": 64,
    # nhead is the number of attention heads. d_model must be divisible by nhead.
    "nhead": 4,
    # num_layers is the number of stacked transformer encoder blocks.
    "num_layers": 2,
    # dim_feedforward controls the hidden width inside each transformer block.
    "dim_feedforward": 128,
    # Dropout randomly masks activations during training to reduce overfitting.
    "dropout": 0.15,
    # epochs is the maximum pass count; early stopping can stop sooner.
    "epochs": 6,
    # batch_size controls examples per optimizer step. Larger is faster on GPU but uses more memory.
    "batch_size": 256,
    # learning_rate controls optimizer step size.
    "learning_rate": 1e-3,
    # weight_decay is mild L2 regularization on model weights.
    "weight_decay": 1e-4,
    # loss can be "mse" or "huber".
    "loss": FORECAST_LOSS,
    # patience and min_delta control early stopping on good-validation loss.
    "patience": 2,
    "min_delta": 1e-4,
    # Gradient clipping prevents rare large updates from destabilizing training.
    "gradient_clip_norm": 1.0,
    # A 95th-percentile good-validation threshold worked better for conductivity plugs locally.
    "threshold_quantile": FORECAST_THRESHOLD_QUANTILE,
}

display(pd.Series(FORECAST_TRANSFORMER_CONFIG, name="value").rename_axis("forecast_parameter").to_frame())
print(
    {
        "training_data": "good reviewed rows only",
        "anomaly_score": "absolute standardised next-value prediction error",
        "feature_history": "target included" if FORECAST_TRANSFORMER_CONFIG["include_target_history"] else "target excluded",
    }
)


#### Build Good-Only Forecasting Windows

Each training example contains `lookback` previous rows and one next value to predict.

For training, the context rows and the next target row must all have good QC/custom labels. The validation-loss windows are also good-only, because they control early stopping for a normal-behaviour model. For validation/test scoring, we keep both good and issue rows so prediction error can be compared against the reviewed labels.


In [ ]:
def select_evenly_spaced_positions(values: np.ndarray, limit: int | None) -> np.ndarray:
    """Keep positions spread across time when a split has more candidate windows than we want."""
    if limit is None or len(values) <= limit:
        return values
    positions = np.linspace(0, len(values) - 1, num=int(limit), dtype=int)
    return values[positions]


def build_next_value_forecast_windows(
    frame: pd.DataFrame,
    *,
    feature_columns: list[str],
    target_column: str,
    label_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    lookback: int,
    max_windows: int | None,
    require_good_window: bool,
    max_gap_seconds: float | None,
) -> dict[str, object]:
    """Build transformer windows for next-value forecasting.

    `require_good_window=True` is used for training and validation loss so the
    model learns normal dynamics only. `max_gap_seconds` prevents windows from
    jumping across large time gaps and treating distant rows as adjacent.
    """
    if not feature_columns:
        raise ValueError("feature_columns must contain at least one column.")
    if lookback < 2:
        raise ValueError("lookback must be at least 2")

    work_columns = list(dict.fromkeys(["Time UTC", "source_file", label_column, target_column, *feature_columns]))
    work = frame[work_columns].copy()
    work["Time UTC"] = pd.to_datetime(work["Time UTC"], utc=True)
    for column in [label_column, target_column, *feature_columns]:
        work[column] = pd.to_numeric(work[column], errors="coerce")
    work = work.dropna(subset=["Time UTC", "source_file", label_column, target_column, *feature_columns]).copy()
    if work.empty:
        return {
            "X": np.empty((0, lookback, len(feature_columns)), dtype=np.float32),
            "y": np.empty((0,), dtype=np.float32),
            "metadata": pd.DataFrame(columns=["Time UTC", "source_file", "target_value", "qc_label", "is_good", "is_issue"]),
        }

    source_groups = list(work.groupby("source_file", sort=False, observed=False))
    per_source_limit = None if max_windows is None else max(1, int(np.ceil(int(max_windows) / max(len(source_groups), 1))))
    good_label_set = {int(label) for label in good_labels}
    issue_label_set = {int(label) for label in issue_labels}

    X_parts = []
    y_values = []
    metadata_rows = []

    for source_file, source_frame in source_groups:
        source_frame = source_frame.sort_values("Time UTC").reset_index(drop=True)
        if len(source_frame) <= lookback:
            continue

        values = source_frame[feature_columns].to_numpy(dtype=np.float32)
        target_values = source_frame[target_column].to_numpy(dtype=np.float32)
        labels = source_frame[label_column].astype(int).to_numpy()
        time_values = source_frame["Time UTC"].astype("int64").to_numpy(copy=False)
        finite_rows = np.isfinite(values).all(axis=1) & np.isfinite(target_values)
        good_rows = np.isin(labels, list(good_label_set))
        max_gap_ns = None if max_gap_seconds is None else int(max_gap_seconds * 1_000_000_000)

        candidate_ends = []
        for end_index in range(lookback, len(source_frame)):
            context_slice = slice(end_index - lookback, end_index)
            if not finite_rows[context_slice].all() or not finite_rows[end_index]:
                continue
            if max_gap_ns is not None:
                window_times = time_values[end_index - lookback : end_index + 1]
                if np.diff(window_times).max() > max_gap_ns:
                    continue
            if require_good_window and (not good_rows[context_slice].all() or not good_rows[end_index]):
                continue
            candidate_ends.append(end_index)

        selected_ends = select_evenly_spaced_positions(np.asarray(candidate_ends, dtype=int), per_source_limit)
        for end_index in selected_ends.tolist():
            context_slice = slice(end_index - lookback, end_index)
            X_parts.append(values[context_slice])
            y_values.append(float(target_values[end_index]))
            qc_label = int(labels[end_index])
            metadata_rows.append(
                {
                    "Time UTC": source_frame.loc[end_index, "Time UTC"],
                    "source_file": source_file,
                    "target_value": float(target_values[end_index]),
                    "qc_label": qc_label,
                    "is_good": qc_label in good_label_set,
                    "is_issue": qc_label in issue_label_set,
                }
            )

    if not X_parts:
        return {
            "X": np.empty((0, lookback, len(feature_columns)), dtype=np.float32),
            "y": np.empty((0,), dtype=np.float32),
            "metadata": pd.DataFrame(columns=["Time UTC", "source_file", "target_value", "qc_label", "is_good", "is_issue"]),
        }

    X = np.stack(X_parts).astype(np.float32, copy=False)
    y = np.asarray(y_values, dtype=np.float32)
    metadata_frame = pd.DataFrame(metadata_rows)

    if max_windows is not None and len(X) > int(max_windows):
        keep = select_evenly_spaced_positions(np.arange(len(X)), int(max_windows))
        X = X[keep]
        y = y[keep]
        metadata_frame = metadata_frame.iloc[keep].reset_index(drop=True)
    else:
        metadata_frame = metadata_frame.reset_index(drop=True)

    return {"X": X, "y": y, "metadata": metadata_frame}


def select_contiguous_time_blocks(
    frame: pd.DataFrame,
    *,
    max_rows: int | None,
    block_rows: int,
    lookback: int,
    time_column: str = "Time UTC",
    source_column: str = "source_file",
) -> pd.DataFrame:
    """Keep a row budget as local time blocks instead of isolated sampled rows.

    Forecasting windows need adjacent rows. Evenly sampling individual rows can
    turn a 96-row history into 96 distant timestamps, so this helper samples
    whole contiguous blocks spread across source files and time.
    """
    if max_rows is None or len(frame) <= int(max_rows):
        return frame.sort_values([source_column, time_column]).reset_index(drop=True)

    block_rows = max(int(block_rows), int(lookback) + 1)
    candidate_blocks = []
    for source_file, source_frame in frame.groupby(source_column, sort=False, observed=False):
        source_frame = source_frame.sort_values(time_column).reset_index(drop=True)
        for start in range(0, len(source_frame), block_rows):
            stop = min(start + block_rows, len(source_frame))
            if stop - start >= int(lookback) + 1:
                candidate_blocks.append(source_frame.iloc[start:stop])

    if not candidate_blocks:
        return frame.iloc[0:0].copy()
    block_limit = max(1, int(np.ceil(int(max_rows) / block_rows)))
    selected_block_indices = select_evenly_spaced_positions(np.arange(len(candidate_blocks)), block_limit)
    selected = pd.concat([candidate_blocks[index] for index in selected_block_indices], ignore_index=True)
    if len(selected) > int(max_rows):
        selected = selected.iloc[: int(max_rows)]
    return selected.sort_values([source_column, time_column]).reset_index(drop=True)


if not globals().get("PYTORCH_READY", False):
    FORECAST_TRANSFORMER_RUN = False
    print("Good-data forecasting transformer skipped because PyTorch is not available in the current notebook state.")
else:
    # The target is predicted one row into the future. It can be included or excluded from the input history.
    forecast_target_column = FORECAST_TRANSFORMER_CONFIG["target_column"]
    if forecast_target_column not in ROW_USE_COLUMNS:
        raise ValueError(f"Forecast target {forecast_target_column!r} is not in the active row columns.")
    # Start with all measurement columns available in the row-level cache.
    forecast_feature_columns = [column for column in measurement_columns if column in ROW_USE_COLUMNS]
    if not FORECAST_TRANSFORMER_CONFIG["include_target_history"]:
        # Exogenous mode: remove the target from the input features, but still predict it as y.
        forecast_feature_columns = [column for column in forecast_feature_columns if column != forecast_target_column]
    if forecast_target_column not in forecast_feature_columns:
        if FORECAST_TRANSFORMER_CONFIG["include_target_history"]:
            raise ValueError(f"Forecast target {forecast_target_column!r} is not in the active measurement columns.")
    if not forecast_feature_columns:
        raise ValueError("No forecast feature columns remain after applying include_target_history.")

    # Use the full reviewed label table when it is available so contiguous forecast windows stay realistic.
    forecast_label_df = reviewed_label_df if "reviewed_label_df" in globals() else reviewed_model_df
    forecast_split_block_rows = FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_BLOCK_ROWS is not None else INTERLEAVED_BLOCK_ROWS
    # Rebuild the same train/validation/test split on the label table used for forecasting.
    forecast_label_splits = split_frame_by_strategy(
        forecast_label_df,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        strategy=FIXED_SPLIT_STRATEGY,
        source_column="source_file",
        block_rows=forecast_split_block_rows,
        issue_column="issue",
        target_column=TARGET_FLAG,
        issue_labels=ISSUE_LABELS,
        episode_context_rows=EPISODE_CONTEXT_ROWS,
        episode_merge_gap_rows=EPISODE_MERGE_GAP_ROWS,
        purge_gap_rows=EPISODE_PURGE_GAP_ROWS,
    )
    # Row budgets are applied as contiguous blocks, not isolated rows, because sequence models need neighbours.
    forecast_selection_frames = {
        "train": select_contiguous_time_blocks(
            forecast_label_splits["train"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_train_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
        "validation": select_contiguous_time_blocks(
            forecast_label_splits["validation"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_eval_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
        "test": select_contiguous_time_blocks(
            forecast_label_splits["test"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_eval_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
    }
    # Load the measurement columns for only the selected forecast rows from parquet.
    forecast_split_rows = materialize_reviewed_split_frames(
        selected_paths,
        forecast_selection_frames,
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    lookback = int(FORECAST_TRANSFORMER_CONFIG["lookback"])
    # Training windows are good-only: the model should learn normal next-value behaviour.
    forecast_train_bundle = build_next_value_forecast_windows(
        forecast_split_rows["train"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_train_windows"],
        require_good_window=True,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    # Good-only validation controls early stopping without learning from known issue windows.
    forecast_valid_good_bundle = build_next_value_forecast_windows(
        forecast_split_rows["validation"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=True,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    # Scoring windows include good and issue labels so we can compare error against reviewed labels.
    forecast_valid_bundle = build_next_value_forecast_windows(
        forecast_split_rows["validation"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=False,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    forecast_test_bundle = build_next_value_forecast_windows(
        forecast_split_rows["test"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=False,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )

    if (
        len(forecast_train_bundle["X"]) == 0
        or len(forecast_valid_good_bundle["X"]) == 0
        or len(forecast_valid_bundle["X"]) == 0
        or len(forecast_test_bundle["X"]) == 0
    ):
        FORECAST_TRANSFORMER_RUN = False
        print(
            {
                "skip_reason": "At least one split produced no next-value forecasting windows.",
                "train_windows": int(len(forecast_train_bundle["X"])),
                "validation_good_windows": int(len(forecast_valid_good_bundle["X"])),
                "validation_windows": int(len(forecast_valid_bundle["X"])),
                "test_windows": int(len(forecast_test_bundle["X"])),
            }
        )
    else:
        forecast_feature_mean = forecast_train_bundle["X"].mean(axis=(0, 1), keepdims=True)
        forecast_feature_std = forecast_train_bundle["X"].std(axis=(0, 1), keepdims=True) + 1e-6
        forecast_target_mean = float(forecast_train_bundle["y"].mean())
        forecast_target_std = float(forecast_train_bundle["y"].std() + 1e-6)

        for bundle in [forecast_train_bundle, forecast_valid_good_bundle, forecast_valid_bundle, forecast_test_bundle]:
            bundle["X_scaled"] = ((bundle["X"] - forecast_feature_mean) / forecast_feature_std).astype(np.float32, copy=False)
            bundle["y_scaled"] = ((bundle["y"] - forecast_target_mean) / forecast_target_std).astype(np.float32, copy=False)

        FORECAST_TRANSFORMER_RUN = True
        print(
            {
                "target_column": forecast_target_column,
                "lookback": lookback,
                "features_per_step": len(forecast_feature_columns),
                "train_source_rows_loaded": int(len(forecast_selection_frames["train"])),
                "validation_source_rows_loaded": int(len(forecast_selection_frames["validation"])),
                "test_source_rows_loaded": int(len(forecast_selection_frames["test"])),
                "train_good_windows": int(len(forecast_train_bundle["X"])),
                "validation_good_windows": int(len(forecast_valid_good_bundle["X"])),
                "validation_scoring_windows": int(len(forecast_valid_bundle["X"])),
                "test_scoring_windows": int(len(forecast_test_bundle["X"])),
            }
        )


#### Train A Next-Value Transformer On Good Windows

This model has the same encoder ingredients as the classifier: input projection, positional embedding, transformer encoder blocks, and a small prediction head.

The difference is the loss. Instead of cross-entropy against QC labels, this model minimizes a regression loss for the next value of `FORECAST_TRANSFORMER_CONFIG["target_column"]`. The conductivity-plug default uses Huber loss because it was more stable than plain MSE in local GPU experiments.


In [ ]:
if not FORECAST_TRANSFORMER_RUN:
    print("Good-data forecasting transformer fit skipped.")
else:
    torch.manual_seed(SEED)
    forecast_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if forecast_device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class NextValueTransformer(nn.Module):
        def __init__(self, input_dim: int) -> None:
            super().__init__()
            # Project raw feature columns into the transformer embedding space.
            self.input_projection = nn.Linear(input_dim, FORECAST_TRANSFORMER_CONFIG["d_model"])
            # Learned positional embeddings let the transformer distinguish early vs late context rows.
            self.position_embedding = nn.Parameter(
                torch.zeros(1, FORECAST_TRANSFORMER_CONFIG["lookback"], FORECAST_TRANSFORMER_CONFIG["d_model"])
            )
            # Each encoder layer applies self-attention plus a feed-forward network.
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=FORECAST_TRANSFORMER_CONFIG["d_model"],
                nhead=FORECAST_TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=FORECAST_TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=FORECAST_TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=FORECAST_TRANSFORMER_CONFIG["num_layers"])
            # The final context row summarises the history used to predict the next value.
            self.norm = nn.LayerNorm(FORECAST_TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(FORECAST_TRANSFORMER_CONFIG["d_model"], 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            x = self.norm(x[:, -1, :])
            return self.head(x).squeeze(-1)

    forecast_loader_kwargs = {}
    num_workers = int(TRANSFORMER_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        forecast_loader_kwargs["num_workers"] = num_workers
        forecast_loader_kwargs["persistent_workers"] = bool(TRANSFORMER_LOADER_CONFIG["persistent_workers"])
        forecast_loader_kwargs["prefetch_factor"] = int(TRANSFORMER_LOADER_CONFIG["prefetch_factor"])
    if TRANSFORMER_LOADER_CONFIG["pin_memory"] and forecast_device.type == "cuda":
        forecast_loader_kwargs["pin_memory"] = True
    forecast_non_blocking = bool(forecast_loader_kwargs.get("pin_memory", False) and forecast_device.type == "cuda")

    forecast_train_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_train_bundle["X_scaled"])),
        torch.from_numpy(forecast_train_bundle["y_scaled"]),
    )
    forecast_valid_good_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_valid_good_bundle["X_scaled"])),
        torch.from_numpy(forecast_valid_good_bundle["y_scaled"]),
    )
    forecast_valid_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_valid_bundle["X_scaled"])),
        torch.from_numpy(forecast_valid_bundle["y_scaled"]),
    )
    forecast_test_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_test_bundle["X_scaled"])),
        torch.from_numpy(forecast_test_bundle["y_scaled"]),
    )
    forecast_train_loader = DataLoader(
        forecast_train_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=True,
        **forecast_loader_kwargs,
    )
    forecast_valid_good_loader = DataLoader(
        forecast_valid_good_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )
    forecast_valid_loader = DataLoader(
        forecast_valid_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )
    forecast_test_loader = DataLoader(
        forecast_test_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )

    forecast_model = NextValueTransformer(input_dim=len(forecast_feature_columns)).to(forecast_device)
    # Huber is less sensitive to rare large errors; MSE penalizes large errors more strongly.
    forecast_loss_fn = nn.SmoothL1Loss() if FORECAST_TRANSFORMER_CONFIG["loss"] == "huber" else nn.MSELoss()
    forecast_optimizer = torch.optim.AdamW(
        forecast_model.parameters(),
        lr=FORECAST_TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=FORECAST_TRANSFORMER_CONFIG["weight_decay"],
    )

    forecast_history = []
    forecast_best_state = None
    forecast_best_loss = np.inf
    forecast_best_epoch = 0
    forecast_patience_counter = 0

    def run_forecast_epoch(loader, training: bool) -> float:
        forecast_model.train(training)
        total_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(forecast_device, non_blocking=forecast_non_blocking)
            batch_y = batch_y.to(forecast_device, non_blocking=forecast_non_blocking)
            with torch.set_grad_enabled(training):
                prediction = forecast_model(batch_x)
                loss = forecast_loss_fn(prediction, batch_y)
                if training:
                    forecast_optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if FORECAST_TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(forecast_model.parameters(), FORECAST_TRANSFORMER_CONFIG["gradient_clip_norm"])
                    forecast_optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
        return total_loss / max(len(loader.dataset), 1)

    for epoch in range(1, FORECAST_TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss = run_forecast_epoch(forecast_train_loader, training=True)
        valid_loss = run_forecast_epoch(forecast_valid_good_loader, training=False)
        forecast_history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})
        print(f"Forecast epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")

        if valid_loss < forecast_best_loss - FORECAST_TRANSFORMER_CONFIG["min_delta"]:
            forecast_best_loss = valid_loss
            forecast_best_epoch = epoch
            forecast_patience_counter = 0
            forecast_best_state = copy.deepcopy(forecast_model.state_dict())
        else:
            forecast_patience_counter += 1
            if forecast_patience_counter >= FORECAST_TRANSFORMER_CONFIG["patience"]:
                print(f"Forecast early stopping triggered at epoch {epoch}")
                break

    if forecast_best_state is not None:
        forecast_model.load_state_dict(forecast_best_state)
    print({"forecast_best_epoch": forecast_best_epoch, "forecast_best_validation_loss": float(forecast_best_loss)})


#### Score Prediction Errors As Anomaly Signals

The anomaly score is the absolute prediction error after standardizing the target column. A high score means the next observed value was far from what the good-data model expected.

We set the threshold from the good validation rows only. Then we ask how often issue rows in validation and test exceed that threshold.


In [ ]:
if not FORECAST_TRANSFORMER_RUN:
    print("Good-data forecasting transformer scoring skipped.")
else:
    def score_forecast_loader(loader, metadata_frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
        forecast_model.eval()
        predictions = []
        targets = []
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(forecast_device, non_blocking=forecast_non_blocking)
                prediction = forecast_model(batch_x).detach().cpu().numpy()
                predictions.append(prediction)
                targets.append(batch_y.detach().cpu().numpy())
        predicted_scaled = np.concatenate(predictions) if predictions else np.array([], dtype=np.float32)
        target_scaled = np.concatenate(targets) if targets else np.array([], dtype=np.float32)
        result = metadata_frame.reset_index(drop=True).copy()
        result["split"] = split_name
        result["predicted_value"] = predicted_scaled * forecast_target_std + forecast_target_mean
        result["actual_value"] = target_scaled * forecast_target_std + forecast_target_mean
        result["forecast_error"] = result["actual_value"] - result["predicted_value"]
        result["forecast_abs_error_scaled"] = np.abs(target_scaled - predicted_scaled)
        return result

    forecast_valid_scores = score_forecast_loader(
        forecast_valid_loader,
        forecast_valid_bundle["metadata"],
        "validation",
    )
    forecast_test_scores = score_forecast_loader(
        forecast_test_loader,
        forecast_test_bundle["metadata"],
        "test",
    )

    threshold_source = forecast_valid_scores.loc[forecast_valid_scores["is_good"], "forecast_abs_error_scaled"]
    if threshold_source.empty:
        threshold_source = forecast_valid_scores["forecast_abs_error_scaled"]
    # The threshold is learned from good validation rows only, not from issue labels.
    forecast_error_threshold = float(
        threshold_source.quantile(float(FORECAST_TRANSFORMER_CONFIG["threshold_quantile"]))
    )

    forecast_score_rows = []
    for split_name, score_frame in [("validation", forecast_valid_scores), ("test", forecast_test_scores)]:
        # Prediction errors above the threshold are treated as anomalies.
        score_frame["predicted_anomaly"] = score_frame["forecast_abs_error_scaled"] >= forecast_error_threshold
        y_true_anomaly = score_frame["is_issue"].astype(int)
        y_pred_anomaly = score_frame["predicted_anomaly"].astype(int)
        forecast_score_rows.append(
            {
                "split": split_name,
                "rows": len(score_frame),
                "issue_rows": int(y_true_anomaly.sum()),
                "predicted_anomaly_rows": int(y_pred_anomaly.sum()),
                "median_abs_error_scaled": float(score_frame["forecast_abs_error_scaled"].median()),
                "p95_abs_error_scaled": float(score_frame["forecast_abs_error_scaled"].quantile(0.95)),
                "forecast_anomaly_f1": float(f1_score(y_true_anomaly, y_pred_anomaly, zero_division=0)),
            }
        )
        print(f"\n{split_name.title()} forecast-error anomaly report")
        print(
            classification_report(
                y_true_anomaly,
                y_pred_anomaly,
                labels=[0, 1],
                target_names=["not issue", "issue"],
                zero_division=0,
            )
        )

    forecast_score_summary = pd.DataFrame(forecast_score_rows).set_index("split")
    display(forecast_score_summary)
    print(
        {
            "threshold_quantile": FORECAST_TRANSFORMER_CONFIG["threshold_quantile"],
            "forecast_error_threshold": forecast_error_threshold,
            "threshold_rows_from_good_validation": int(len(threshold_source)),
        }
    )

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8), sharey=True)
    for axis, (split_name, score_frame) in zip(axes, [("validation", forecast_valid_scores), ("test", forecast_test_scores)]):
        good_scores = score_frame.loc[score_frame["is_good"], "forecast_abs_error_scaled"]
        issue_scores = score_frame.loc[score_frame["is_issue"], "forecast_abs_error_scaled"]
        bins = np.linspace(
            0,
            max(float(score_frame["forecast_abs_error_scaled"].quantile(0.995)), forecast_error_threshold, 1e-6),
            45,
        )
        axis.hist(good_scores, bins=bins, alpha=0.65, label="good next rows", color="#1f77b4")
        axis.hist(issue_scores, bins=bins, alpha=0.65, label="issue next rows", color="#d62728")
        axis.axvline(forecast_error_threshold, color="#111827", linestyle="--", linewidth=1.8, label="threshold")
        axis.set_title(f"{split_name.title()} next-value forecast errors")
        axis.set_xlabel("Absolute standardised prediction error")
        axis.grid(axis="y", alpha=0.2)
    axes[0].set_ylabel("Window count")
    axes[1].legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    plt.show()


#### Reflection Questions: Forecast-Error Transformer

1. Are large prediction errors lining up with known issue flags, or mostly with legitimate rapid changes?
2. Would this detector work better if it predicted several future steps instead of only the next value?
3. Should the anomaly score combine several measurement columns, or is one operationally important column enough?
4. Does training only on good rows make the detector more sensitive to novel failure modes, or too sensitive to normal regime changes?


### Date-Range Demo: See Transformer Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
TRANSFORMER_RANGE_START = None
TRANSFORMER_RANGE_END = None
TRANSFORMER_AUTO_SELECT_TEST_RANGE = True
TRANSFORMER_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "TRANSFORMER_RANGE_START": TRANSFORMER_RANGE_START,
        "TRANSFORMER_RANGE_END": TRANSFORMER_RANGE_END,
        "TRANSFORMER_AUTO_SELECT_TEST_RANGE": TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        "TRANSFORMER_MAX_POINTS_TO_PLOT": TRANSFORMER_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer date-range demo skipped because the transformer was not trained in this run.")
else:
    transformer_range_selection = select_time_range(
        test_df,
        time_column="Time UTC",
        label_column=TARGET_FLAG,
        start=TRANSFORMER_RANGE_START,
        end=TRANSFORMER_RANGE_END,
        auto_select=TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
    )
    transformer_interval_metrics = None

    transformer_interval_rows = load_rows_for_time_range(
        metadata,
        row_cache_dir=Path(ROW_CACHE_DIR),
        start=transformer_range_selection["start"],
        end=transformer_range_selection["end"],
        columns=ROW_USE_COLUMNS,
    )

    if transformer_interval_rows.empty:
        print("No row-level data was found in the requested transformer range.")
    else:
        transformer_interval_model_df, _, _ = build_model_frame(
            transformer_interval_rows,
            target_flag=TARGET_FLAG,
            measurement_columns=measurement_columns,
            task_mode=task_mode,
            model_row_limit=None,
        )
        transformer_interval_model_df = transformer_interval_model_df[
            (transformer_interval_model_df["Time UTC"] >= transformer_range_selection["start"])
            & (transformer_interval_model_df["Time UTC"] <= transformer_range_selection["end"])
        ].reset_index(drop=True)
        transformer_interval_origin = infer_interval_origin(
            transformer_range_selection["start"],
            transformer_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        transformer_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}

        if TRANSFORMER_CONFIG["output_mode"] == "window":
            transformer_interval_bundle = build_window_classification_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                task_mode=task_mode,
                window_size=TRANSFORMER_CONFIG["window_size"],
                label_reduction=TRANSFORMER_CONFIG["label_reduction"],
            )

            if transformer_interval_bundle["window_frame"].empty:
                print("The selected transformer range is shorter than one full window after preprocessing, so the window-level demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_window_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_window_frame = transformer_interval_bundle["window_frame"].copy()
                transformer_window_frame["predicted_label"] = transformer_predicted_labels
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_window_frame["true_label"],
                    transformer_window_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "true_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "predicted_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "window_count_in_interval": int(len(transformer_window_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_window_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_window_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True window {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer {FLAG_EXAMPLE_DISPLAY_NAME} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer window predictions on a selected time range",
                    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
                    legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
                )
                plt.show()
        else:
            transformer_interval_bundle = build_sequence_label_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                window_size=TRANSFORMER_CONFIG["window_size"],
            )

            if len(transformer_interval_bundle["raw_sequences"]) == 0:
                print("The selected transformer range is shorter than one full window after preprocessing, so the per-timestep demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_sequence_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_point_frame = pd.DataFrame(
                    {
                        "Time UTC": pd.to_datetime(transformer_interval_bundle["raw_times"].reshape(-1), utc=True),
                        "true_label": transformer_interval_bundle["raw_targets"].reshape(-1).astype(int),
                        "predicted_label": transformer_predicted_labels.reshape(-1).astype(int),
                    }
                )
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_point_frame["true_label"],
                    transformer_point_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="true_label")
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="predicted_label")
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "point_count_in_interval": int(len(transformer_point_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_point_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_point_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True per-timestep {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer per-timestep {FLAG_EXAMPLE_DISPLAY_NAME} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer per-timestep predictions on a selected time range",
                    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
                    legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
                )
                plt.show()

        if transformer_interval_metrics is not None:
            transformer_cm_fig, transformer_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=transformer_interval_metrics["confusion_matrix"],
                display_labels=transformer_interval_metrics["display_labels"],
            ).plot(ax=transformer_cm_ax, cmap="Blues", colorbar=False)
            transformer_cm_ax.set_title("Transformer confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()


### Reflection Questions: Transformer

1. Does the selected interval contain longer-context behaviour that you would expect a transformer to use better than the CNN?
2. If the transformer is not clearly helping here, do you think the issue is window length, data volume, or model size?
3. Would you change pooling, positional information, or the target definition first if you wanted a better interval-level result?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 8 — Reflection and Next Steps

End by comparing what each model family sees, what each one misses, and which changes are most promising for the next round of experiments.


### Wrap-Up, Transformer Note, And Questions To Explore

We now have three very different model families in one notebook:

- a tabular tree ensemble,
- a local-pattern sequence model,
- and a small attention-based sequence model.

That is a useful comparison because each one sees the data a little differently.

Good next questions to explore:

1. Are there other features that might help the Random Forest?
Hint: think about lag features, rolling means, rolling standard deviations, slopes, or relationships between sensors.

2. Are there ways to improve class `3`, `4`, or `9` performance specifically?
Hint: compare the class distribution to the confusion matrix and ask which classes are hardest to distinguish.

3. Does the target itself need to change?
Hint: sometimes a binary `good` vs `issue` target, or a collapsed set of flags, is closer to the operational decision. For conductivity or turbidity, that might mean merging `3` and `4`. For oxygen, it can make more sense to ask whether `1/2` belong together and whether `3/4` belong together, rather than reusing the CTD collapse blindly.

4. What parts of the CNN are worth experimenting with?
Hint: try changing the window length, number of channels, dropout, or the way we reduce row labels into one window label.

5. Could we use even more context from time?
Hint: Random Forest only sees engineered features. CNN sees short windows. Transformer sees relationships across an entire window. There are still other options too, such as TCNs, GRUs/LSTMs, or hierarchical windows.
